# Fixed-capture Marlin diagnostic
Numerical variation and paired timings; no serving qualification.


In [ ]:
"""First notebook cell: retain actual inventory and reject the wrong accelerator."""
import json
import subprocess
from pathlib import Path

hardware_dir = Path('/kaggle/working/marlin-replay')
hardware_dir.mkdir(parents=True, exist_ok=True)
inventory = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'],
    capture_output=True, text=True, timeout=15)
rows = [line.strip().split(',') for line in inventory.stdout.splitlines() if line.strip()]
valid = (inventory.returncode == 0 and len(rows) == 1 and len(rows[0]) == 3
         and 'RTX PRO 6000' in rows[0][0]
         and rows[0][1].strip().isdigit() and int(rows[0][1]) >= 90000)
receipt = {'run_id': 'marlin-fixed-replay-r2-20261011', 'returncode': inventory.returncode,
           'stdout': inventory.stdout, 'stderr': inventory.stderr, 'valid': valid}
(hardware_dir / 'hardware.json').write_text(json.dumps(receipt, indent=2) + '\n')
print('retry hardware:', json.dumps(receipt), flush=True)
if not valid:
    raise RuntimeError('Retry requires exactly one RTX PRO 6000 with >=90000 MiB; abort before loading')


In [ ]:
import hashlib,json,os,re,shlex,shutil,subprocess,sys,time,zipfile
from pathlib import Path
SESSION_DEADLINE=time.monotonic()+3300
WORKING_DIR=Path('/kaggle/working')
OUT=WORKING_DIR/'marlin-replay';OUT.mkdir(exist_ok=True)
KIT=WORKING_DIR/'diagnostic-kit';KIT.mkdir(exist_ok=False)
def _sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for data in iter(lambda:f.read(8<<20),b''):h.update(data)
    return h.hexdigest()
def run(cmd,*,env=None):
    cmd=list(map(str,cmd));print('$',shlex.join(cmd),flush=True)
    limit=min(900,SESSION_DEADLINE-time.monotonic()-120)
    assert limit>0,'no setup budget remaining'
    done=subprocess.run(cmd,env=env,capture_output=True,text=True,timeout=limit)
    print(done.stdout[-6000:],flush=True)
    if done.returncode:
        print(done.stderr[-6000:],flush=True)
        raise RuntimeError('setup subprocess failed')
    return done
def find_unique(root,name,*,directory=False):
    paths=[p for p in Path(root).rglob(name) if (p.is_dir() if directory else p.is_file())]
    assert len(paths)==1,(name,len(paths))
    return paths[0]
expected_source={'kernel_id': 138024107, 'version': 1, 'slug': 'pranavagrawai/marlin-reduction-replay-v3-20261010', 'files': {'capture/cases.jsonl': '1fa4b6169bc352fe9e20d44f6abbb2f56144dabfe5705e686dd1301396310683', 'server-args.json': '808cdee4a0d8ab849a339898f88106360fdc3f412b51edf3b00d49bb9d56bd43', 'patch-manifest.json': 'f5089fe8755a69bd36919f0381de70221d255dbf66bbdec32426161af5e3a606', 'target-config.json': '0fabfa21fab8bfe69f02234f7ae8df4ed91b785fca2e22011c1230f6a07e5329', 'capture/layer-00-replay-08.pt': 'e825e2860c7d01c17285980888cb9ba4f0fb7435f47e6bdb4d679ab584a436d3', 'capture/layer-00-replay-16.pt': 'be669cc2efb688f0bed90625ead12325cd737ba66f3377718d91cd304b128abb', 'capture/layer-00-replay-32.pt': 'ac61e19f1777c63574c8fd2458943b54e420472c913911597a3a6d7f4bd3a9bc', 'capture/layer-00-weights.pt': 'fbb1f78f7b85641ccf3a8247d258952c85e77d4ea9bfa3d5f0171c0342acb2fb', 'capture/layer-24-replay-08.pt': '8b8ecb68fa8030e8d76e988a07db595be0f824b7c7468771378840ba8fca9b25', 'capture/layer-24-replay-16.pt': 'b4a914e40f3cb1e03c815936e0f6df7b62daab25190e6675f7e465b259e91c9e', 'capture/layer-24-replay-32.pt': '761a3631d5f5e947b09ba9ff36a8bbe7034aa66d3ee7873c615038d7bcb1e69a', 'capture/layer-24-weights.pt': 'cb25283316fbd1ea365d458f449fbcaf5b11eb3a6fe49b653f78560416c08552', 'capture/layer-47-replay-08.pt': 'cafa2feae652a83f59ac46185193f9d030a81e611e1a69205bd91b0b07357dd8', 'capture/layer-47-replay-16.pt': '3bfa0d1c8e24c759a07057222d7f074880dd0697d12489033456a4d7de7fcc20', 'capture/layer-47-replay-32.pt': '9acdcc6a4654f9851a30a578da8322b63e90f6481c83dfec5a3af10490958e6b', 'capture/layer-47-weights.pt': '81e0db8df583abf92db9abb3abbdad3a21a1db6bd3a1d3687fb579edb748c809'}, 'records': [{'case': 'layer-00-replay-08.pt', 'sha256': 'e825e2860c7d01c17285980888cb9ba4f0fb7435f47e6bdb4d679ab584a436d3', 'weights': 'layer-00-weights.pt', 'weights_sha256': 'fbb1f78f7b85641ccf3a8247d258952c85e77d4ea9bfa3d5f0171c0342acb2fb', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 8, 'expert_counts': [[1, 1], [3, 1], [4, 1], [14, 2], [17, 2], [19, 1], [20, 2], [21, 1], [23, 1], [26, 1], [28, 1], [30, 3], [32, 3], [33, 2], [34, 2], [37, 1], [44, 3], [45, 1], [46, 2], [50, 1], [52, 2], [53, 1], [57, 2], [58, 1], [59, 1], [60, 1], [61, 2], [63, 1], [67, 1], [69, 2], [70, 1], [71, 3], [73, 3], [74, 3], [77, 1], [79, 1], [80, 3], [81, 1], [83, 1], [84, 2], [85, 2], [87, 3], [88, 1], [89, 1], [90, 2], [92, 1], [93, 5], [94, 1], [95, 1], [96, 1], [97, 3], [102, 3], [103, 1], [107, 2], [109, 1], [110, 1], [111, 1], [116, 2], [118, 2], [119, 1], [123, 1], [125, 1], [129, 3], [130, 3], [131, 1], [138, 3], [142, 1], [143, 2], [149, 1], [150, 1], [151, 2], [153, 1], [155, 2], [156, 1], [158, 2], [159, 5], [160, 3], [161, 1], [165, 2], [166, 1], [169, 2], [170, 2], [172, 2], [173, 1], [177, 1], [178, 1], [182, 2], [183, 2], [184, 2], [185, 1], [187, 1], [188, 2], [192, 1], [194, 1], [196, 1], [200, 1], [202, 1], [203, 1], [208, 1], [212, 4], [213, 1], [214, 1], [215, 1], [217, 3], [218, 1], [221, 3], [222, 1], [223, 2], [224, 1], [225, 1], [227, 3], [228, 1], [230, 1], [234, 1], [235, 1], [238, 3], [239, 1], [240, 2], [241, 1], [243, 2], [245, 1], [247, 1], [248, 2], [249, 1], [251, 1], [252, 1], [254, 1], [256, 1], [265, 1], [266, 3], [268, 4], [272, 1], [273, 1], [274, 1], [275, 1], [278, 3], [280, 4], [285, 1], [286, 1], [287, 4], [289, 1], [291, 2], [293, 4], [294, 1], [295, 1], [297, 2], [300, 1], [302, 3], [303, 1], [305, 1], [309, 1], [313, 2], [314, 1], [315, 2], [317, 1], [320, 1], [322, 5], [327, 1], [329, 1], [330, 4], [333, 2], [334, 1], [335, 4], [338, 1], [341, 1], [342, 3], [346, 1], [348, 1], [349, 2], [350, 1], [351, 1], [352, 1], [354, 2], [356, 1], [358, 1], [359, 2], [363, 2], [364, 3], [368, 2], [369, 3], [370, 1], [372, 1], [375, 1], [379, 1], [380, 1], [381, 1], [383, 1], [384, 1], [386, 1], [389, 1], [390, 2], [392, 2], [393, 1], [394, 1], [395, 2], [396, 1], [399, 2], [403, 2], [406, 1], [407, 1], [414, 1], [423, 1], [425, 2], [428, 2], [432, 2], [434, 1], [435, 1], [436, 1], [440, 1], [442, 1], [443, 2], [445, 2], [447, 1], [449, 2], [450, 2], [451, 2], [453, 1], [456, 2], [457, 1], [458, 1], [459, 3], [462, 2], [464, 1], [466, 3], [469, 1], [470, 1], [471, 2], [472, 1], [473, 1], [474, 1], [475, 2], [476, 1], [480, 1], [481, 2], [482, 1], [484, 1], [485, 2], [486, 1], [489, 1], [491, 1], [493, 2], [494, 2], [495, 2], [496, 2], [498, 2], [499, 1], [507, 3]]}, {'case': 'layer-24-replay-08.pt', 'sha256': '8b8ecb68fa8030e8d76e988a07db595be0f824b7c7468771378840ba8fca9b25', 'weights': 'layer-24-weights.pt', 'weights_sha256': 'cb25283316fbd1ea365d458f449fbcaf5b11eb3a6fe49b653f78560416c08552', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 8, 'expert_counts': [[3, 3], [6, 1], [14, 1], [15, 1], [17, 1], [19, 1], [27, 1], [30, 2], [31, 2], [32, 1], [49, 4], [54, 1], [59, 1], [60, 2], [64, 3], [66, 4], [67, 5], [68, 3], [70, 3], [72, 1], [73, 1], [75, 2], [77, 1], [78, 4], [80, 3], [81, 5], [86, 1], [87, 2], [89, 1], [92, 1], [97, 1], [100, 2], [103, 1], [108, 11], [109, 2], [111, 1], [112, 2], [116, 3], [123, 2], [126, 4], [127, 1], [132, 1], [133, 4], [134, 9], [138, 3], [143, 1], [144, 2], [145, 6], [148, 2], [150, 2], [154, 4], [156, 1], [158, 2], [159, 1], [171, 1], [174, 2], [177, 1], [178, 4], [183, 1], [184, 5], [189, 3], [194, 1], [198, 7], [202, 5], [204, 3], [205, 2], [206, 1], [215, 3], [216, 2], [220, 7], [221, 2], [223, 3], [225, 1], [227, 1], [228, 3], [229, 2], [232, 2], [233, 1], [236, 1], [239, 4], [241, 1], [249, 5], [250, 1], [256, 1], [260, 1], [261, 5], [270, 3], [272, 1], [273, 1], [274, 2], [275, 2], [276, 1], [292, 2], [301, 5], [303, 1], [304, 1], [309, 4], [313, 1], [315, 1], [322, 1], [324, 1], [327, 1], [328, 1], [330, 4], [331, 1], [334, 1], [340, 3], [346, 4], [347, 3], [350, 6], [353, 5], [356, 1], [357, 2], [360, 2], [363, 2], [365, 2], [375, 4], [376, 4], [377, 3], [383, 3], [387, 6], [389, 3], [390, 1], [396, 2], [399, 1], [400, 1], [402, 1], [403, 6], [404, 4], [405, 3], [412, 2], [418, 1], [419, 4], [424, 2], [427, 7], [429, 16], [436, 1], [442, 1], [447, 1], [451, 2], [454, 1], [455, 1], [465, 1], [466, 1], [468, 1], [471, 5], [474, 1], [475, 1], [478, 2], [484, 1], [489, 2], [497, 2], [499, 16], [500, 1], [503, 1], [505, 3], [509, 4]]}, {'case': 'layer-47-replay-08.pt', 'sha256': 'cafa2feae652a83f59ac46185193f9d030a81e611e1a69205bd91b0b07357dd8', 'weights': 'layer-47-weights.pt', 'weights_sha256': '81e0db8df583abf92db9abb3abbdad3a21a1db6bd3a1d3687fb579edb748c809', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 8, 'expert_counts': [[4, 4], [9, 11], [12, 2], [13, 1], [16, 1], [17, 2], [20, 1], [25, 1], [26, 2], [27, 3], [28, 1], [29, 3], [31, 1], [33, 2], [36, 2], [37, 6], [49, 1], [50, 3], [55, 2], [56, 1], [60, 1], [61, 3], [64, 2], [66, 4], [70, 5], [73, 1], [75, 1], [76, 1], [78, 1], [79, 3], [80, 2], [85, 1], [88, 4], [89, 4], [90, 6], [93, 8], [97, 1], [98, 1], [99, 1], [100, 2], [102, 1], [106, 2], [108, 2], [112, 3], [117, 1], [119, 1], [120, 1], [125, 2], [126, 2], [127, 1], [128, 2], [131, 1], [132, 1], [136, 1], [137, 1], [140, 1], [142, 1], [143, 3], [144, 1], [149, 2], [150, 1], [151, 2], [152, 5], [155, 2], [156, 1], [166, 4], [168, 2], [169, 5], [171, 1], [173, 2], [181, 1], [185, 5], [188, 3], [189, 2], [202, 1], [203, 2], [204, 3], [205, 1], [209, 1], [210, 1], [217, 1], [218, 4], [220, 2], [221, 6], [225, 1], [227, 1], [231, 1], [232, 5], [233, 4], [234, 2], [235, 2], [242, 16], [247, 1], [249, 1], [254, 2], [260, 4], [265, 1], [270, 1], [272, 4], [273, 1], [277, 2], [280, 1], [281, 1], [284, 1], [296, 1], [298, 1], [299, 3], [303, 3], [305, 1], [307, 2], [317, 1], [319, 1], [320, 1], [323, 6], [324, 1], [326, 1], [328, 1], [332, 1], [334, 2], [336, 1], [338, 2], [340, 3], [342, 1], [344, 1], [345, 2], [346, 5], [348, 2], [351, 2], [353, 1], [356, 4], [357, 1], [359, 1], [360, 1], [370, 2], [372, 2], [376, 1], [379, 3], [381, 3], [385, 1], [387, 1], [390, 4], [393, 2], [395, 1], [398, 7], [399, 1], [400, 3], [401, 2], [403, 1], [408, 1], [410, 1], [412, 3], [413, 1], [414, 4], [416, 4], [417, 1], [420, 3], [425, 1], [431, 1], [433, 1], [438, 2], [442, 2], [444, 2], [445, 1], [451, 2], [452, 1], [454, 5], [457, 5], [458, 1], [460, 1], [464, 1], [467, 1], [469, 6], [470, 3], [472, 1], [479, 1], [485, 3], [491, 1], [492, 1], [494, 2], [496, 1], [497, 3], [502, 1], [506, 1], [508, 1], [509, 1]]}, {'case': 'layer-00-replay-16.pt', 'sha256': 'be669cc2efb688f0bed90625ead12325cd737ba66f3377718d91cd304b128abb', 'weights': 'layer-00-weights.pt', 'weights_sha256': 'fbb1f78f7b85641ccf3a8247d258952c85e77d4ea9bfa3d5f0171c0342acb2fb', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 16, 'expert_counts': [[0, 1], [4, 2], [5, 1], [12, 2], [14, 1], [15, 1], [16, 3], [17, 1], [19, 1], [20, 2], [21, 1], [24, 1], [27, 6], [28, 1], [30, 2], [31, 1], [32, 2], [33, 3], [34, 2], [35, 2], [36, 1], [43, 1], [44, 3], [46, 2], [53, 1], [54, 1], [57, 2], [60, 2], [61, 3], [63, 2], [65, 1], [69, 2], [70, 1], [71, 1], [73, 1], [74, 3], [77, 1], [80, 2], [84, 2], [85, 3], [87, 5], [93, 4], [95, 2], [102, 1], [105, 1], [106, 1], [107, 5], [108, 2], [110, 1], [111, 2], [126, 1], [128, 2], [129, 2], [130, 1], [131, 1], [132, 1], [133, 1], [138, 3], [142, 1], [143, 3], [144, 1], [145, 2], [147, 2], [148, 1], [151, 1], [155, 3], [156, 2], [158, 1], [159, 6], [160, 1], [166, 1], [168, 1], [170, 1], [172, 1], [173, 1], [174, 2], [183, 1], [188, 1], [189, 3], [190, 1], [192, 1], [200, 2], [201, 1], [202, 1], [205, 1], [209, 1], [211, 1], [212, 2], [217, 2], [222, 3], [223, 1], [224, 1], [227, 3], [228, 2], [233, 1], [235, 1], [237, 1], [238, 4], [239, 2], [240, 3], [241, 1], [243, 1], [246, 1], [248, 3], [251, 2], [252, 1], [253, 1], [254, 1], [257, 1], [258, 1], [264, 1], [265, 1], [266, 7], [268, 4], [270, 1], [272, 1], [273, 1], [274, 2], [275, 3], [278, 1], [280, 7], [281, 1], [283, 1], [286, 1], [287, 5], [288, 1], [289, 2], [290, 1], [293, 1], [294, 2], [295, 1], [297, 4], [300, 1], [302, 1], [305, 1], [308, 4], [312, 1], [313, 2], [314, 1], [316, 1], [317, 2], [319, 1], [320, 1], [321, 1], [322, 3], [325, 1], [327, 1], [329, 1], [334, 1], [335, 1], [337, 1], [338, 1], [339, 3], [340, 1], [341, 1], [342, 1], [346, 1], [347, 2], [350, 3], [351, 1], [352, 2], [354, 1], [356, 2], [359, 1], [362, 1], [363, 2], [364, 4], [366, 1], [367, 3], [369, 1], [370, 1], [371, 1], [375, 1], [378, 2], [382, 1], [383, 2], [384, 3], [386, 1], [389, 2], [390, 2], [392, 1], [393, 1], [399, 1], [403, 3], [406, 1], [407, 1], [409, 1], [412, 1], [416, 1], [423, 2], [425, 1], [426, 1], [427, 3], [430, 1], [431, 1], [432, 2], [436, 1], [437, 1], [439, 1], [440, 1], [442, 1], [444, 2], [445, 1], [446, 1], [447, 1], [449, 3], [450, 1], [451, 1], [456, 2], [457, 1], [458, 1], [461, 2], [462, 3], [463, 1], [468, 1], [470, 1], [471, 2], [472, 1], [474, 1], [475, 2], [480, 1], [481, 2], [482, 1], [485, 1], [486, 1], [491, 3], [493, 3], [494, 6], [496, 4], [498, 1], [499, 1], [500, 3], [503, 1], [507, 2], [508, 1]]}, {'case': 'layer-24-replay-16.pt', 'sha256': 'b4a914e40f3cb1e03c815936e0f6df7b62daab25190e6675f7e465b259e91c9e', 'weights': 'layer-24-weights.pt', 'weights_sha256': 'cb25283316fbd1ea365d458f449fbcaf5b11eb3a6fe49b653f78560416c08552', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 16, 'expert_counts': [[3, 1], [6, 1], [30, 1], [32, 1], [35, 1], [39, 1], [40, 1], [41, 1], [44, 4], [48, 2], [49, 3], [59, 2], [60, 1], [64, 2], [67, 4], [70, 2], [75, 3], [78, 6], [79, 1], [81, 5], [82, 2], [85, 4], [86, 3], [87, 1], [88, 2], [93, 1], [108, 6], [109, 1], [110, 1], [111, 3], [112, 1], [116, 3], [122, 1], [123, 2], [126, 5], [127, 1], [130, 1], [132, 1], [133, 9], [134, 10], [136, 2], [138, 5], [143, 3], [144, 2], [145, 4], [146, 1], [147, 2], [148, 2], [153, 1], [154, 5], [155, 1], [156, 1], [158, 2], [159, 3], [161, 1], [174, 1], [175, 1], [178, 5], [181, 1], [184, 3], [189, 3], [198, 6], [202, 6], [204, 5], [205, 5], [213, 3], [215, 1], [216, 1], [220, 6], [221, 3], [223, 3], [224, 1], [225, 2], [226, 1], [229, 3], [230, 2], [232, 4], [233, 5], [236, 2], [248, 1], [249, 1], [260, 1], [261, 4], [264, 1], [274, 1], [275, 2], [281, 2], [298, 2], [301, 1], [303, 1], [304, 2], [309, 4], [315, 4], [319, 1], [322, 1], [324, 3], [330, 2], [334, 2], [335, 1], [340, 3], [347, 2], [350, 2], [353, 1], [356, 5], [357, 5], [360, 4], [363, 1], [366, 4], [375, 9], [376, 6], [378, 1], [382, 1], [383, 3], [387, 8], [389, 4], [390, 1], [392, 1], [400, 1], [401, 1], [403, 4], [404, 6], [405, 1], [408, 1], [412, 2], [414, 3], [418, 4], [419, 3], [420, 1], [421, 1], [427, 3], [429, 9], [430, 1], [431, 1], [436, 3], [437, 1], [438, 1], [442, 2], [443, 6], [448, 2], [450, 1], [451, 2], [459, 1], [464, 2], [466, 1], [467, 1], [469, 4], [470, 2], [472, 2], [484, 2], [487, 1], [491, 1], [497, 1], [499, 11], [500, 1], [503, 2], [505, 1]]}, {'case': 'layer-47-replay-16.pt', 'sha256': '3bfa0d1c8e24c759a07057222d7f074880dd0697d12489033456a4d7de7fcc20', 'weights': 'layer-47-weights.pt', 'weights_sha256': '81e0db8df583abf92db9abb3abbdad3a21a1db6bd3a1d3687fb579edb748c809', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 16, 'expert_counts': [[2, 1], [4, 7], [9, 13], [11, 1], [12, 3], [17, 3], [20, 1], [27, 2], [28, 1], [29, 6], [31, 1], [33, 1], [36, 2], [37, 6], [39, 2], [50, 7], [55, 1], [61, 1], [64, 7], [65, 1], [66, 7], [70, 3], [73, 1], [75, 2], [76, 2], [78, 5], [80, 4], [88, 1], [89, 4], [90, 9], [93, 5], [97, 2], [99, 1], [100, 2], [106, 1], [108, 4], [112, 5], [119, 2], [123, 1], [125, 1], [126, 1], [127, 3], [132, 1], [135, 1], [137, 1], [142, 1], [143, 1], [144, 2], [145, 1], [149, 1], [150, 1], [152, 4], [154, 2], [155, 1], [166, 2], [168, 1], [171, 2], [174, 1], [175, 1], [184, 1], [188, 1], [195, 1], [202, 1], [203, 3], [205, 2], [210, 2], [217, 4], [218, 2], [220, 2], [221, 1], [222, 1], [231, 3], [232, 1], [233, 3], [234, 3], [235, 1], [237, 1], [239, 2], [242, 8], [249, 1], [260, 5], [269, 1], [270, 1], [272, 2], [273, 2], [276, 1], [278, 2], [280, 1], [281, 1], [284, 1], [289, 1], [298, 1], [299, 4], [303, 2], [304, 2], [305, 2], [313, 1], [314, 1], [315, 2], [318, 3], [320, 1], [321, 1], [323, 4], [324, 1], [326, 2], [333, 1], [334, 1], [336, 2], [337, 1], [338, 4], [340, 2], [342, 1], [344, 2], [345, 2], [346, 3], [348, 1], [349, 1], [350, 1], [351, 1], [354, 1], [357, 2], [358, 2], [360, 1], [370, 4], [372, 2], [377, 1], [379, 4], [381, 2], [384, 1], [390, 5], [398, 3], [407, 2], [410, 3], [412, 1], [414, 2], [415, 1], [416, 5], [417, 1], [419, 1], [420, 1], [425, 5], [427, 1], [433, 2], [439, 1], [442, 6], [446, 1], [448, 2], [454, 9], [456, 1], [457, 4], [458, 1], [459, 1], [460, 1], [464, 1], [467, 1], [469, 10], [470, 2], [472, 1], [473, 1], [479, 3], [481, 2], [485, 13], [487, 2], [491, 5], [494, 2], [496, 1], [497, 2], [501, 2], [508, 2], [509, 1]]}, {'case': 'layer-00-replay-32.pt', 'sha256': 'ac61e19f1777c63574c8fd2458943b54e420472c913911597a3a6d7f4bd3a9bc', 'weights': 'layer-00-weights.pt', 'weights_sha256': 'fbb1f78f7b85641ccf3a8247d258952c85e77d4ea9bfa3d5f0171c0342acb2fb', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 32, 'expert_counts': [[1, 1], [2, 2], [4, 1], [5, 1], [9, 1], [12, 3], [17, 2], [19, 1], [20, 1], [22, 1], [24, 1], [28, 1], [31, 1], [32, 1], [33, 2], [34, 3], [35, 4], [37, 2], [42, 1], [44, 5], [48, 3], [57, 2], [59, 1], [60, 1], [61, 1], [63, 3], [67, 1], [68, 1], [69, 3], [71, 1], [72, 1], [74, 2], [75, 1], [79, 2], [80, 1], [81, 1], [83, 2], [84, 2], [85, 3], [87, 1], [88, 2], [90, 1], [91, 2], [93, 5], [94, 2], [95, 2], [98, 1], [102, 1], [103, 1], [106, 1], [107, 3], [108, 3], [111, 1], [116, 1], [119, 3], [122, 1], [123, 1], [125, 2], [126, 2], [128, 1], [130, 1], [131, 1], [136, 1], [138, 2], [142, 3], [144, 1], [147, 1], [148, 2], [149, 2], [151, 1], [153, 1], [155, 1], [159, 3], [160, 1], [162, 2], [165, 1], [166, 2], [167, 1], [168, 1], [169, 2], [170, 1], [172, 1], [174, 2], [178, 1], [179, 3], [180, 2], [183, 1], [184, 1], [185, 1], [188, 1], [192, 1], [200, 1], [204, 1], [205, 1], [211, 1], [212, 4], [213, 1], [214, 2], [217, 1], [218, 1], [219, 2], [223, 2], [224, 1], [226, 2], [227, 1], [228, 1], [230, 2], [233, 1], [234, 1], [235, 1], [238, 3], [239, 2], [240, 2], [241, 3], [242, 2], [243, 1], [245, 1], [247, 3], [248, 2], [251, 3], [255, 1], [256, 1], [259, 1], [261, 2], [262, 1], [265, 2], [266, 2], [268, 3], [269, 1], [272, 1], [274, 1], [275, 2], [277, 1], [278, 2], [280, 1], [282, 3], [283, 3], [285, 1], [286, 1], [287, 3], [291, 1], [294, 3], [295, 2], [297, 4], [299, 1], [302, 2], [303, 1], [305, 1], [308, 1], [309, 1], [316, 2], [319, 1], [320, 2], [329, 1], [330, 1], [332, 1], [333, 2], [335, 4], [339, 1], [342, 1], [346, 1], [347, 1], [352, 3], [353, 2], [354, 1], [356, 1], [364, 3], [367, 3], [368, 1], [371, 1], [376, 1], [377, 1], [379, 1], [380, 1], [381, 2], [384, 7], [386, 1], [392, 2], [393, 1], [394, 1], [395, 1], [399, 4], [401, 1], [403, 3], [406, 1], [407, 1], [409, 1], [411, 1], [412, 1], [414, 1], [416, 3], [419, 1], [426, 1], [428, 2], [431, 1], [432, 2], [434, 1], [436, 2], [438, 1], [439, 2], [443, 2], [444, 3], [445, 1], [446, 1], [448, 1], [449, 1], [456, 2], [457, 1], [458, 4], [459, 2], [461, 3], [462, 1], [464, 2], [465, 1], [468, 1], [469, 2], [470, 6], [471, 2], [472, 1], [473, 3], [474, 2], [475, 1], [476, 1], [483, 1], [485, 5], [491, 3], [493, 4], [494, 3], [496, 2], [498, 1], [499, 2], [501, 1], [503, 2], [504, 1], [505, 1]]}, {'case': 'layer-24-replay-32.pt', 'sha256': '761a3631d5f5e947b09ba9ff36a8bbe7034aa66d3ee7873c615038d7bcb1e69a', 'weights': 'layer-24-weights.pt', 'weights_sha256': 'cb25283316fbd1ea365d458f449fbcaf5b11eb3a6fe49b653f78560416c08552', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 32, 'expert_counts': [[3, 1], [4, 1], [6, 2], [10, 1], [13, 2], [14, 2], [17, 1], [19, 1], [26, 1], [27, 3], [30, 1], [31, 1], [32, 1], [46, 3], [49, 1], [52, 1], [59, 1], [64, 2], [66, 1], [67, 2], [68, 2], [70, 1], [73, 2], [75, 3], [78, 8], [80, 2], [81, 2], [85, 3], [86, 1], [87, 4], [92, 3], [97, 2], [108, 2], [109, 2], [110, 1], [112, 1], [113, 2], [116, 7], [117, 1], [126, 4], [127, 1], [130, 2], [132, 2], [133, 3], [134, 2], [136, 1], [138, 3], [143, 4], [146, 1], [147, 3], [148, 1], [149, 1], [151, 1], [154, 3], [157, 1], [158, 1], [159, 2], [162, 4], [163, 1], [167, 1], [171, 1], [175, 2], [177, 2], [178, 3], [182, 1], [184, 2], [189, 8], [198, 2], [200, 1], [202, 3], [203, 1], [204, 1], [205, 3], [206, 1], [210, 1], [216, 5], [217, 1], [220, 5], [221, 3], [223, 5], [224, 1], [227, 3], [228, 3], [229, 1], [233, 3], [236, 3], [239, 2], [241, 3], [242, 1], [243, 1], [244, 1], [245, 1], [248, 2], [249, 3], [252, 1], [256, 2], [259, 1], [261, 3], [262, 3], [268, 1], [270, 1], [276, 1], [280, 2], [298, 2], [300, 1], [301, 1], [303, 4], [304, 1], [309, 1], [316, 2], [319, 3], [322, 1], [325, 1], [327, 1], [330, 1], [331, 1], [332, 4], [335, 1], [340, 1], [347, 1], [350, 4], [353, 4], [356, 2], [357, 1], [360, 1], [361, 3], [363, 3], [370, 2], [372, 1], [375, 6], [376, 3], [377, 2], [380, 1], [382, 2], [383, 3], [386, 1], [387, 3], [389, 8], [394, 1], [399, 1], [402, 1], [403, 5], [404, 3], [405, 4], [418, 1], [419, 3], [424, 2], [425, 2], [427, 2], [429, 13], [436, 4], [437, 4], [440, 1], [447, 2], [451, 5], [453, 1], [455, 2], [458, 3], [464, 1], [465, 1], [467, 2], [469, 5], [470, 3], [472, 2], [474, 1], [478, 1], [482, 1], [486, 3], [491, 2], [493, 1], [496, 1], [497, 8], [499, 8], [500, 1], [503, 1], [505, 2], [509, 5]]}, {'case': 'layer-47-replay-32.pt', 'sha256': '9acdcc6a4654f9851a30a578da8322b63e90f6481c83dfec5a3af10490958e6b', 'weights': 'layer-47-weights.pt', 'weights_sha256': '81e0db8df583abf92db9abb3abbdad3a21a1db6bd3a1d3687fb579edb748c809', 'graph_key': 'ShapeKey(size=10, stream_idx=None, variant_label=None, dsa_variant=None)', 'raw_bs': 10, 'rows': 40, 'hidden_shape': [40, 2560], 'route_shape': [40, 10], 'mode': 'TARGET_VERIFY', 'replay_ordinal': 32, 'expert_counts': [[0, 2], [2, 2], [4, 1], [9, 14], [11, 1], [20, 2], [27, 1], [29, 4], [31, 2], [33, 1], [36, 3], [37, 6], [39, 2], [48, 1], [49, 2], [50, 1], [54, 1], [55, 5], [61, 3], [63, 1], [64, 2], [65, 1], [66, 6], [70, 2], [72, 1], [73, 1], [76, 1], [78, 3], [79, 3], [80, 6], [88, 1], [89, 3], [90, 2], [93, 4], [94, 1], [95, 3], [96, 2], [99, 1], [100, 3], [108, 5], [111, 2], [112, 8], [117, 2], [120, 3], [122, 4], [125, 1], [126, 1], [127, 3], [137, 1], [142, 1], [143, 3], [144, 1], [149, 6], [150, 1], [152, 7], [154, 1], [166, 4], [171, 5], [175, 2], [178, 3], [186, 4], [188, 1], [195, 1], [199, 2], [201, 2], [203, 4], [204, 2], [209, 3], [210, 1], [213, 1], [217, 1], [218, 2], [222, 1], [225, 1], [227, 1], [228, 1], [232, 2], [233, 4], [234, 1], [242, 14], [248, 2], [250, 1], [252, 1], [254, 2], [260, 5], [264, 3], [265, 2], [270, 1], [272, 3], [277, 1], [278, 1], [280, 1], [281, 3], [285, 1], [287, 1], [292, 1], [296, 2], [298, 2], [299, 2], [301, 1], [304, 1], [305, 4], [313, 1], [315, 3], [320, 2], [321, 1], [323, 5], [326, 1], [330, 1], [333, 1], [336, 3], [337, 3], [338, 1], [340, 5], [344, 1], [346, 5], [348, 4], [349, 2], [356, 2], [360, 1], [367, 1], [369, 1], [370, 2], [379, 3], [387, 1], [390, 6], [393, 1], [395, 2], [398, 9], [399, 2], [400, 2], [401, 1], [410, 2], [411, 1], [412, 2], [413, 1], [414, 2], [416, 5], [417, 2], [424, 3], [425, 2], [427, 4], [433, 1], [442, 1], [444, 1], [445, 2], [446, 1], [454, 6], [458, 1], [461, 1], [463, 2], [469, 10], [470, 2], [472, 1], [479, 3], [480, 2], [485, 4], [487, 2], [492, 2], [497, 2], [501, 1], [502, 2], [508, 1]]}], 'source_identity': {'wheel_sha256': 'd0620216732b82b6efd4f8a78cc9b5f55343a4d3837fdfa3bcbb841b7718a225', 'members': {'sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py': 'fab293ee5aaa0d285d161c3bbf23606d2908fe3273eae5f32eb5b49db52f81e4', 'sglang/srt/hardware_backend/gpu/quantization/gptq_kernels.py': '375085abdd2f4abca3d5831045d5f11d012c1877e0a19f79ebe89593b5568370', 'sglang/kernels/ops/moe/moe_wna16_marlin.py': 'a34fa1daf24a6286d5d027c6ae45dac5fea144abd606a65b722fecd091513398', 'sglang/kernels/jit/csrc/gemm/marlin_moe/marlin_template.h': '6b43bc72d64a591bc86f62df2db14bdea5adce7b70b00af1a3e601a3c0d24fdb', 'sglang/srt/layers/moe/moe_runner/marlin.py': 'e1db2b78559fd7ed482710f0b95104339ebcd41721a3cb85e17fbc63ab683734', 'sglang/srt/layers/quantization/marlin_utils.py': '1f0529ca2b24e2af804eaaf51fe72d76484cf5c7ff91a768612c5c9ddfba0bf3', 'sglang/srt/utils/custom_op.py': 'b427fdae1e11dfd9f7ba72cf5fa3e2df4b1ea9ecdf7e769cbd37567240ac8d23', 'sglang/kernels/jit/csrc/gemm/marlin/gptq_marlin_repack.cuh': 'baf9c493860b8f2d7af160c82a65b5b963f1da77afb02adfeb9b266435c72aaa', 'sglang/srt/layers/quantization/gptq/gptq.py': '0daebc972a53d42cf289bde29423c04a76d095fd3149a1ed17bbeeee981e8ee6', 'sglang/srt/layers/quantization/gptq/schemes/gptq_moe.py': '47e65140e2480a8637b9ea5bb746ff77e9c09c01e448fd2056c0b485e4f3a10a', 'sglang/srt/model_executor/runner/decode_cuda_graph_runner.py': 'ed359b9f3a0ad36bf290c4e71237fa9103e2b6ba82b28f4ef07ad2fe23e61140', 'sglang/srt/model_executor/model_runner.py': '8d84ad370871e4073973922d3b0cf4231bd3b4c793c0275e3c3b11370eba57a8', 'sglang/test/test_marlin_utils.py': 'e9911ab643ba2b793959fa4d0bf35f7f1acd490761e394dbf72fea0375379e2a'}}}
candidates=[p.parent.parent for p in Path('/kaggle/input').rglob('cases.jsonl') if p.parent.name=='capture']
assert len(candidates)==1,('expected one mounted capture',len(candidates))
SOURCE=candidates[0]
readback={name:_sha(SOURCE/name) for name in expected_source['files']}
assert readback==expected_source['files'],'mounted capture differs from accepted v3 version'
(OUT/'capture-source.json').write_text(json.dumps({'kernel_id':expected_source['kernel_id'],'version':expected_source['version'],'files':readback},indent=2))
for name in ['server-args.json','target-config.json']:
    shutil.copy2(SOURCE/name,OUT/name)
WHEELHOUSE_DIR=find_unique('/kaggle/input','requirements.lock').parent
PREFIX='/tmp/marlin-fixed';VENV=PREFIX+'/venv';PYTHON=VENV+'/bin/python';SGLANG=VENV+'/bin/sglang'
"""Verify the exact packaged wheel and installed source without importing SGLang."""
import hashlib
from pathlib import Path
import zipfile


def package_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def verify_wheel(wheel, contract):
    wheel = Path(wheel)
    expected = contract['wheel']
    if wheel.name != expected['filename'] or package_sha256(wheel) != expected['sha256']:
        raise ValueError('packaged wheel identity mismatch')
    with zipfile.ZipFile(wheel) as archive:
        members = archive.namelist()
        for relative, digest in contract['files'].items():
            member = 'sglang/' + relative
            if members.count(member) != 1 or hashlib.sha256(archive.read(member)).hexdigest() != digest:
                raise ValueError('wheel source mismatch: ' + relative)
    return expected['sha256']


def verify_installed(root, contract):
    for relative, digest in contract['files'].items():
        if package_sha256(Path(root) / relative) != digest:
            raise ValueError('installed source mismatch: ' + relative)
    return contract['files']


def verify_bundle(wheel_directory, lock_path, contract):
    wheels = sorted(Path(wheel_directory).glob('sglang-*.whl'))
    if len(wheels) != 1:
        raise ValueError('expected one packaged SGLang wheel')
    verify_wheel(wheels[0], contract)
    if package_sha256(lock_path) != contract['wheel']['requirements_lock_sha256']:
        raise ValueError('package requirements lock mismatch')
    return contract['wheel']

contract={'commit': 'd00d88efc8d6281b12be4f4073126aec95038c55', 'files': {'srt/observability/metrics_collector.py': '6400f15a91a52b96d8a206b8794d1aaf4ab71c4c5d9ae1c7027aa651d6ff9591', 'srt/mem_cache/kv_cache_configurator.py': '1751fd5464932ce1eadbdca19f069bbdc354b429432b43aabfc3e1fcb776dae7', 'srt/managers/scheduler_components/metrics_reporter.py': '6149a4ae4ebc2318517f8583915255de4f3549064d4fd5a4caaf015a48aafc02', 'srt/managers/scheduler_components/profiler_manager.py': '8e4a29923065c2c674151e94d1d4355ee295f1fca0104d36d5de70e5f7e22fe2', 'srt/managers/scheduler_components/batch_result_processor.py': '8c9626605546de2bc54d5f25a7ae16caeb9ec96e273af50adf91076df543e05b', 'srt/managers/scheduler.py': 'a1209592d8146328c11f64e338d3448438428260ab9c45e1702347663d917ffd', 'srt/entrypoints/http_server.py': '833d5da3572305a96df3607833256ee31f6e7a03d7ea91c260a8bde076846ca6'}, 'provenance': {'qualified_utc': '2026-10-08T18:04:26.628930+00:00', 'observed_kernel': 'pranavagrawai/serving-headroom-v2-20261008/1', 'observed_identity_sha256': 'c86b4cf3c8c7419f8dc4e8f80eaf2d1a670be4654b3943b89dd2f3ac0917fa25', 'changed_from_git_fixture': {'srt/mem_cache/kv_cache_configurator.py': {'local_git': 'a759ae971d6da987cf11bf81de85d6f67d577fc487bb52bf241c4019a825a138', 'packaged': '1751fd5464932ce1eadbdca19f069bbdc354b429432b43aabfc3e1fcb776dae7'}}}, 'source_basis': 'exact wheel matched against v2 installed runtime', 'wheel': {'dataset': 'dfranzen/pennyroyal-v253', 'filename': 'sglang-0.5.19+gd00d88efc8d6-cp312-cp312-linux_x86_64.whl', 'sha256': 'd0620216732b82b6efd4f8a78cc9b5f55343a4d3837fdfa3bcbb841b7718a225', 'bytes': 24833268, 'requirements_lock_sha256': '4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02'}}
verify_bundle(Path(WHEELHOUSE_DIR)/'wheels',Path(WHEELHOUSE_DIR)/'requirements.lock',contract)
(OUT/'package-identity.json').write_text(json.dumps(contract['wheel'],indent=2))
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)



In [ ]:
modules={'benchmark.py': '"""GPU worker, only after separate launch authority. Capture files are local/trusted."""\nimport argparse\nimport contextlib\nimport types\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nfrom reference import dequantize, errors, numerical_gate, paired_summary, qualify_outputs, unpack_marlin, unpermute_scales\n\n\ndef cpu_array(tensor):\n    return tensor.detach().float().cpu().numpy()\n\n\ndef fingerprint(tensor):\n    return hashlib.sha256(tensor.detach().contiguous().view(__import__(\'torch\').uint8).cpu().numpy().tobytes()).hexdigest()\n\n\ndef verify_pack_sentinel():\n    import torch\n    from sglang.kernels.ops.quantization.gptq_marlin_repack import gptq_marlin_repack\n    from sglang.srt.layers.quantization.marlin_utils import marlin_permute_scales\n    # Known integers, not values recovered from GEMM outputs; every W4 code appears.\n    rng = np.random.default_rng(20261009)\n    q = rng.integers(0,16,size=(128,128),dtype=np.int32)\n    packed = np.zeros((16,128),dtype=np.uint32)\n    for i in range(8): packed |= q[i::8].astype(np.uint32) << (4*i)\n    repacked = gptq_marlin_repack(torch.from_numpy(packed.view(np.int32)).cuda(), torch.empty(0,dtype=torch.int32,device=\'cuda\'),128,128,4)\n    np.testing.assert_array_equal(unpack_marlin(repacked.cpu().numpy(),128,128),q-8)\n    for groups in [1,2]:\n        original = torch.arange(groups*128,dtype=torch.float32,device=\'cuda\').reshape(groups,128)\n        result = marlin_permute_scales(original,128,128,128//groups if groups>1 else -1)\n        np.testing.assert_array_equal(unpermute_scales(cpu_array(result),128),cpu_array(original))\n    return {\'integer_codes\':16,\'shape\':[128,128],\'scale_groups\':[1,2],\'passed\':True}\n\n\ndef fp32_reference(sample, weights):\n    import torch\n    x = sample[\'hidden_states\'].cuda().float()\n    ids = sample[\'topk_ids\'].cuda().long()\n    route = sample[\'topk_weights\'].cuda().float()\n    output = torch.zeros_like(x)\n    k = x.shape[1]; n = weights[\'w2\'].shape[1]*16\n    for expert in torch.unique(ids).tolist():\n        if expert < 0 or expert >= weights[\'w1\'].shape[0]:\n            raise ValueError(\'Invalid local expert id\')\n        tokens, slots = torch.where(ids == expert)\n        w1 = torch.from_numpy(dequantize(weights[\'w1\'][expert].numpy(),cpu_array(weights[\'w1_scale\'][expert]),k,2*n)).cuda()\n        w2 = torch.from_numpy(dequantize(weights[\'w2\'][expert].numpy(),cpu_array(weights[\'w2_scale\'][expert]),n,k)).cuda()\n        intermediate = x[tokens] @ w1\n        activated = torch.nn.functional.silu(intermediate[:,:n])*intermediate[:,n:]\n        values = (activated @ w2)*route[tokens,slots,None]\n        output.index_add_(0,tokens,values)\n    return output * (sample[\'routed_scaling_factor\'] if sample[\'routed_scaling_factor\'] is not None else 1.)\n\n\ndef dtype_reference(sample, weights):\n    """Diagnostic BF16 boundaries; does not model split-K or atomic order."""\n    import torch\n    x=sample[\'hidden_states\'].cuda().float()\n    ids=sample[\'topk_ids\'].cuda().long()\n    route=sample[\'topk_weights\'].cuda().bfloat16().float()\n    k=x.shape[1];n=weights[\'w2\'].shape[1]*16\n    values=torch.zeros((*ids.shape,k),device=x.device,dtype=torch.float32)\n    for expert in torch.unique(ids).tolist():\n        tokens,slots=torch.where(ids==expert)\n        w1=torch.from_numpy(dequantize(weights[\'w1\'][expert].numpy(),cpu_array(weights[\'w1_scale\'][expert]),k,2*n)).cuda().bfloat16().float()\n        w2=torch.from_numpy(dequantize(weights[\'w2\'][expert].numpy(),cpu_array(weights[\'w2_scale\'][expert]),n,k)).cuda().bfloat16().float()\n        first=(x[tokens]@w1).bfloat16().float()\n        activated=(torch.nn.functional.silu(first[:,:n])*first[:,n:]).bfloat16().float()\n        second=(activated@w2).bfloat16().float()\n        values[tokens,slots]=(second*route[tokens,slots,None]).bfloat16().float()\n    result=torch.zeros_like(x)\n    for slot in range(ids.shape[1]):result+=values[:,slot]\n    return (result*(sample[\'routed_scaling_factor\'] or 1.)).bfloat16().float()\n\n\ndef repeatability_metrics(baseline, treatment, independent_control, fp32):\n    """Fixed contrasts chosen before this run; envelope is descriptive only."""\n    within_base=[errors(x,baseline[0]) for x in baseline[1:]]\n    within_treatment=[errors(x,treatment[0]) for x in treatment[1:]]\n    control=[errors(x,baseline[0]) for x in independent_control]\n    drift=[errors(t,b) for b,t in zip(baseline,treatment)]\n    pairs=within_base+control\n    max_control=max(m[\'max_abs\'] for m in pairs)\n    rms_control=max(m[\'rms\'] for m in pairs)\n    return {\'baseline_repeats\':within_base,\'treatment_repeats\':within_treatment,\n            \'independent_control_graph\':control,\'paired_treatment_control\':drift,\n            \'control_max_abs_envelope\':max_control,\'control_rms_envelope\':rms_control,\n            \'max_cross_arm_exceeds_control_envelope\':max(m[\'max_abs\'] for m in drift)>max_control,\n            \'rms_cross_arm_exceeds_control_envelope\':max(m[\'rms\'] for m in drift)>rms_control,\n            \'baseline_vs_fp32\':[errors(x,fp32) for x in baseline],\n            \'treatment_vs_fp32\':[errors(x,fp32) for x in treatment],\n            \'limits\':[\'finite repeat count is not a worst-case bound\',\'comparison is descriptive and grants no numerical acceptance\']}\n\n\ndef callable_for(sample, weights, override):\n    import torch\n    from sglang.srt.layers.moe.fused_moe_triton.fused_marlin_moe import fused_marlin_moe\n    from sglang.srt.layers.quantization.marlin_utils import marlin_make_workspace\n    kwargs = {**weights,**sample,\'use_atomic_add_override\':override,\'experiment_capture_tag\':None}\n    kwargs = {k: v.cuda() if isinstance(v,torch.Tensor) else v for k,v in kwargs.items()}\n    def run():\n        return fused_marlin_moe(**kwargs,workspace=marlin_make_workspace(kwargs[\'hidden_states\'].device,max_blocks_per_sm=4))\n    return run, kwargs\n\n\ndef capture_graph(fn):\n    import torch\n    stream = torch.cuda.Stream()\n    stream.wait_stream(torch.cuda.current_stream())\n    with torch.cuda.stream(stream):\n        for _ in range(20): fn()\n    torch.cuda.current_stream().wait_stream(stream)\n    torch.cuda.synchronize()\n    graph = torch.cuda.CUDAGraph()\n    before = torch.cuda.memory_allocated()\n    with torch.cuda.graph(graph): output = fn()\n    torch.cuda.synchronize()\n    return graph, output, torch.cuda.memory_allocated()-before\n\n\ndef time_calls(fn, repeats=64):\n    import torch\n    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n    torch.cuda.synchronize()\n    start.record()\n    for _ in range(repeats): fn()\n    end.record(); end.synchronize()\n    return start.elapsed_time(end)/repeats\n\n\ndef timed_pairs(base, treatment, path):\n    import torch\n    for _ in range(20):\n        base(); treatment()\n    torch.cuda.synchronize()\n    pairs = []\n    with path.open(\'x\') as stream:\n        for i in range(30):\n            order = [\'baseline\',\'treatment\'] if i%2 == 0 else [\'treatment\',\'baseline\']\n            values = {}\n            for arm in order: values[arm] = time_calls(base if arm==\'baseline\' else treatment)\n            record = {\'pair\':i,\'order\':order,\'repeats\':64,\'ms_per_call\':values,\'monotonic\':time.monotonic()}\n            stream.write(json.dumps(record)+\'\\n\');stream.flush()\n            pairs.append([values[\'baseline\'],values[\'treatment\']])\n    return paired_summary(pairs)\n\n\n\n@contextlib.contextmanager\ndef poison_actual_scratch():\n    import importlib\n    module=importlib.import_module(\'sglang.kernels.ops.moe.moe_wna16_marlin\')\n    original=module._jit_moe_wna16_marlin_module\n    def poisoned_module(*args):\n        native=original(*args)\n        def gemm(*values):\n            scratch=values[15]\n            if scratch.numel(): scratch.fill_(float(\'nan\'))\n            return native.moe_wna16_marlin_gemm(*values)\n        return types.SimpleNamespace(moe_wna16_marlin_gemm=gemm)\n    module._jit_moe_wna16_marlin_module=poisoned_module\n    try:yield\n    finally:module._jit_moe_wna16_marlin_module=original\n\n\ndef check_graph_outputs(graphs, outputs, kwargs, weights, original_ref, case):\n    import torch\n    records = []\n    # Change graph inputs between replays, including routes, to expose stale scratch.\n    original = [{k: value[k].clone() for k in [\'hidden_states\',\'topk_ids\',\'topk_weights\']} for value in kwargs]\n    for variant in [\'original\',\'changed\',\'restored\']:\n        for graph, value, saved in zip(graphs, kwargs, original):\n            for key in saved:\n                new = saved[key]\n                if variant == \'changed\':\n                    new = new*.5 if key != \'topk_ids\' else (new+1)%weights[\'w1\'].shape[0]\n                value[key].copy_(new)\n            graph.replay()\n        torch.cuda.synchronize()\n        reference = original_ref if variant != \'changed\' else fp32_reference(kwargs[0],weights)\n        arrays={\'baseline\':cpu_array(outputs[0]),\'treatment\':cpu_array(outputs[1]),\'poisoned\':cpu_array(outputs[2]),\'fp32\':cpu_array(reference)}\n        np.savez(case/f\'{variant}-outputs.npz\',**arrays)\n        repeats={\'baseline\':[],\'treatment\':[],\'independent_control\':[]}\n        for _ in range(8):\n            for name,index in [(\'baseline\',0),(\'treatment\',1),(\'independent_control\',3)]:\n                graphs[index].replay();torch.cuda.synchronize()\n                repeats[name].append(cpu_array(outputs[index]).copy())\n        repeated={k:np.stack(v) for k,v in repeats.items()}\n        np.savez(case/f\'{variant}-repeats.npz\',**repeated)\n        dtype=cpu_array(dtype_reference(kwargs[0],weights))\n        np.savez(case/f\'{variant}-dtype-reference.npz\',dtype_reference=dtype)\n        row={\'variant\':variant,**qualify_outputs(**arrays),\n             \'repeatability\':repeatability_metrics(**repeated,fp32=arrays[\'fp32\']),\n             \'dtype_reference\':{arm:errors(arrays[arm],dtype) for arm in [\'baseline\',\'treatment\']}}\n        if not all(np.isfinite(a).all() for a in [*arrays.values(),*repeated.values(),dtype]):\n            raise ValueError(\'Nonfinite output; abort diagnostic timings\')\n        if not row[\'poison_agreement\'][\'finite\'] or row[\'poison_agreement\'][\'nrms\']>.001 or row[\'poison_agreement\'][\'nmax\']>.01:\n            raise ValueError(\'Scratch poison output differs; abort diagnostic timings\')\n        records.append(row)\n    return records\n\n\n\ndef observed_calls(base, treatment):\n    import torch\n    import importlib\n    module = importlib.import_module(\'sglang.srt.layers.moe.fused_moe_triton.fused_marlin_moe\')\n    original = module.moe_wna16_marlin_gemm\n    receipts = []\n    def observe(*args, **kw):\n        sms = torch.cuda.get_device_properties(args[0].device).multi_processor_count\n        scratch = min(kw[\'size_n\']*args[10].size(0),sms*4*kw[\'moe_block_size\']*256)\n        if kw[\'moe_block_size\']==8: scratch *= 2\n        receipt = {key:kw[key] for key in [\'use_atomic_add\',\'use_fp32_reduce\',\'size_m\',\'size_n\',\'size_k\',\'moe_block_size\',\'top_k\',\'mul_topk_weights\']}\n        receipt[\'scratch_expected_bytes\'] = scratch*4 if kw[\'use_fp32_reduce\'] and not kw[\'use_atomic_add\'] else 0\n        receipts.append(receipt)\n        return original(*args,**kw)\n    module.moe_wna16_marlin_gemm = observe\n    try:\n        outputs = [base(),treatment()]\n    finally:\n        module.moe_wna16_marlin_gemm = original\n    if len(receipts)!=4 or [r[\'use_atomic_add\'] for r in receipts] != [True,True,False,False] or not all(r[\'use_fp32_reduce\'] for r in receipts):\n        raise ValueError(\'Actual GEMM reduction flags differ from declared contrast\')\n    return outputs,receipts\n\n\ndef run_case(record, capture, out):\n    import torch\n    case = out/Path(record[\'case\']).stem;case.mkdir()\n    sample = torch.load(capture/record[\'case\'],map_location=\'cpu\',weights_only=True)\n    weights = torch.load(capture/record[\'weights\'],map_location=\'cpu\',weights_only=True)\n    ref = fp32_reference(sample,weights)\n    shared_sample = {k:v.cuda() if isinstance(v,torch.Tensor) else v for k,v in sample.items()}\n    shared_weights = {k:v.cuda() if isinstance(v,torch.Tensor) else v for k,v in weights.items()}\n    base, kb = callable_for(shared_sample,shared_weights,None)\n    treatment, kt = callable_for(shared_sample,shared_weights,False)\n    assert torch.cuda.get_device_capability()[0]>=9\n    inputs_before = {k:fingerprint(kb[k]) for k in [\'hidden_states\',\'topk_ids\',\'topk_weights\']}\n    (eager_base, eager_treatment), calls = observed_calls(base,treatment)\n    torch.save({\'baseline\':eager_base.detach().cpu(),\'treatment\':eager_treatment.detach().cpu(),\'fp32\':ref.detach().cpu()},case/\'eager-outputs.pt\')\n    eager_metrics = {\'baseline\':errors(cpu_array(eager_base),cpu_array(ref)), \'treatment\':errors(cpu_array(eager_treatment),cpu_array(ref))}\n    torch.cuda.reset_peak_memory_stats()\n    gb, ob, mb = capture_graph(base)\n    gt, ot, mt = capture_graph(treatment)\n    second_base,ks=callable_for(shared_sample,shared_weights,None)\n    gs,obs,ms=capture_graph(second_base)\n    poison_fn, kp = callable_for(shared_sample,shared_weights,False)\n    with poison_actual_scratch():\n        gp, op, mp = capture_graph(poison_fn)\n    numeric = check_graph_outputs([gb,gt,gp,gs],[ob,ot,op,obs],[kb,kt,kp,ks],weights,ref,case)\n    np.savez(case/\'eager-outputs.npz\',baseline=cpu_array(eager_base),treatment=cpu_array(eager_treatment))\n    eager_graph = [errors(cpu_array(ob),cpu_array(eager_base)),errors(cpu_array(ot),cpu_array(eager_treatment))]\n    result = {\'record\':record,\'actual_gemm_calls\':calls,\'eager_reference\':eager_metrics,\'numerical\':numeric,\'eager_graph\':eager_graph,\n              \'graph_allocated_bytes\':{\'baseline\':mb,\'treatment\':mt,\'poison_qualification\':mp,\'second_control\':ms},\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(),\n              \'peak_reserved_bytes\':torch.cuda.max_memory_reserved()}\n    passed = all(row[\'passed\'] for row in numeric) and all(m[\'finite\'] and m[\'nrms\']<=.001 and m[\'nmax\']<=.01 for m in eager_graph)\n    result[\'numerical_passed\'] = passed\n    torch.save({\'baseline\':ob.detach().cpu(),\'treatment\':ot.detach().cpu(),\'poisoned_scratch\':op.detach().cpu(),\'fp32\':ref.detach().cpu()},case/\'outputs.pt\')\n    if not all(m[\'finite\'] and m[\'nrms\']<=.001 and m[\'nmax\']<=.01 for m in eager_graph):\n        raise ValueError(\'Eager/graph disagreement; abort diagnostic timings\')\n    result[\'graph_timing\'] = timed_pairs(gb.replay,gt.replay,case/\'graph-pairs.jsonl\')\n    result[\'eager_timing\'] = timed_pairs(base,treatment,case/\'eager-pairs.jsonl\')\n    result[\'control_graph_timing\'] = timed_pairs(gb.replay,gs.replay,case/\'control-graph-pairs.jsonl\')\n    result[\'timing_scope\']=\'diagnostic even when v3 numerical gate rejects; no serving advancement\'\n    inputs_after = {k:fingerprint(kb[k]) for k in inputs_before}\n    assert inputs_before==inputs_after\n    assert inputs_after=={k:fingerprint(kt[k]) for k in inputs_before}\n    assert inputs_after=={k:fingerprint(ks[k]) for k in inputs_before}\n    assert inputs_after=={k:fingerprint(kp[k]) for k in inputs_before}\n    result[\'inputs_before\']=inputs_before;result[\'inputs_after\']=inputs_after\n    (case/\'result.json\').write_text(json.dumps(result,indent=2,allow_nan=False)+\'\\n\')\n    return result\n\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'--capture\',type=Path,required=True);parser.add_argument(\'--out\',type=Path,required=True)\n    args=parser.parse_args();args.out.mkdir(exist_ok=False)\n    import torch\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    import sglang\n    from sglang.srt.server_args import prepare_server_args, set_global_server_args_for_scheduler\n    runtime_argv=json.loads((args.capture.parent/\'server-args.json\').read_text())\n    if runtime_argv[1]!=\'serve\':raise ValueError(\'Unexpected preserved serving command\')\n    from runtime_views import prepare_runtime_views\n    prepare_runtime_views(runtime_argv,args.out.parent/\'runtime-view.json\')\n    set_global_server_args_for_scheduler(prepare_server_args(runtime_argv[2:]))\n    patch_manifest=json.loads((args.capture.parent/\'patch-manifest.json\').read_text())\n    installed_root=Path(sglang.__file__).parent.parent\n    for name,identity in patch_manifest.items():\n        if hashlib.sha256((installed_root/name).read_bytes()).hexdigest()!=identity[\'after\']:\n            raise ValueError(\'Installed candidate source changed before replay: \'+name)\n    records=[json.loads(line) for line in (args.capture/\'cases.jsonl\').read_text().splitlines()]\n    expected={f\'layer-{layer:02d}-replay-{step:02d}.pt\' for layer in [0,24,47] for step in [8,16,32]}\n    assert len(records)==9 and {r[\'case\'] for r in records}==expected\n    for record in records:\n        assert record[\'raw_bs\']==10 and record[\'mode\']==\'TARGET_VERIFY\'\n        assert record[\'rows\']==record[\'hidden_shape\'][0]\n        for key,hash_key in [(\'case\',\'sha256\'),(\'weights\',\'weights_sha256\')]:\n            assert hashlib.sha256((args.capture/record[key]).read_bytes()).hexdigest()==record[hash_key]\n    sentinel=verify_pack_sentinel()\n    results=[]\n    for record in records:\n        result=run_case(record,args.capture,args.out);results.append(result)\n        if sum(p.stat().st_size for p in args.out.rglob(\'*\') if p.is_file())>1024**3:\n            raise ValueError(\'Diagnostic output exceeds 1 GiB cap\')\n        # The original tolerance verdict is retained, but does not suppress diagnostics.\n        torch.cuda.empty_cache()\n    timing=[r[\'graph_timing\'] for r in results if \'graph_timing\' in r]\n    eligible=(len(results)==9 and all(r[\'numerical_passed\'] for r in results)\n              and all(t[\'geometric_speedup\']>=1.03 and t[\'bootstrap_95\'][0]>1.01 and t[\'faster_pairs\']>=27 for t in timing))\n    result={\'sentinel\':sentinel,\'runtime_args_sha256\':hashlib.sha256((args.capture.parent/\'server-args.json\').read_bytes()).hexdigest(),\'torch\':torch.__version__,\'device\':torch.cuda.get_device_name(),\n            \'cases_completed\':len(results),\'advance_to_serving_preparation\':False,\'original_gate_and_performance_passed\':eligible,\'case_graph_summaries\':timing,\n            \'limits\':[\'one GPU/session; no between-run serving uncertainty\',\'micro graphs do not certify full model graph residency\',\'numerical tolerance is not gameplay or output-token equivalence\']}\n    (args.out/\'RESULT.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n\n\nif __name__==\'__main__\':main()\n', 'reference.py': '"""Independent inverse of W4 Marlin\'s 16x64 repack; no kernel output as oracle."""\nimport math\nimport numpy as np\n\n\ndef unpack_marlin(packed, k, n):\n    if k % 16 or n % 64 or np.asarray(packed).size != k*n//8:\n        raise ValueError(\'Unsupported packed shape\')\n    words = np.asarray(packed, dtype=np.int32).view(np.uint32).reshape(k//16, n//64, 128)\n    nibble = ((words[..., None] >> (np.arange(8, dtype=np.uint32)*4)) & 15).reshape(k//16, n//64, 1024)\n    mapping = np.empty((16,64), dtype=np.int64)\n    order = [0,2,4,6,1,3,5,7]\n    for lane in range(32):\n        for warp in range(4):\n            for pos, val in enumerate(order):\n                row = 2*(lane % 4) + [0,1,8,9][val % 4]\n                col = 16*warp + lane//4 + (8 if val >= 4 else 0)\n                mapping[row,col] = (lane*4+warp)*8 + pos\n    tiles = nibble[..., mapping.ravel()].reshape(k//16,n//64,16,64)\n    return tiles.transpose(0,2,1,3).reshape(k,n).astype(np.int16)-8\n\n\ndef unpermute_scales(scales, k):\n    scales = np.asarray(scales, dtype=np.float32)\n    groups, n = scales.shape\n    perm = [i+8*j for i in range(8) for j in range(8)] if groups > 1 else [2*i+j for i in range(4) for j in [0,1,8,9,16,17,24,25]]\n    if k % groups or scales.size % len(perm):\n        raise ValueError(\'Unsupported scale grouping\')\n    return scales.reshape(-1,len(perm))[:,np.argsort(perm)].reshape(groups,n)\n\n\ndef dequantize(packed, scales, k, n):\n    q = unpack_marlin(packed,k,n).astype(np.float32)\n    s = unpermute_scales(scales,k)\n    return q * np.repeat(s, k//s.shape[0], axis=0)\n\n\ndef errors(output, reference):\n    actual = np.asarray(output,dtype=np.float64)\n    expected = np.asarray(reference,dtype=np.float64)\n    if actual.shape != expected.shape:\n        raise ValueError(\'Output/reference shape mismatch\')\n    if not (np.isfinite(actual).all() and np.isfinite(expected).all()):\n        return dict(finite=False,max_abs=None,rms=None,reference_rms=None,nrms=None,nmax=None)\n    delta = actual-expected\n    rms_ref = float(np.sqrt(np.mean(expected**2)))\n    return {\'finite\': bool(np.isfinite(actual).all() and np.isfinite(expected).all()),\n            \'max_abs\': float(np.max(np.abs(delta))), \'rms\': float(np.sqrt(np.mean(delta**2))),\n            \'reference_rms\': rms_ref, \'nrms\': float(np.sqrt(np.mean(delta**2)))/max(rms_ref,1e-12),\n            \'nmax\': float(np.max(np.abs(delta)))/max(rms_ref,1e-12)}\n\n\ndef numerical_gate(base, treatment, drift):\n    return (all(x[\'finite\'] for x in [base,treatment,drift])\n            and max(base[\'nrms\'],treatment[\'nrms\']) <= .03\n            and max(base[\'nmax\'],treatment[\'nmax\']) <= .25\n            and treatment[\'nrms\'] <= 1.10*base[\'nrms\']+.0001\n            and treatment[\'nmax\'] <= 1.10*base[\'nmax\']+.001\n            and drift[\'nrms\'] <= .005 and drift[\'nmax\'] <= .05)\n\n\ndef paired_summary(pairs):\n    arr = np.asarray(pairs,dtype=np.float64)\n    if arr.ndim != 2 or arr.shape[1] != 2 or not np.isfinite(arr).all() or (arr <= 0).any():\n        raise ValueError(\'Expected positive finite paired baseline/treatment milliseconds\')\n    logs = np.log(arr[:,0]/arr[:,1])\n    rng = np.random.default_rng(20261009)\n    bootstrap = logs[rng.integers(0,len(logs),size=(10000,len(logs)))].mean(axis=1)\n    lo, hi = np.exp(np.quantile(bootstrap,[.025,.975]))\n    return {\'pairs\': len(arr), \'geometric_speedup\': math.exp(float(logs.mean())),\n            \'bootstrap_95\': [float(lo),float(hi)], \'faster_pairs\': int((logs>0).sum()),\n            \'median_speedup\': float(np.median(arr[:,0]/arr[:,1]))}\n\n\ndef qualify_outputs(baseline, treatment, poisoned, fp32):\n    base=errors(baseline,fp32);candidate=errors(treatment,fp32)\n    drift=errors(treatment,baseline);poison=errors(poisoned,fp32)\n    agreement=errors(poisoned,treatment)\n    scale=max(base[\'reference_rms\'] or 0,1e-12)\n    if drift[\'finite\']:drift.update(nrms=drift[\'rms\']/scale,nmax=drift[\'max_abs\']/scale)\n    passed=(numerical_gate(base,candidate,drift) and poison[\'finite\']\n            and poison[\'nrms\']<=1.10*base[\'nrms\']+.0001 and poison[\'nmax\']<=1.10*base[\'nmax\']+.001\n            and agreement[\'finite\'] and agreement[\'nrms\']<=.001 and agreement[\'nmax\']<=.01)\n    return {\'baseline\':base,\'treatment\':candidate,\'drift\':drift,\'poisoned_scratch\':poison,\'poison_agreement\':agreement,\'passed\':passed}\n', 'runtime_views.py': 'import copy,hashlib,json\nfrom pathlib import Path\nEXPERT_TARGET=r"re:^mtp\\.layers\\.0\\.mlp\\.experts\\.[0-9]+\\.(gate_proj|up_proj|down_proj)$"\nDENSE_IGNORE=r"re:^(?!mtp\\.layers\\.0\\.mlp\\.experts(?:\\.|$)).*"\n\ndef indexed_shards(root):\n    root = Path(root)\n    index = json.loads((root / "model.safetensors.index.json").read_text())\n    names = sorted(set(index["weight_map"].values()))\n    if not names: raise RuntimeError(f"Empty weight index: {root}")\n    for name in names:\n        rel = Path(name)\n        if rel.is_absolute() or ".." in rel.parts:\n            raise RuntimeError(f"Invalid shard path: {name}")\n        p = root / rel\n        if not p.is_file() or p.stat().st_size < 8:\n            raise RuntimeError(f"Missing/empty shard: {p}")\n    return index, names\n\ndef prepare_draft_view(target, source_root, work_root):\n    """Adapt metadata only; link original dataset weights without copying them."""\n    target, source_root, work_root = map(Path, (target, source_root, work_root))\n    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []\n    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)\n    candidates = []\n    for path in configs:\n        cfg = json.loads(path.read_text())\n        q = cfg.get("quantization_config", {})\n        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):\n            candidates.append((path.parent, cfg))\n    if len(candidates) != 1:\n        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")\n    source, cfg = candidates[0]\n    index, shards = indexed_shards(source)\n    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):\n        raise RuntimeError("Draft index does not contain the expected MTP experts")\n    cfg = copy.deepcopy(cfg)\n    q = cfg["quantization_config"]\n    group = q["config_groups"]["mtp_routed_experts"]\n    w = group["weights"]\n    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):\n        raise RuntimeError("Expected symmetric INT4 group32 draft experts")\n    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):\n        raise RuntimeError(f"Unexpected draft target rules: {group[\'targets\']}")\n    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):\n        raise RuntimeError("Unexpected draft ignore rules")\n    group["targets"] = [EXPERT_TARGET]\n    q["ignore"] = [DENSE_IGNORE]\n    # Content/path-specific directory makes notebook reruns idempotent without\n    # overwriting an unrelated view. We never modify the read-only datasets.\n    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)\n    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]\n    view = work_root / f"draft-view-{suffix}"\n    view.mkdir(parents=True, exist_ok=True)\n    config_path = view / "config.json"\n    config_text = json.dumps(cfg, indent=2) + "\\n"\n    if config_path.exists() or config_path.is_symlink():\n        if config_path.is_symlink() or config_path.read_text() != config_text:\n            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")\n    else:\n        config_path.write_text(config_text)\n    links = {name: source / name for name in shards}\n    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"\n    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):\n        if (target / name).is_file(): links[name] = target / name\n    for name, src in links.items():\n        dest = view / name\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        if dest.exists() or dest.is_symlink():\n            if not dest.is_symlink() or dest.resolve() != src.resolve():\n                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")\n        else:\n            dest.symlink_to(src.resolve())\n    print(f"Draft source: {source}\\nSGLang draft view: {view} ({len(shards)} linked weight shards)")\n    return view\n\n\ndef prepare_runtime_views(argv,receipt):\n    target=Path(argv[argv.index(\'--model-path\')+1])\n    expected=Path(argv[argv.index(\'--speculative-draft-model-path\')+1])\n    source=Path(\'/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1\')\n    if expected.parent!=Path(\'/tmp/sgl-intel\') or not expected.name.startswith(\'draft-view-\'):\n        raise ValueError(\'Unexpected captured draft view path\')\n    actual=prepare_draft_view(target,source,expected.parent)\n    if actual!=expected:\n        raise ValueError(\'Recreated draft config identity differs from v3 runtime\')\n    receipt.write_text(json.dumps({\'view\':str(actual),\'config_sha256\':hashlib.sha256((actual/\'config.json\').read_bytes()).hexdigest(),\'weights_loaded\':False},indent=2)+\'\\n\')\n    return actual\n', 'marlin_capture.py': '"""Explicit experiment hooks. Config absent means no binding or capture.\n\nRequires patched exact wheel. Device copies are recorded inside capture graphs;\nCPU serialization runs only after real, armed S10 target verification replays.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\n_ACTIVE = None\n_BUFFERS = {}\n_COUNTS = {}\nCONFIG = None\nWEIGHTS = [\'w1\', \'w2\', \'w1_scale\', \'w2_scale\', \'g_idx1\', \'g_idx2\', \'sort_indices1\', \'sort_indices2\', \'w1_zeros\', \'w2_zeros\', \'w1_global_scale\', \'w2_global_scale\', \'w1_bias\', \'w2_bias\', \'expert_map\']\nINPUTS = [\'hidden_states\', \'gating_output\', \'topk_weights\', \'topk_ids\']\nSCALARS = [\'global_num_experts\', \'num_bits\', \'is_k_full\', \'inplace\', \'routed_scaling_factor\', \'clamp_limit\', \'gemm1_alpha\', \'activation\', \'is_gated\']\n\n\ndef config():\n    global CONFIG\n    name = os.environ.get(\'MARLIN_EXPERIMENT_CONFIG\')\n    if name is None:\n        return None\n    if CONFIG is None:\n        CONFIG = json.loads(Path(name).read_text())\n        if CONFIG[\'mode\'] not in {\'capture\', \'control\', \'treatment\'}:\n            raise ValueError(\'Unsupported experiment mode\')\n    return CONFIG\n\n\ndef target_kernels(runner):\n    if runner.is_draft_worker:\n        return []\n    from sglang.srt.hardware_backend.gpu.quantization.gptq_kernels import GPTQMarlinMoEKernel\n    return [(name, layer, layer.scheme.kernel) for name, layer in runner.model.named_modules()\n            if isinstance(getattr(getattr(layer, \'scheme\', None), \'kernel\', None), GPTQMarlinMoEKernel)]\n\n\ndef bind_model(runner):\n    cfg = config()\n    if cfg is None:\n        return\n    kernels = target_kernels(runner)\n    out = Path(cfg[\'out\']); out.mkdir(parents=True, exist_ok=True)\n    receipt = {\'is_draft_worker\': runner.is_draft_worker, \'mode\': cfg[\'mode\'], \'modules\': [n for n, _, _ in kernels]}\n    if not runner.is_draft_worker:\n        if len(kernels) != cfg[\'expected_target_layers\']:\n            raise ValueError(f\'Expected {cfg["expected_target_layers"]} target GPTQ MoE kernels, got {len(kernels)}\')\n        for ordinal, (name, layer, kernel) in enumerate(kernels):\n            kernel.experiment_atomic_override = False if cfg[\'mode\'] == \'treatment\' else None\n            if cfg[\'mode\'] == \'capture\' and ordinal in cfg[\'layer_ordinals\']:\n                kernel.experiment_capture_tag = f\'layer-{ordinal:02d}\'\n                receipt.setdefault(\'selected\', {})[kernel.experiment_capture_tag] = name\n    role = \'draft\' if runner.is_draft_worker else \'target\'\n    (out / f\'binding-{role}.json\').write_text(json.dumps(receipt, indent=2)+\'\\n\')\n\n\ndef begin_capture(runner, key):\n    global _ACTIVE\n    cfg = config()\n    _ACTIVE = None\n    if cfg is not None and cfg[\'mode\'] == \'capture\' and not runner.model_runner.is_draft_worker:\n        _ACTIVE = (id(runner.model_runner), repr(key))\n\n\ndef end_capture():\n    global _ACTIVE\n    _ACTIVE = None\n\n\ndef validate_call(args):\n    import torch\n    if args[\'num_bits\'] != 4 or args[\'hidden_states\'].dtype != torch.bfloat16:\n        raise ValueError(\'Reference supports target W4/BF16 only\')\n    if any(args[key].dtype != torch.bfloat16 for key in [\'w1_scale\',\'w2_scale\']) or any(args[key].dtype != torch.int32 for key in [\'w1\',\'w2\']):\n        raise ValueError(\'Reference requires integer W4 with BF16 scales\')\n    for key in [\'expert_map\',\'w1_zeros\',\'w2_zeros\',\'w1_global_scale\',\'w2_global_scale\',\'w1_bias\',\'w2_bias\']:\n        if args[key] is not None:\n            raise ValueError(f\'Unsupported reference feature: {key}\')\n    for key in [\'g_idx1\',\'g_idx2\',\'sort_indices1\',\'sort_indices2\']:\n        if args[key] is not None and args[key].numel() != 0:\n            raise ValueError(f\'Unsupported reference feature: {key}\')\n    if not args[\'is_k_full\'] or args[\'inplace\'] or not args[\'is_gated\'] or args[\'activation\'] != \'silu\' or args[\'clamp_limit\'] is not None or args[\'gemm1_alpha\'] is not None:\n        raise ValueError(\'Reference supports full-K gated SiLU with no clamp/alpha\')\n\n\ndef snapshot(tag, args):\n    if _ACTIVE is None:\n        return\n    import torch\n    validate_call(args)\n    # Replace warmup buffers: the final call records these copy nodes into graph.\n    buf = {key: torch.empty_like(args[key]) for key in INPUTS}\n    for key in INPUTS:\n        buf[key].copy_(args[key])\n    _BUFFERS[(_ACTIVE, tag)] = (buf, {key: args[key] for key in WEIGHTS + SCALARS})\n\n\ndef save_exclusive(path, value):\n    import torch\n    with path.open(\'xb\') as stream:\n        torch.save(value, stream)\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'): digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef after_execute(runner, forward_batch):\n    cfg = config()\n    if cfg is None or cfg[\'mode\'] != \'capture\' or runner.model_runner.is_draft_worker:\n        return\n    out = Path(cfg[\'out\'])\n    if not (out / \'ARM-CAPTURE\').is_file() or forward_batch.batch_size != 10 or forward_batch.forward_mode.name != \'TARGET_VERIFY\':\n        return\n    key = (id(runner.model_runner), repr(runner._replay_graph_key))\n    count = _COUNTS.get(key, 0) + 1; _COUNTS[key] = count\n    if count not in cfg[\'replay_ordinals\']:\n        return\n    import torch\n    torch.cuda.synchronize()\n    records = []\n    for tag in cfg[\'selected_tags\']:\n        buf, args = _BUFFERS[(key, tag)]  # missing snapshots invalidate, never substitute another graph\n        weights_path = out / f\'{tag}-weights.pt\'\n        if not weights_path.exists():\n            required = sum(v.numel()*v.element_size() for k,v in args.items() if k in WEIGHTS and isinstance(v,torch.Tensor))\n            if sum(p.stat().st_size for p in out.glob(\'*.pt\')) + required + 65536 > cfg[\'max_capture_bytes\']:\n                raise RuntimeError(\'Capture weight storage cap would be exceeded\')\n            weights_hash = save_exclusive(weights_path, {k: v.detach().cpu() if isinstance(v, torch.Tensor) else v for k, v in args.items() if k in WEIGHTS})\n            (out / f\'{tag}-weights.sha256\').write_text(weights_hash+\'\\n\')\n        case_path = out / f\'{tag}-replay-{count:02d}.pt\'\n        sample = {k: v.detach().cpu() for k, v in buf.items()}\n        sample.update({k: args[k] for k in SCALARS})\n        sha = save_exclusive(case_path, sample)\n        ids = sample[\'topk_ids\'].flatten()\n        experts, counts = torch.unique(ids, return_counts=True)\n        record = {\'case\': case_path.name, \'sha256\': sha, \'weights\': weights_path.name,\n                  \'weights_sha256\': (out / f\'{tag}-weights.sha256\').read_text().strip(),\n                  \'graph_key\': key[1], \'raw_bs\': forward_batch.batch_size, \'rows\': runner.raw_num_token,\n                  \'hidden_shape\': list(buf[\'hidden_states\'].shape), \'route_shape\': list(buf[\'topk_ids\'].shape),\n                  \'mode\': forward_batch.forward_mode.name, \'replay_ordinal\': count,\n                  \'expert_counts\': list(zip(experts.tolist(), counts.tolist()))}\n        records.append(record)\n    with (out / \'cases.jsonl\').open(\'a\') as stream:\n        for record in records: stream.write(json.dumps(record)+\'\\n\')\n    if sum(p.stat().st_size for p in out.glob(\'*.pt\')) > cfg[\'max_capture_bytes\']:\n        raise RuntimeError(\'Capture storage cap exceeded\')\n'}
replacements={'sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py': 'from typing import Optional\n\nimport torch\nimport torch.nn.functional as F\nimport triton\nimport triton.language as tl\n\nfrom sglang.srt.layers import zero_copy_context\nfrom sglang.srt.utils import is_cuda\nfrom sglang.srt.utils.custom_op import register_custom_op\n\n_is_cuda = is_cuda()\n\nif _is_cuda:\n    from sgl_kernel import moe_sum_reduce\n\n    from sglang.kernels.ops.activation.activation import silu_and_mul\n    from sglang.kernels.ops.moe.moe_wna16_marlin import moe_wna16_marlin_gemm\n\n\n@triton.jit\ndef _tl_tanh(x):\n    return 2.0 * tl.sigmoid(2.0 * x) - 1.0\n\n\n@triton.jit\ndef _situ_and_mul_kernel(\n    x_ptr,  # [M, 2N] gate;up halves (non-interleaved)\n    out_ptr,  # [M, N]\n    N,\n    situ_beta,\n    linear_beta,\n    stride_xm,\n    stride_om,\n    BLOCK_N: tl.constexpr,\n    HAS_LINEAR_BETA: tl.constexpr,\n):\n    pid_m = tl.program_id(0)\n    pid_n = tl.program_id(1)\n    offs = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)\n    mask = offs < N\n    base = x_ptr + pid_m * stride_xm\n    gate = tl.load(base + offs, mask=mask, other=0.0).to(tl.float32)\n    up = tl.load(base + N + offs, mask=mask, other=0.0).to(tl.float32)\n    gate = situ_beta * _tl_tanh(gate / situ_beta) * tl.sigmoid(gate)\n    if HAS_LINEAR_BETA:\n        up = linear_beta * _tl_tanh(up / linear_beta)\n    out = gate * up\n    tl.store(\n        out_ptr + pid_m * stride_om + offs,\n        out.to(out_ptr.dtype.element_ty),\n        mask=mask,\n    )\n\n\ndef situ_and_mul(\n    output: torch.Tensor,\n    x: torch.Tensor,\n    situ_beta: float,\n    linear_beta: Optional[float],\n) -> None:\n    """SiTU gated activation (Kimi K3), fused into one elementwise kernel:\n    out = situ_beta*tanh(gate/situ_beta)*sigmoid(gate) * linear_beta*tanh(up/linear_beta)\n    where x = [gate; up] halves along the last dim.\n    """\n    M, N2 = x.shape\n    N = N2 // 2\n    assert output.shape == (M, N)\n    BLOCK_N = 1024\n    grid = (M, triton.cdiv(N, BLOCK_N))\n    _situ_and_mul_kernel[grid](\n        x,\n        output,\n        N,\n        float(situ_beta),\n        float(linear_beta) if linear_beta is not None else 0.0,\n        x.stride(0),\n        output.stride(0),\n        BLOCK_N=BLOCK_N,\n        HAS_LINEAR_BETA=linear_beta is not None,\n    )\n\n\ndef get_scalar_type(\n    num_bits: int,\n    has_zp: bool,\n    scales: Optional[torch.Tensor] = None,\n    global_scale: Optional[torch.Tensor] = None,\n):\n    from sgl_kernel.scalar_type import scalar_types\n\n    if (\n        not has_zp\n        and num_bits == 4\n        and scales is not None\n        and (scales.dtype == torch.float8_e8m0fnu or global_scale is not None)\n    ):\n        return scalar_types.float4_e2m1f\n    if has_zp:\n        assert num_bits == 4\n        return scalar_types.uint4\n    else:\n        return scalar_types.uint4b8 if num_bits == 4 else scalar_types.uint8b128\n\n\ndef swiglu_limit_func(\n    output: torch.Tensor,\n    input: torch.Tensor,  # first half is gate, second half is up\n    swiglu_limit: float = 0.0,\n) -> None:\n    d = input.shape[1] // 2\n    gate = input[:, :d]\n    up = input[:, d:]\n\n    if swiglu_limit > 0:\n        gate = torch.clamp(gate, max=swiglu_limit)\n        up = torch.clamp(up, min=-swiglu_limit, max=swiglu_limit)\n\n    output.copy_(F.silu(gate) * up)\n\n\ndef swiglu_gpt_oss_sigmoid_alpha_contiguous(\n    output: torch.Tensor,\n    input: torch.Tensor,  # first half is gate, second half is up\n    gemm1_alpha: float,\n    gemm1_limit: float,\n) -> None:\n    d = input.shape[1] // 2\n    gate = input[:, :d].clamp(max=gemm1_limit)\n    up = input[:, d:].clamp(min=-gemm1_limit, max=gemm1_limit)\n    output.copy_(gate * torch.sigmoid(gate * gemm1_alpha) * (up + 1))\n\n\n@register_custom_op(out_shape="hidden_states")\ndef fused_marlin_moe(\n    hidden_states: torch.Tensor,\n    w1: torch.Tensor,\n    w2: torch.Tensor,\n    w1_scale: torch.Tensor,\n    w2_scale: torch.Tensor,\n    gating_output: torch.Tensor,\n    topk_weights: torch.Tensor,\n    topk_ids: torch.Tensor,\n    global_num_experts: int = -1,\n    expert_map: Optional[torch.Tensor] = None,\n    g_idx1: Optional[torch.Tensor] = None,\n    g_idx2: Optional[torch.Tensor] = None,\n    sort_indices1: Optional[torch.Tensor] = None,\n    sort_indices2: Optional[torch.Tensor] = None,\n    w1_zeros: Optional[torch.Tensor] = None,\n    w2_zeros: Optional[torch.Tensor] = None,\n    w1_global_scale: Optional[torch.Tensor] = None,\n    w2_global_scale: Optional[torch.Tensor] = None,\n    w1_bias: Optional[torch.Tensor] = None,\n    w2_bias: Optional[torch.Tensor] = None,\n    workspace: Optional[torch.Tensor] = None,\n    num_bits: int = 8,\n    is_k_full: bool = True,\n    inplace: bool = False,\n    routed_scaling_factor: Optional[float] = None,\n    clamp_limit: Optional[float] = None,\n    gemm1_alpha: Optional[float] = None,\n    activation: str = "silu",\n    is_gated: bool = True,\n    use_atomic_add_override: Optional[bool] = None,\n    experiment_capture_tag: Optional[str] = None,\n) -> torch.Tensor:\n    """\n    This function computes a Mixture of Experts (MoE) layer using two sets of\n    weights, w1 and w2, and top-k gating mechanism.\n\n    Parameters:\n    - hidden_states (torch.Tensor): The input tensor to the MoE layer.\n    - w1 (torch.Tensor): The first set of expert weights.\n    - w2 (torch.Tensor): The second set of expert weights.\n    - w1_scale (torch.Tensor): Scale to be used for w1.\n    - w2_scale (torch.Tensor): Scale to be used for w2.\n    - gating_output (torch.Tensor): The output of the gating operation\n        (before softmax).\n    - g_idx1 (Optional[torch.Tensor]): The first set of act_order indices.\n    - g_idx2 (Optional[torch.Tensor]): The second set of act_order indices.\n    - sort_indices1 (Optional[torch.Tensor]): The first act_order input\n        permutation.\n    - sort_indices2 (Optional[torch.Tensor]): The second act_order input\n        permutation.\n    - topk_weights (torch.Tensor): Top-k weights.\n    - topk_ids (torch.Tensor): Indices of topk-k elements.\n    - w1_zeros (Optional[torch.Tensor]): Optional zero points to be used for w1.\n    - w2_zeros (Optional[torch.Tensor]): Optional zero points to be used for w2.\n    - num_bits (int): The number of bits in expert weights quantization.\n\n    Returns:\n    - torch.Tensor: The output tensor after applying the MoE layer.\n    """\n    from sglang.srt.layers.moe.fused_moe_triton import moe_align_block_size\n\n    assert hidden_states.shape[0] == gating_output.shape[0], "Number of tokens mismatch"\n    assert hidden_states.shape[1] == w1.shape[1] * 16, "Hidden size mismatch w1"\n    assert hidden_states.shape[1] == w2.shape[2] // (\n        num_bits // 2\n    ), "Hidden size mismatch w2"\n    assert hidden_states.is_contiguous(), "Hidden_states must be contiguous"\n    assert w1.is_contiguous(), "Expert weights1 must be contiguous"\n    assert w2.is_contiguous(), "Expert weights2 must be contiguous"\n    assert hidden_states.dtype in [torch.float16, torch.bfloat16]\n    is_mxfp4_marlin = (\n        num_bits == 4\n        and w1_zeros is None\n        and w2_zeros is None\n        and w1_scale.dtype == torch.float8_e8m0fnu\n        and w2_scale.dtype == torch.float8_e8m0fnu\n    )\n    is_nvfp4_marlin = (\n        num_bits == 4\n        and w1_zeros is None\n        and w2_zeros is None\n        and w1_global_scale is not None\n        and w2_global_scale is not None\n    )\n    if is_mxfp4_marlin:\n        assert hidden_states.dtype == torch.bfloat16, (\n            "MXFP4 Marlin with E8M0 scales is only instantiated for bfloat16 "\n            f"activations, got {hidden_states.dtype}"\n        )\n    elif not is_nvfp4_marlin:\n        assert (\n            hidden_states.dtype == w1_scale.dtype\n        ), f"moe_wna16_marlin_gemm assumes hidden_states.dtype ({hidden_states.dtype}) == w1_scale.dtype ({w1_scale.dtype})"\n        assert (\n            hidden_states.dtype == w2_scale.dtype\n        ), f"moe_wna16_marlin_gemm assumes hidden_states.dtype ({hidden_states.dtype}) == w2_scale.dtype ({w2_scale.dtype})"\n    assert num_bits in [4, 8]\n\n    if experiment_capture_tag is not None:\n        from marlin_capture import snapshot\n        snapshot(experiment_capture_tag, locals())\n\n    M, K = hidden_states.shape\n    E = w1.shape[0]\n    N = w2.shape[1] * 16\n    topk = topk_ids.shape[1]\n    gemm1_n = 2 * N if is_gated else N\n\n    # M block size selection logic\n    # TODO: tune this further for specific models\n    for block_size_m in [8, 16, 32, 48, 64]:\n        if M * topk / E / block_size_m < 0.9:\n            break\n\n    if global_num_experts == -1:\n        global_num_experts = E\n    if (\n        M == 1\n        and topk <= 32\n        and expert_map is None\n        # The JIT kernel is int32-only; torch-native topk emits int64 -- let\n        # that (test-only) shape take the generic path instead of casting.\n        and topk_ids.dtype == torch.int32\n    ):\n        # Single-token decode: top-k ids are distinct, so alignment is a\n        # single-warp sort instead of the align + count_and_sort kernel pair.\n        from sglang.kernels.ops.moe.moe_align_single_token import moe_align_single_token\n\n        sorted_token_ids, expert_ids, num_tokens_post_padded = moe_align_single_token(\n            topk_ids, block_size_m\n        )\n    else:\n        sorted_token_ids, expert_ids, num_tokens_post_padded = moe_align_block_size(\n            topk_ids, block_size_m, global_num_experts\n        )\n\n    if workspace is None:\n        max_workspace_size = (max(2 * N, K) // 64) * (\n            sorted_token_ids.size(0) // block_size_m\n        )\n        device = hidden_states.device\n        sms = torch.cuda.get_device_properties(device).multi_processor_count\n        max_workspace_size = min(max_workspace_size, sms * 4)\n        workspace = torch.zeros(\n            max_workspace_size, dtype=torch.int, device=device, requires_grad=False\n        )\n\n    scalar_type1 = get_scalar_type(\n        num_bits, w1_zeros is not None, w1_scale, w1_global_scale\n    )\n    scalar_type2 = get_scalar_type(\n        num_bits, w2_zeros is not None, w2_scale, w2_global_scale\n    )\n\n    intermediate_cache2 = torch.empty(\n        (M * topk_ids.shape[1], N),\n        device=hidden_states.device,\n        dtype=hidden_states.dtype,\n    )\n    # Marlin skips masked expert rows, so their shared cache must start at zero.\n    intermediate_cache13 = torch.zeros(\n        (M * topk_ids.shape[1] * max(gemm1_n, K),),\n        device=hidden_states.device,\n        dtype=hidden_states.dtype,\n    )\n    intermediate_cache1 = intermediate_cache13[: M * topk_ids.shape[1] * gemm1_n]\n    intermediate_cache1 = intermediate_cache1.view(-1, gemm1_n)\n    intermediate_cache3 = intermediate_cache13[: M * topk_ids.shape[1] * K]\n    intermediate_cache3 = intermediate_cache3.view(-1, K)\n\n    use_atomic_add = (\n        hidden_states.dtype == torch.half\n        or torch.cuda.get_device_capability(hidden_states.device)[0] >= 9\n    ) and (not is_mxfp4_marlin)\n\n    if use_atomic_add_override is not None:\n        use_atomic_add = use_atomic_add_override\n\n    intermediate_cache1 = moe_wna16_marlin_gemm(\n        hidden_states,\n        intermediate_cache1,\n        w1,\n        w1_bias,\n        w1_scale,\n        w1_global_scale,\n        w1_zeros,\n        g_idx1,\n        sort_indices1,\n        workspace,\n        sorted_token_ids,\n        expert_ids,\n        num_tokens_post_padded,\n        topk_weights,\n        moe_block_size=block_size_m,\n        top_k=topk,\n        mul_topk_weights=False,\n        is_ep=expert_map is not None,\n        b_q_type=scalar_type1,\n        size_m=M,\n        size_n=gemm1_n,\n        size_k=K,\n        is_k_full=is_k_full,\n        use_atomic_add=use_atomic_add,\n        use_fp32_reduce=True,\n        is_zp_float=False,\n    )\n\n    if activation == "silu" and is_gated and gemm1_alpha is not None:\n        if clamp_limit is None:\n            raise ValueError("GPT-OSS Marlin activation requires clamp_limit.")\n        swiglu_gpt_oss_sigmoid_alpha_contiguous(\n            intermediate_cache2,\n            intermediate_cache1.view(-1, gemm1_n),\n            gemm1_alpha,\n            clamp_limit,\n        )\n    elif activation == "silu" and is_gated and clamp_limit is not None:\n        swiglu_limit_func(\n            intermediate_cache2,\n            intermediate_cache1.view(-1, gemm1_n),\n            clamp_limit,\n        )\n    elif activation == "silu" and is_gated:\n        silu_and_mul(intermediate_cache1.view(-1, gemm1_n), intermediate_cache2)\n    elif activation == "situ" and is_gated:\n        situ_and_mul(\n            intermediate_cache2,\n            intermediate_cache1.view(-1, gemm1_n),\n            situ_beta=gemm1_alpha if gemm1_alpha is not None else 4.0,\n            linear_beta=clamp_limit,\n        )\n    elif activation == "silu" and not is_gated:\n        intermediate_cache2 = F.silu(intermediate_cache1.view(-1, N))\n    elif activation == "relu2" and not is_gated:\n        intermediate_cache2 = torch.square(F.relu(intermediate_cache1.view(-1, N)))\n    else:\n        raise ValueError(f"Unsupported activation: {activation=}, with {is_gated=}")\n\n    if expert_map is not None:\n        intermediate_cache3.zero_()\n\n    intermediate_cache3 = moe_wna16_marlin_gemm(\n        intermediate_cache2,\n        intermediate_cache3,\n        w2,\n        w2_bias,\n        w2_scale,\n        w2_global_scale,\n        w2_zeros,\n        g_idx2,\n        sort_indices2,\n        workspace,\n        sorted_token_ids,\n        expert_ids,\n        num_tokens_post_padded,\n        topk_weights,\n        moe_block_size=block_size_m,\n        top_k=1,\n        mul_topk_weights=True,\n        is_ep=expert_map is not None,\n        b_q_type=scalar_type2,\n        size_m=M * topk,\n        size_n=K,\n        size_k=N,\n        is_k_full=is_k_full,\n        use_atomic_add=use_atomic_add,\n        use_fp32_reduce=True,\n        is_zp_float=False,\n    ).view(-1, topk, K)\n\n    output = zero_copy_context.get_moe_output(hidden_states)\n    if output is None:\n        output = hidden_states if inplace else torch.empty_like(hidden_states)\n\n    if is_mxfp4_marlin:\n        # Top-k weights (incl. routed scaling) are already applied above via\n        # mul_topk_weights, so this is a plain sum over the topk dim. The JIT\n        # vectorized pass (~1.5us at decode shapes) beats sgl_kernel\'s\n        # moe_sum_reduce_kernel_general (~5.7us) and the generic at::native\n        # reduce_kernel torch.sum dispatches to (~6.7us).\n        if (\n            intermediate_cache3.dtype == torch.bfloat16\n            and intermediate_cache3.is_contiguous()\n            and output.is_contiguous()\n            and intermediate_cache3.shape[-1] % 8 == 0\n        ):\n            from sglang.kernels.ops.moe.moe_topk_sum import moe_topk_sum\n\n            moe_topk_sum(intermediate_cache3, output)\n        else:\n            moe_sum_reduce(intermediate_cache3, output, 1.0)\n        return output\n    else:\n        if routed_scaling_factor is None:\n            routed_scaling_factor = 1.0\n\n        moe_sum_reduce(\n            intermediate_cache3,\n            output,\n            routed_scaling_factor,\n        )\n        return output\n', 'sglang/srt/layers/moe/moe_runner/marlin.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import TYPE_CHECKING, Optional\n\nimport torch\nimport triton\nimport triton.language as tl\n\nfrom sglang.srt.layers.moe.moe_runner.base import (\n    MoeQuantInfo,\n    MoeRunnerConfig,\n    RunnerInput,\n    RunnerOutput,\n    register_fused_func,\n)\nfrom sglang.srt.layers.moe.utils import MoeRunnerBackend\n\nif TYPE_CHECKING:\n    from sglang.srt.layers.moe.token_dispatcher import (\n        StandardCombineInput,\n        StandardDispatchOutput,\n    )\n\n\n@triton.jit\ndef _unpack_packed_topk_kernel(packed_ptr, ids_ptr, w_ptr, numel, BLOCK: tl.constexpr):\n    pid = tl.program_id(0)\n    offs = pid * BLOCK + tl.arange(0, BLOCK)\n    mask = offs < numel\n    packed = tl.load(\n        packed_ptr + offs, mask=mask\n    )  # int32 (id << 16) | bf16-weight-bits\n    tl.store(ids_ptr + offs, packed >> 16, mask=mask)  # expert id (high 16 bits)\n    wbits = (packed & 0xFFFF).to(tl.int16)  # bf16 weight bits (low 16 bits)\n    tl.store(\n        w_ptr + offs, wbits.to(tl.bfloat16, bitcast=True).to(tl.float32), mask=mask\n    )\n\n\ndef _fused_unpack_packed_topk(packed: torch.Tensor):\n    """Single-launch inverse of the fused topk pack ((id << 16) | bf16-weight-bits).\n\n    Returns (topk_ids int32, topk_weights float32). Collapses the ~5 elementwise\n    ops (shift / mask / int16 / bitcast / cast) the torch reference emits per call\n    into one Triton launch. Mirrors _pack_topk_kernel (trtllm_lora_temp/topk_pack).\n    """\n    packed = packed.contiguous()\n    ids = torch.empty_like(packed, dtype=torch.int32)\n    w = torch.empty(packed.shape, dtype=torch.float32, device=packed.device)\n    numel = packed.numel()\n    if numel:\n        BLOCK = 1024\n        _unpack_packed_topk_kernel[(triton.cdiv(numel, BLOCK),)](\n            packed, ids, w, numel, BLOCK=BLOCK\n        )\n    return ids, w\n\n\n@dataclass\nclass MarlinRunnerInput(RunnerInput):\n    """Input bundle passed to the Marlin runner core."""\n\n    hidden_states: torch.Tensor\n    topk_weights: torch.Tensor\n    topk_ids: torch.Tensor\n    router_logits: torch.Tensor\n\n    @property\n    def runner_backend(self) -> MoeRunnerBackend:\n        return MoeRunnerBackend.MARLIN\n\n\n@dataclass\nclass MarlinRunnerOutput(RunnerOutput):\n    """Output bundle returned from the Marlin runner core."""\n\n    hidden_states: torch.Tensor\n\n    @property\n    def runner_backend(self) -> MoeRunnerBackend:\n        return MoeRunnerBackend.MARLIN\n\n\n@dataclass\nclass MarlinMoeQuantInfo(MoeQuantInfo):\n    """Quantization payload consumed by the Marlin backend."""\n\n    w13_qweight: torch.Tensor\n    w2_qweight: torch.Tensor\n    w13_scales: torch.Tensor\n    w2_scales: torch.Tensor\n    w13_g_idx_sort_indices: Optional[torch.Tensor]\n    w2_g_idx_sort_indices: Optional[torch.Tensor]\n    weight_bits: int\n    use_atomic_add_override: Optional[bool] = None\n    experiment_capture_tag: Optional[str] = None\n\n    # GPTQ specific (Optional)\n    w13_g_idx: Optional[torch.Tensor] = None\n    w2_g_idx: Optional[torch.Tensor] = None\n    is_k_full: bool = True\n\n    # AWQ specific (Optional)\n    w13_qzeros: Optional[torch.Tensor] = None\n    w2_qzeros: Optional[torch.Tensor] = None\n\n    # Optional\n    expert_map: Optional[torch.Tensor] = None\n    global_num_experts: int = -1\n    w13_global_scale: Optional[torch.Tensor] = None\n    w2_global_scale: Optional[torch.Tensor] = None\n    w13_bias: Optional[torch.Tensor] = None\n    w2_bias: Optional[torch.Tensor] = None\n\n\n@register_fused_func("none", "marlin")\ndef fused_experts_none_to_marlin(\n    dispatch_output: StandardDispatchOutput,\n    quant_info: MarlinMoeQuantInfo,\n    runner_config: MoeRunnerConfig,\n) -> StandardCombineInput:\n    from sglang.srt.layers.moe.fused_moe_triton.fused_marlin_moe import fused_marlin_moe\n    from sglang.srt.layers.moe.token_dispatcher.standard import StandardCombineInput\n    from sglang.srt.layers.quantization.marlin_utils import marlin_make_workspace\n\n    hidden_states = dispatch_output.hidden_states\n    topk_output = dispatch_output.topk_output\n\n    # The fused gate+topk kernel (SGLANG_OPT_USE_FUSED_GATE_TOPK) emits a\n    # PackedTopKOutput -- int32 (expert_id << 16) | bf16-weight-bits -- for the\n    # FlashInfer trtllm routed kernel, which unpacks device-side. The Marlin\n    # runner reads topk_ids / topk_weights separately, so unpack it here.\n    if not hasattr(topk_output, "topk_weights"):\n        from sglang.srt.layers.moe.topk import StandardTopKOutput\n\n        topk_ids, topk_weights = _fused_unpack_packed_topk(topk_output.packed_topk_ids)\n        topk_output = StandardTopKOutput(\n            topk_weights=topk_weights,\n            topk_ids=topk_ids,\n            router_logits=topk_output.router_logits,\n        )\n\n    if runner_config.is_gated:\n        assert runner_config.activation in {\n            "silu",\n            "situ",\n        }, f"Only gated SiLU/SiTU is supported, got {runner_config.activation}."\n    elif runner_config.activation not in {"silu", "relu2"}:\n        raise ValueError(\n            f"Unsupported Marlin MoE activation: {runner_config.activation}"\n        )\n\n    # Use a per-call workspace so captured graphs cannot alias Marlin\'s\n    # inter-block reduction locks and deadlock during capture.\n    workspace = marlin_make_workspace(hidden_states.device, max_blocks_per_sm=4)\n\n    marlin_hidden_states = hidden_states\n    # Avoid aliasing the MoE input buffer until Marlin output semantics are\n    # fully validated across shared-expert and overlap paths.\n    marlin_inplace = False\n    if (\n        quant_info.weight_bits == 4\n        and quant_info.w13_qzeros is None\n        and quant_info.w2_qzeros is None\n        and quant_info.w13_scales.dtype == torch.float8_e8m0fnu\n        and quant_info.w2_scales.dtype == torch.float8_e8m0fnu\n        and hidden_states.dtype == torch.float16\n    ):\n        # MXFP4(E8M0) Marlin kernels are only numerically valid on the bf16\n        # activation path. The fp16 + E8M0 path is intentionally not generated\n        # in sgl-kernel, so upcast activations here and cast the result back.\n        marlin_hidden_states = hidden_states.to(torch.bfloat16)\n        marlin_inplace = False\n\n    output = fused_marlin_moe(\n        hidden_states=marlin_hidden_states,\n        w1=quant_info.w13_qweight,\n        w2=quant_info.w2_qweight,\n        w1_scale=quant_info.w13_scales,\n        w2_scale=quant_info.w2_scales,\n        gating_output=topk_output.router_logits,\n        topk_weights=topk_output.topk_weights,\n        topk_ids=topk_output.topk_ids,\n        global_num_experts=quant_info.global_num_experts,\n        expert_map=quant_info.expert_map,\n        g_idx1=quant_info.w13_g_idx,\n        g_idx2=quant_info.w2_g_idx,\n        sort_indices1=quant_info.w13_g_idx_sort_indices,\n        sort_indices2=quant_info.w2_g_idx_sort_indices,\n        w1_zeros=quant_info.w13_qzeros,\n        w2_zeros=quant_info.w2_qzeros,\n        w1_global_scale=quant_info.w13_global_scale,\n        w2_global_scale=quant_info.w2_global_scale,\n        w1_bias=quant_info.w13_bias,\n        w2_bias=quant_info.w2_bias,\n        workspace=workspace,\n        num_bits=quant_info.weight_bits,\n        is_k_full=quant_info.is_k_full,\n        inplace=marlin_inplace,\n        routed_scaling_factor=runner_config.routed_scaling_factor,\n        clamp_limit=(\n            runner_config.gemm1_clamp_limit\n            if runner_config.gemm1_alpha is not None\n            else runner_config.swiglu_limit\n        ),\n        gemm1_alpha=runner_config.gemm1_alpha,\n        activation=runner_config.activation,\n        is_gated=runner_config.is_gated,\n        use_atomic_add_override=quant_info.use_atomic_add_override,\n        experiment_capture_tag=quant_info.experiment_capture_tag,\n    ).to(hidden_states.dtype)\n\n    return StandardCombineInput(\n        hidden_states=output,\n    )\n\n\n# ===== TO BE REFACTORED ====\nfrom sglang.srt.lora.marlin_lora_temp import sgl_backend  # noqa: E402,F401\n\n# ===== END TO BE REFACTORED ====\n', 'sglang/srt/hardware_backend/gpu/quantization/gptq_kernels.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import TYPE_CHECKING, Callable, Optional\n\nimport torch\n\nfrom sglang.srt.layers.moe import MoeRunner, MoeRunnerBackend, get_moe_runner_backend\nfrom sglang.srt.layers.moe.moe_runner.marlin import MarlinMoeQuantInfo\nfrom sglang.srt.layers.parameter import BasevLLMParameter, permute_param_layout_\nfrom sglang.srt.layers.quantization.marlin_utils import (\n    apply_gptq_marlin_linear,\n    check_marlin_supports_shape,\n    marlin_is_k_full,\n    marlin_make_empty_g_idx,\n    marlin_make_workspace,\n    marlin_moe_permute_scales,\n    marlin_permute_scales,\n    marlin_sort_g_idx,\n    marlin_zero_points,\n)\nfrom sglang.srt.layers.quantization.utils import (\n    get_scalar_types,\n    replace_parameter,\n    unpack_cols,\n)\n\nif TYPE_CHECKING:\n    from sglang.srt.layers.moe import MoeRunnerConfig\n    from sglang.srt.layers.moe.token_dispatcher import (\n        CombineInput,\n        StandardDispatchOutput,\n    )\n    from sglang.srt.layers.quantization.base_config import QuantizationConfig\n\nScalarType, _ = get_scalar_types()\n\n\ndef _unsupported_kernel(*args, **kwargs):\n    raise RuntimeError("GPTQ CUDA kernels are unavailable on the current platform.")\n\n\ngptq_gemm = _unsupported_kernel\ngptq_marlin_repack = _unsupported_kernel\ngptq_shuffle = _unsupported_kernel\n\ntry:\n    from sgl_kernel import gptq_gemm, gptq_shuffle\n\n    from sglang.kernels.ops.quantization.gptq_marlin_repack import gptq_marlin_repack\nexcept Exception:\n    pass\n\n\n@dataclass\nclass MarlinLinearLayerConfig:\n    full_weight_shape: tuple[int, int]  # [in, out]\n    partition_weight_shape: tuple[int, int]\n    weight_type: ScalarType\n    act_type: torch.dtype\n    group_size: int\n    zero_points: bool\n    has_g_idx: bool\n\n\ndef gptq_marlin_moe_repack(\n    b_q_weight: torch.Tensor,\n    perm: torch.Tensor,\n    size_k: int,\n    size_n: int,\n    num_bits: int,\n) -> torch.Tensor:\n    num_experts = b_q_weight.shape[0]\n    assert size_k % 16 == 0\n    output = torch.empty(\n        (num_experts, size_k // 16, size_n * (num_bits // 2)),\n        device=b_q_weight.device,\n        dtype=b_q_weight.dtype,\n    )\n    for e in range(num_experts):\n        output[e] = gptq_marlin_repack(b_q_weight[e], perm[e], size_k, size_n, num_bits)\n    return output\n\n\nclass GPTQLinearKernel:\n    def __init__(self, quant_config: Optional[QuantizationConfig] = None):\n        self.quant_config = quant_config\n        self.use_shuffle = True\n\n    def process_weights_after_loading(self, layer: torch.nn.Module) -> None:\n        # for torch.compile\n        layer.qzeros = torch.nn.Parameter(layer.qzeros.data, requires_grad=False)\n        layer.qweight = torch.nn.Parameter(layer.qweight.data, requires_grad=False)\n        layer.g_idx = torch.nn.Parameter(layer.g_idx.data, requires_grad=False)\n        layer.scales = torch.nn.Parameter(layer.scales.data, requires_grad=False)\n\n        # exllama needs to shuffle the weight after the weight is loaded\n        # here we do the shuffle on first forward pass\n        if self.use_shuffle:\n            if self.quant_config.desc_act:\n                layer.g_idx.data = torch.argsort(layer.g_idx).to(torch.int)\n            else:\n                layer.g_idx.data = torch.empty(\n                    (0,), dtype=torch.int, device=layer.g_idx.device\n                )\n            gptq_shuffle(layer.qweight, layer.g_idx, self.quant_config.weight_bits)\n\n    def apply(\n        self,\n        layer: torch.nn.Module,\n        x: torch.Tensor,\n        bias: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        out_shape = x.shape[:-1] + (layer.qweight.shape[-1],)\n        reshaped_x = x.reshape(-1, x.shape[-1])\n\n        output = gptq_gemm(\n            reshaped_x,\n            layer.qweight,\n            layer.qzeros,\n            layer.scales,\n            layer.g_idx,\n            self.use_shuffle,\n            self.quant_config.weight_bits,\n        )\n        if bias is not None:\n            output.add_(bias)\n        return output.reshape(out_shape)\n\n\nclass GPTQMarlinLinearKernel:\n    def __init__(self, quant_config: Optional[QuantizationConfig] = None):\n        self.quant_config = quant_config\n\n    def process_weights_after_loading(self, layer: torch.nn.Module) -> None:\n        device = getattr(layer, "qweight").device\n        c = self.kernel_config\n\n        check_marlin_supports_shape(\n            c.partition_weight_shape[1],  # out_features\n            c.partition_weight_shape[0],  # in_features\n            c.full_weight_shape[0],  # in_features\n            c.group_size,\n        )\n\n        row_parallel = c.partition_weight_shape[0] != c.full_weight_shape[0]\n        self.is_k_full = marlin_is_k_full(c.has_g_idx, row_parallel)\n\n        # Allocate marlin workspace.\n        self.workspace = marlin_make_workspace(device)\n\n        # Default names since marlin requires empty parameters for these,\n        # TODO: remove this requirement from marlin (allow optional tensors)\n        self.w_q_name = "qweight"\n        self.w_s_name = "scales"\n        self.w_zp_name = "qzeros"\n        self.w_gidx_name = "g_idx"\n\n        def _transform_param(\n            layer: torch.nn.Module, name: Optional[str], fn: Callable\n        ) -> None:\n            if name is not None and getattr(layer, name, None) is not None:\n\n                old_param = getattr(layer, name)\n                new_param = fn(old_param)\n                # replace the parameter with torch.nn.Parameter for TorchDynamo\n                # compatibility\n                replace_parameter(\n                    layer, name, torch.nn.Parameter(new_param.data, requires_grad=False)\n                )\n\n        def transform_w_q(x):\n            assert isinstance(x, BasevLLMParameter)\n            permute_param_layout_(x, input_dim=0, output_dim=1, packed_dim=0)\n            x.data = gptq_marlin_repack(\n                x.data.contiguous(),\n                perm=layer.g_idx_sort_indices,\n                size_k=c.partition_weight_shape[0],\n                size_n=c.partition_weight_shape[1],\n                num_bits=c.weight_type.size_bits,\n            )\n            return x\n\n        def transform_w_s(x):\n            assert isinstance(x, BasevLLMParameter)\n            permute_param_layout_(x, input_dim=0, output_dim=1)\n            x.data = marlin_permute_scales(\n                x.data.contiguous(),\n                size_k=c.partition_weight_shape[0],\n                size_n=c.partition_weight_shape[1],\n                group_size=c.group_size,\n            )\n            return x\n\n        if c.has_g_idx:\n            g_idx, g_idx_sort_indices = marlin_sort_g_idx(\n                getattr(layer, self.w_gidx_name)\n            )\n            _transform_param(layer, self.w_gidx_name, lambda _: g_idx)\n            layer.g_idx_sort_indices = g_idx_sort_indices\n        else:\n            setattr(layer, self.w_gidx_name, marlin_make_empty_g_idx(device))\n            layer.g_idx_sort_indices = marlin_make_empty_g_idx(device)\n\n        if c.zero_points:\n            grouped_k = (\n                c.partition_weight_shape[0] // c.group_size if c.group_size != -1 else 1\n            )\n            _transform_param(\n                layer,\n                self.w_zp_name,\n                lambda x: marlin_zero_points(\n                    unpack_cols(\n                        x.t(),\n                        c.weight_type.size_bits,\n                        grouped_k,\n                        c.partition_weight_shape[1],\n                    ),\n                    size_k=grouped_k,\n                    size_n=c.partition_weight_shape[1],\n                    num_bits=c.weight_type.size_bits,\n                ),\n            )\n        else:\n            setattr(layer, self.w_zp_name, marlin_make_empty_g_idx(device))\n        _transform_param(layer, self.w_q_name, transform_w_q)\n        _transform_param(layer, self.w_s_name, transform_w_s)\n\n    def apply(\n        self,\n        layer: torch.nn.Module,\n        x: torch.Tensor,\n        bias: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        c = self.kernel_config\n\n        def _get_weight_params(\n            layer: torch.nn.Module,\n        ) -> tuple[\n            torch.Tensor,  # w_q\n            torch.Tensor,  # w_s\n            Optional[torch.Tensor],  # w_zp,\n            Optional[torch.Tensor],  # w_gidx\n        ]:\n            return (\n                getattr(layer, self.w_q_name),\n                getattr(layer, self.w_s_name),\n                getattr(layer, self.w_zp_name or "", None),\n                getattr(layer, self.w_gidx_name or "", None),\n            )\n\n        w_q, w_s, w_zp, w_gidx = _get_weight_params(layer)\n\n        # `process_weights_after_loading` will ensure w_zp and w_gidx are not\n        #  None for marlin\n        return apply_gptq_marlin_linear(\n            input=x,\n            weight=w_q,\n            weight_scale=w_s,\n            weight_zp=w_zp,  # type: ignore\n            g_idx=w_gidx,  # type: ignore\n            g_idx_sort_indices=layer.g_idx_sort_indices,\n            workspace=self.workspace,\n            wtype=c.weight_type,\n            input_size_per_partition=c.partition_weight_shape[0],\n            output_size_per_partition=c.partition_weight_shape[1],\n            is_k_full=self.is_k_full,\n            bias=bias,\n        )\n\n\nclass GPTQMarlinMoEKernel:\n    def __init__(self, quant_config: Optional[QuantizationConfig] = None):\n        self.quant_config = quant_config\n\n    def process_weights_after_loading(self, layer: torch.nn.Module) -> None:\n\n        # Process act_order\n        if self.quant_config.desc_act:\n            # Get sorting based on g_idx\n            num_experts = layer.w13_g_idx.shape[0]\n            w13_g_idx_sort_indices = torch.empty_like(layer.w13_g_idx)\n            w2_g_idx_sort_indices = torch.empty_like(layer.w2_g_idx)\n            w13_sorted_g_idx = torch.empty_like(layer.w13_g_idx)\n            w2_sorted_g_idx = torch.empty_like(layer.w2_g_idx)\n            for e in range(num_experts):\n                w13_g_idx_sort_indices[e] = torch.argsort(layer.w13_g_idx[e]).to(\n                    torch.int32\n                )\n                w2_g_idx_sort_indices[e] = torch.argsort(layer.w2_g_idx[e]).to(\n                    torch.int32\n                )\n                w13_sorted_g_idx[e] = layer.w13_g_idx[e][w13_g_idx_sort_indices[e]]\n                w2_sorted_g_idx[e] = layer.w2_g_idx[e][w2_g_idx_sort_indices[e]]\n            replace_parameter(layer, "w13_g_idx", w13_sorted_g_idx)\n            replace_parameter(layer, "w2_g_idx", w2_sorted_g_idx)\n            replace_parameter(layer, "w13_g_idx_sort_indices", w13_g_idx_sort_indices)\n            replace_parameter(layer, "w2_g_idx_sort_indices", w2_g_idx_sort_indices)\n        else:\n            # Reset g_idx related tensors\n            num_experts = layer.w13_g_idx.shape[0]\n            device = layer.w13_g_idx.device\n            layer.w13_g_idx = torch.nn.Parameter(\n                torch.empty((num_experts, 0), dtype=torch.int32, device=device),\n                requires_grad=False,\n            )\n            layer.w2_g_idx = torch.nn.Parameter(\n                torch.empty((num_experts, 0), dtype=torch.int32, device=device),\n                requires_grad=False,\n            )\n            layer.w13_g_idx_sort_indices = torch.nn.Parameter(\n                torch.empty((num_experts, 0), dtype=torch.int32, device=device),\n                requires_grad=False,\n            )\n            layer.w2_g_idx_sort_indices = torch.nn.Parameter(\n                torch.empty((num_experts, 0), dtype=torch.int32, device=device),\n                requires_grad=False,\n            )\n        # Repack weights\n        marlin_w13_qweight = gptq_marlin_moe_repack(\n            layer.w13_qweight,\n            layer.w13_g_idx_sort_indices,\n            layer.w13_qweight.shape[1] * self.quant_config.pack_factor,\n            layer.w13_qweight.shape[2],\n            self.quant_config.weight_bits,\n        )\n        replace_parameter(layer, "w13_qweight", marlin_w13_qweight)\n        marlin_w2_qweight = gptq_marlin_moe_repack(\n            layer.w2_qweight,\n            layer.w2_g_idx_sort_indices,\n            layer.w2_qweight.shape[1] * self.quant_config.pack_factor,\n            layer.w2_qweight.shape[2],\n            self.quant_config.weight_bits,\n        )\n        replace_parameter(layer, "w2_qweight", marlin_w2_qweight)\n        # Repack scales\n        marlin_w13_scales = marlin_moe_permute_scales(\n            s=layer.w13_scales,\n            size_k=layer.intermediate_size_per_partition,\n            size_n=layer.w13_scales.shape[2],\n            group_size=self.quant_config.group_size,\n        )\n        replace_parameter(layer, "w13_scales", marlin_w13_scales)\n        marlin_w2_scales = marlin_moe_permute_scales(\n            s=layer.w2_scales,\n            size_k=layer.w2_scales.shape[1]\n            * (\n                self.quant_config.group_size\n                if self.quant_config.group_size != -1\n                else self.quant_config.pack_factor\n            ),\n            size_n=layer.w2_scales.shape[2],\n            group_size=self.quant_config.group_size,\n        )\n        replace_parameter(layer, "w2_scales", marlin_w2_scales)\n\n    def create_moe_runner(\n        self, layer: torch.nn.Module, moe_runner_config: MoeRunnerConfig\n    ):\n        assert get_moe_runner_backend().is_auto()\n        self.moe_runner_config = moe_runner_config\n        self.runner = MoeRunner(MoeRunnerBackend.MARLIN, moe_runner_config)\n\n    def apply(\n        self,\n        layer: torch.nn.Module,\n        dispatch_output: StandardDispatchOutput,\n    ) -> CombineInput:\n        quant_info = MarlinMoeQuantInfo(\n            w13_qweight=layer.w13_qweight,\n            w2_qweight=layer.w2_qweight,\n            w13_scales=layer.w13_scales,\n            w2_scales=layer.w2_scales,\n            w13_g_idx=layer.w13_g_idx,\n            w2_g_idx=layer.w2_g_idx,\n            w13_g_idx_sort_indices=layer.w13_g_idx_sort_indices,\n            w2_g_idx_sort_indices=layer.w2_g_idx_sort_indices,\n            weight_bits=self.quant_config.weight_bits,\n            use_atomic_add_override=getattr(self, "experiment_atomic_override", None),\n            experiment_capture_tag=getattr(self, "experiment_capture_tag", None),\n            is_k_full=self.is_k_full,\n        )\n\n        return self.runner.run(dispatch_output, quant_info)\n', 'sglang/srt/model_executor/model_runner.py': '# Copyright 2023-2024 SGLang Team\n# Licensed under the Apache License, Version 2.0 (the "License");\n# you may not use this file except in compliance with the License.\n# You may obtain a copy of the License at\n#\n#     http://www.apache.org/licenses/LICENSE-2.0\n#\n# Unless required by applicable law or agreed to in writing, software\n# distributed under the License is distributed on an "AS IS" BASIS,\n# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n# See the License for the specific language governing permissions and\n# limitations under the License.\n# ==============================================================================\n"""ModelRunner runs the forward passes of the models."""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport inspect\nimport logging\nimport time\nfrom dataclasses import dataclass\nfrom typing import Optional, Union\n\nimport torch\nimport torch.distributed as dist\n\nfrom sglang.srt.configs.load_config import LoadConfig\nfrom sglang.srt.configs.model_config import (\n    AttentionArch,\n    ModelConfig,\n    ModelImpl,\n)\nfrom sglang.srt.configs.update_config import adjust_config_with_unaligned_cpu_tp\nfrom sglang.srt.debug_utils.dumper import dumper\nfrom sglang.srt.distributed import bootstrap\nfrom sglang.srt.distributed.device_communicators.mooncake_transfer_engine import (\n    maybe_init_shared_mooncake_transfer_engine,\n)\nfrom sglang.srt.distributed.parallel_state_wrapper import ParallelState\nfrom sglang.srt.dllm.config import DllmConfig\nfrom sglang.srt.elastic_ep.elastic_ep import (\n    ElasticEPStateManager,\n    get_healthy_expert_location_src_rank,\n    get_scale_cohort_target,\n    join_process_groups,\n    join_scale_process_group,\n    maybe_rebalance_after_rank_fault,\n    maybe_recover_ep_ranks,\n    register_scale_cohort,\n    try_admit_scale_ranks,\n)\nfrom sglang.srt.elastic_ep.expert_backup_client import ExpertBackupClient\nfrom sglang.srt.environ import envs\nfrom sglang.srt.eplb.eplb_manager import EPLBManager\nfrom sglang.srt.eplb.expert_distribution import (\n    ExpertDistributionMetrics,\n    ExpertDistributionRecorder,\n    get_global_expert_distribution_recorder,\n    set_global_expert_distribution_recorder,\n)\nfrom sglang.srt.eplb.expert_location import (\n    ExpertLocationMetadata,\n    append_trivial_expert_slots,\n    broadcast_global_expert_location_metadata,\n    compute_initial_expert_location_metadata,\n    format_expert_location_layout,\n    get_global_expert_location_metadata,\n    set_global_expert_location_metadata,\n)\nfrom sglang.srt.eplb.expert_location_updater import ExpertLocationUpdater\nfrom sglang.srt.kv_canary.api import install_canary\nfrom sglang.srt.kv_canary.runner.canary_manager import context_tuple\nfrom sglang.srt.kv_canary.token_oracle.install import install_token_oracle_from_env\nfrom sglang.srt.layers import deep_gemm_wrapper, model_parallel\nfrom sglang.srt.layers.attention.dsa.utils import is_dsa_enable_prefill_cp\nfrom sglang.srt.layers.cp.utils import (\n    get_cp_strategy,\n    is_cp_v2_active,\n)\nfrom sglang.srt.layers.logits_processor import LogitsProcessorOutput\nfrom sglang.srt.layers.sampler import create_sampler\nfrom sglang.srt.layers.utils.cp_utils import is_mla_prefill_cp_enabled\nfrom sglang.srt.lora.lora_manager import LoRAManager, init_lora_cuda_graph_moe_buffers\nfrom sglang.srt.lora.lora_registry import LoRARef\nfrom sglang.srt.managers.schedule_batch import sanity_check_mm_pad_shift_value\nfrom sglang.srt.mem_cache import kv_cache_dtype\nfrom sglang.srt.mem_cache.allocator import BaseTokenToKVPoolAllocator\nfrom sglang.srt.mem_cache.kv_cache_configurator import (\n    KVCacheConfigurator,\n)\nfrom sglang.srt.mem_cache.memory_pool import HybridReqToTokenPool, ReqToTokenPool\nfrom sglang.srt.model_executor.cuda_graph_config import (\n    cuda_graph_fully_disabled,\n)\nfrom sglang.srt.model_executor.forward_batch_info import (\n    ForwardBatch,\n    PPProxyTensors,\n)\nfrom sglang.srt.model_executor.forward_context import (\n    ForwardContext,\n    forward_context,\n    has_forward_context,\n)\nfrom sglang.srt.model_executor.graph_memory_usage import (\n    replace_graph_memory_usage,\n    replace_graph_time_usage,\n)\nfrom sglang.srt.model_executor.model_runner_components import misc_utils\nfrom sglang.srt.model_executor.model_runner_components.attention_backend_setup import (\n    build_attention_backends,\n    configure_aux_hidden_state_capture,\n    get_attention_backend,\n    resolve_attention_backend_strs,\n)\nfrom sglang.srt.model_executor.model_runner_components.cuda_graph_setup import (\n    capture_cuda_graphs,\n    capture_decode_graph,\n    capture_prefill_graph,\n)\nfrom sglang.srt.model_executor.model_runner_components.kv_pool_runtime import (\n    compute_post_capture_kv_resize,\n    is_post_capture_kv_active,\n)\nfrom sglang.srt.model_executor.model_runner_components.layer_setup import (\n    ModelLayerInfo,\n    adjust_hybrid_swa_layer_ids,\n    resolve_layer_indices,\n)\nfrom sglang.srt.model_executor.model_runner_components.load_model_utils import (\n    build_load_config,\n    dist_barrier_after_load,\n    load_kv_cache_scales,\n    load_model_with_memory_saver,\n    maybe_downgrade_dtype_for_legacy_gpu,\n    maybe_enable_ipc_weight_cache,\n    maybe_precompile_model_kernels_after_loading,\n    maybe_register_debug_tensor_dump_hook,\n    maybe_trigger_remote_instance_nccl_send_group,\n    report_online_quantization,\n    resolve_sliding_window_size,\n)\nfrom sglang.srt.model_executor.model_runner_components.moe_ep_setup import (\n    check_quantized_moe_compatibility,\n    init_lplb_solvers,\n    prepare_moe_topk,\n)\nfrom sglang.srt.model_executor.model_runner_components.ngram_embedding_manager import (\n    NgramEmbeddingManager,\n)\nfrom sglang.srt.model_executor.model_runner_components.remote_instance_weight_transporter import (\n    RemoteInstanceWeightTransporter,\n)\nfrom sglang.srt.model_executor.model_runner_components.spec_aux_hidden_state import (\n    SpecAuxHiddenStateConfig,\n    resolve_spec_aux_hidden_state_config,\n)\nfrom sglang.srt.model_executor.model_runner_components.weight_exporter import (\n    WeightExporter,\n)\nfrom sglang.srt.model_executor.model_runner_components.weight_updater import (\n    WeightUpdater,\n)\nfrom sglang.srt.model_executor.pool_configurator import MemoryPoolConfig\nfrom sglang.srt.model_executor.runner import (\n    EagerRunner,\n    get_batch_sizes_to_capture,\n)\nfrom sglang.srt.platforms import current_platform\nfrom sglang.srt.runtime_context import (\n    assert_published,\n    get_context,\n    get_device,\n    get_exec,\n    get_global_dwdp_manager,\n    get_lora,\n    get_memory,\n    get_model,\n    get_observability,\n    get_parallel,\n    get_schedule,\n    get_spec,\n    is_ep_joiner,\n    is_ep_scale_joiner,\n    remote_instance_transfer_engine_enabled,\n    set_global_dwdp_manager,\n)\nfrom sglang.srt.sampling.sampling_batch_info import SamplingBatchInfo\nfrom sglang.srt.sampling.sampling_observer import SamplingObserver\nfrom sglang.srt.server_args import (  # noqa: F401  (re-export)\n    CHUNKED_PREFIX_CACHE_SUPPORTED_ATTENTION_BACKENDS,\n    ServerArgs,\n    add_chunked_prefix_cache_attention_backend,\n    get_global_server_args,\n)\nfrom sglang.srt.speculative.spec_info import SpeculativeAlgorithm\nfrom sglang.srt.speculative.spec_utils import resolve_num_tokens_per_req\nfrom sglang.srt.state_capturer.base import TopkCaptureOutput\nfrom sglang.srt.state_capturer.indexer_topk import (\n    create_indexer_capturer,\n    get_global_indexer_capturer,\n    set_global_indexer_capturer,\n)\nfrom sglang.srt.state_capturer.routed_experts import (\n    RoutedExpertsCapturer,\n    disable_routed_experts_capture_for_draft,\n    get_global_experts_capturer,\n    set_global_experts_capturer,\n)\nfrom sglang.srt.utils import (\n    cpu_has_amx_support,\n    enable_show_time_cost,\n    get_available_gpu_memory,\n    is_host_cpu_arm64,\n    is_npu,\n    numa_utils,\n    require_gathered_buffer,\n    reserve_rope_cache_for_long_sequences,\n    set_cuda_arch,\n    slow_rank_detector,\n)\nfrom sglang.srt.utils.device_timer import device_timer_ctx\nfrom sglang.srt.utils.nvtx_pytorch_hooks import PytHooks\nfrom sglang.srt.utils.nvtx_utils import profile_range\nfrom sglang.srt.utils.offloader import (\n    create_offloader,\n    get_offloader,\n    set_offloader,\n)\nfrom sglang.srt.utils.profile_utils import build_step_span_name\nfrom sglang.srt.utils.torch_memory_saver_adapter import TorchMemorySaverAdapter\nfrom sglang.srt.utils.weight_checker import WeightChecker\n\n_is_npu = is_npu()\n_is_cpu_amx_available = cpu_has_amx_support()\n_is_cpu_arm64 = is_host_cpu_arm64()\n\nif _is_npu:\n    from sglang.srt.hardware_backend.npu.utils import init_npu_backend\n\n    init_npu_backend()\nelif current_platform.is_out_of_tree():\n    current_platform.init_backend()\n\n# Detect stragger ranks in model loading\n\n\nlogger = logging.getLogger(__name__)\n\n\ndef _prefill_cuda_graph_allows_context_parallel(\n    prefill_runner, forward_batch: ForwardBatch\n) -> bool:\n    """Allow CP only through a runner that captured the validated CP-v2 body."""\n    return get_cp_strategy() is None or (\n        bool(getattr(prefill_runner, "enable_cp_v2_bcg_capture", False))\n        and is_cp_v2_active(forward_batch)\n    )\n\n\n@dataclass\nclass ModelRunnerOutput:\n    logits_output: Union[LogitsProcessorOutput, PPProxyTensors]\n    can_run_graph: bool\n    expert_distribution_metrics: Optional[ExpertDistributionMetrics] = None\n    routed_experts_output: Optional[TopkCaptureOutput] = None\n    indexer_topk_output: Optional[TopkCaptureOutput] = None\n\n\ndef resolve_draft_attention_backend(\n    *,\n    draft_attention_backend: Optional[str],\n    is_draft_worker: bool,\n) -> Optional[str]:\n    """The attention backend a runner uses because it is a draft runner.\n\n    ``None`` for a target runner. For a draft: the backend the algorithm that\n    built it resolved (the supported-backend fallback in\n    ``build_draft_tp_worker``), else ``--speculative-draft-attention-backend``.\n    It belongs to the runner, not the process: target and draft coexist.\n    """\n    if not is_draft_worker:\n        return None\n    return draft_attention_backend or get_spec().speculative_draft_attention_backend\n\n\nclass ModelRunner:\n    """ModelRunner runs the forward passes of the models."""\n\n    @property\n    def sampling_observer(self) -> Optional[SamplingObserver]:\n        return self._sampling_observer\n\n    @sampling_observer.setter\n    def sampling_observer(self, observer: Optional[SamplingObserver]) -> None:\n        if observer is not None and not self.supports_sampling_observer():\n            raise ValueError(\n                "sampling observers are not supported by the configured "\n                "sampling path"\n            )\n        self._sampling_observer = observer\n\n    def supports_sampling_observer(self) -> bool:\n        """Whether this runner\'s sampling path publishes observer output."""\n        return self.server_args.dllm_algorithm is None and self.spec_algorithm.is_none()\n\n    def __init__(\n        self,\n        model_config: ModelConfig,\n        mem_fraction_static: float,\n        gpu_id: int,\n        ps: ParallelState,\n        nccl_port: int,\n        server_args: ServerArgs,\n        is_draft_worker: bool = False,\n        req_to_token_pool: Optional[ReqToTokenPool] = None,\n        token_to_kv_pool_allocator: Optional[BaseTokenToKVPoolAllocator] = None,\n        memory_pool_config: Optional[MemoryPoolConfig] = None,\n        draft_model_idx: Optional[int] = None,\n        draft_attention_backend: Optional[str] = None,\n    ):\n        # Parse args\n        self.mem_fraction_static = mem_fraction_static\n        # Set on target by `_resolve_memory_pool_config`; passed in for draft\n        # workers so they reuse target\'s resolved sizes (replaces legacy\n        # `server_args._draft_pool_config` mutation hack).\n        self.memory_pool_config = memory_pool_config\n        self.gpu_id = gpu_id\n        self.ps = ps\n        self.model_config = model_config\n        self.dist_port = nccl_port\n        self.server_args = server_args\n        self.is_draft_worker = is_draft_worker\n        # The process entry published; a draft runner is not one (it must not\n        # clobber the target\'s config), so only the target checks.\n        if not is_draft_worker:\n            assert_published(server_args, role="scheduler")\n        # Set by maybe_init_lora_manager; stays None when LoRA is off and on\n        # draft runners, which serve adapters\' target model unadapted.\n        self.lora_manager: Optional[LoRAManager] = None\n        self.device = get_device().device\n        self.draft_attention_backend = resolve_draft_attention_backend(\n            draft_attention_backend=draft_attention_backend,\n            is_draft_worker=is_draft_worker,\n        )\n        # This runner\'s own load format, resolved before anything keys off it:\n        # the remote-instance transfer engine is initialized at the top of\n        # initialize(), long before the weights are loaded.\n        self.draft_load_format = self._resolve_draft_load_format()\n        self.is_generation = model_config.is_generation\n        self.device_timer = None\n        self.is_multimodal = model_config.is_multimodal\n        self.is_multimodal_chunked_prefill_supported = (\n            model_config.is_multimodal_chunked_prefill_supported\n        )\n        self.spec_algorithm = SpeculativeAlgorithm.from_string(\n            get_spec().speculative_algorithm\n        )\n        self.capture_tail_hooks = []\n        self.page_size = get_schedule().page_size\n        self.req_to_token_pool = req_to_token_pool\n        self.token_to_kv_pool_allocator = token_to_kv_pool_allocator\n        self.mtp_draft_device_pools = ()\n        self.is_hybrid_swa = model_config.is_hybrid_swa\n        self.is_hybrid_swa_compress = model_config.is_hybrid_swa_compress\n        self.use_mla_backend = self.model_config.attention_arch == AttentionArch.MLA\n        self.attention_chunk_size = model_config.attention_chunk_size\n        self.enable_elastic_ep = get_exec().moe.elastic_ep_backend is not None\n        self.forward_pass_id = 0\n        self._pending_elastic_scale_update = None\n        self.init_new_workspace = False\n        self.draft_model_idx = draft_model_idx\n        self.enable_hisparse = get_memory().enable_hisparse\n        self._sampling_observer: Optional[SamplingObserver] = None\n\n        self.init_startup_observability()\n\n        self.init_remote_instance_weight_transporter()\n\n        self.init_msprobe()\n\n        # auxiliary hidden capture mode. TODO: expose this to server args?\n        self.init_spec_aux_hidden_state()\n\n        # Apply the rank zero filter to logger\n        if get_observability().show_time_cost:\n            enable_show_time_cost()\n\n        misc_utils.maybe_disable_chunked_prefix_cache(\n            use_mla_backend=self.use_mla_backend,\n            is_draft_worker=self.is_draft_worker,\n        )\n\n        # Init OpenMP threads binding for CPU\n        if self.device == "cpu":\n            self.init_threads_binding()\n\n        # Set float32 matmul precision\n        if get_exec().features.enable_tf32_matmul:\n            torch.set_float32_matmul_precision("high")\n\n        # Set device early so that TransferEngine init (e.g. Ascend NPU)\n        # can access the device context.\n        try:\n            torch.get_device_module(self.device).set_device(ps.gpu_id)\n        except Exception:\n            import os\n\n            logger.warning(\n                f"Context: {self.device=} {ps.gpu_id=} {os.environ.get(\'CUDA_VISIBLE_DEVICES\')=} {ps.tp_rank=} {ps.tp_size=}"\n            )\n            raise\n\n        # Initialize MooncakeTransferEngine BEFORE init_torch_distributed so\n        # that the shared TE can be passed to the Mooncake PG backend (avoids\n        # creating duplicate TransferEngines).\n        self.init_shared_mooncake_transfer_engine()\n\n        # Get available memory before model loading.\n        # Stored for later use by alloc_memory_pool().\n        self.init_torch_distributed()\n\n        # Init forward stream for overlap schedule\n        self.forward_stream = torch.get_device_module(self.device).Stream()\n\n        # Read-done mailbox: the scheduler\'s WAR barrier reads it from the runner\n        # its worker names, and treats None as the coarse whole-forward fence.\n        self.shared_read_done_event: Optional[torch.cuda.Event] = None\n\n        # CPU offload\n        set_offloader(create_offloader(dp_rank=self.ps.dp_rank))\n\n        self._weight_checker = WeightChecker(get_model=lambda: self.model, ps=self.ps)\n\n        if envs.SGLANG_DETECT_SLOW_RANK.get():\n            slow_rank_detector.execute()\n\n        # Init mindspore running environment when model impl is "mindspore"\n        self.init_mindspore_runner()\n\n        # Update deep gemm configure\n        if deep_gemm_wrapper.ENABLE_JIT_DEEPGEMM:\n            deep_gemm_wrapper.update_deep_gemm_config(gpu_id, server_args)\n\n        # For hisparse (must be set before initialize() so CUDA graph capture can see it)\n        self.hisparse_coordinator = None\n\n        # The native overlap path replaces this during load_model(). Keep the\n        # no-pending-work invariant for lightweight backends that override the\n        # base initialization and weight-loading flow.\n        self.startup_weight_load = None\n\n        # Load model weights and configure\n        self.initialize()\n        self.check_quantized_moe_compatibility()\n\n        self._initialize_elastic_ep_joiner()\n\n        if self.is_multimodal:\n            sanity_check_mm_pad_shift_value(self.model_config.vocab_size)\n\n        # Temporary cached values\n        self.support_pp = (\n            "pp_proxy_tensors" in inspect.signature(self.model.forward).parameters\n        )\n\n        if self.ps.pp_size > 1:\n            assert (\n                self.support_pp\n            ), "Pipeline Parallel is not compatible with this model."\n\n        # For weight updates\n        self.init_weight_updater()\n        self.init_weight_exporter()\n\n    def init_startup_observability(self) -> None:\n        self.weight_load_time = 0.0\n        self.graph_memory_usage: dict[str, float] = {}\n        self.graph_time_usage: dict[str, float] = {}\n\n    def _initialize_elastic_ep_joiner(self) -> None:\n        if not (get_exec().moe.elastic_ep_backend is not None and is_ep_scale_joiner()):\n            return\n\n        join_effective_ep_size = (\n            get_parallel().config.ep_join_rank_offset + self.ps.tp_size\n        )\n        dist.barrier(group=self.tp_group.cpu_group)\n        if self.ps.tp_rank == 0:\n            register_scale_cohort(\n                get_parallel().config.ep_join_rank_offset,\n                join_effective_ep_size,\n            )\n        join_scale_process_group()\n        get_context().override("elastic_ep.scale_join", ep_size=join_effective_ep_size)\n\n        global_ep_rank = self.ps.tp_rank + get_parallel().config.ep_join_rank_offset\n        broadcast_global_expert_location_metadata(\n            model_config=self.model_config,\n            moe_ep_rank=global_ep_rank,\n            src_rank=0,\n        )\n        set_global_expert_distribution_recorder(\n            ExpertDistributionRecorder.init_new(\n                self.server_args,\n                get_global_expert_location_metadata(),\n                rank=global_ep_rank,\n            )\n        )\n\n        from sglang.srt.layers.dp_attention import (\n            enable_joiner_all_gather,\n            update_dp_attention_post_scale,\n        )\n\n        enable_joiner_all_gather()\n        update_dp_attention_post_scale(\n            new_dp_size=join_effective_ep_size,\n            new_dp_rank=global_ep_rank,\n        )\n        get_context().override("elastic_ep.scale_join", dp_size=join_effective_ep_size)\n        if self.eplb_manager is not None:\n            self.eplb_manager.disable_rebalance(\n                "EPLB rebalance is disabled while elastic EP scale-up "\n                "is being finalized"\n            )\n\n        state = ElasticEPStateManager.instance()\n        if state is not None:\n            state.active_ranks.zero_()\n            state.active_ranks[:join_effective_ep_size] = 1\n            state.snapshot_active_to_last()\n            state.sync_active_to_cpu()\n            state.scale_phase = "syncing_new_world"\n        self._elastic_scale_ready_barrier(\n            target_size=join_effective_ep_size,\n            log_tag="JOINER",\n        )\n        if state is not None:\n            state.scale_phase = "serving_expanded"\n        self._rearm_eplb_after_elastic_scale()\n\n    def init_msprobe(self):\n        self.msprobe_debugger = misc_utils.create_msprobe_debugger(self.server_args)\n\n    def init_weight_updater(self):\n        self.weight_updater = WeightUpdater(\n            tp_rank=self.ps.tp_rank,\n            device=self.device,\n            gpu_id=self.gpu_id,\n            model_config=self.model_config,\n            custom_weight_loaders=get_model().custom_weight_loader,\n            get_model=lambda: self.model,\n            update_model_fields=self.update_model_fields,\n            recapture_cuda_graph=self.init_decode_cuda_graph,\n            get_model_runner=lambda: self,\n        )\n\n    def init_spec_aux_hidden_state(self):\n        self.spec_aux_config: SpecAuxHiddenStateConfig = (\n            resolve_spec_aux_hidden_state_config(\n                server_args=self.server_args,\n                model_config=self.model_config,\n                spec_algorithm=self.spec_algorithm,\n                is_draft_worker=self.is_draft_worker,\n            )\n        )\n\n    def init_weight_exporter(self):\n        self.weight_exporter = WeightExporter(\n            tp_rank=self.ps.tp_rank,\n            tp_size=self.ps.tp_size,\n            gpu_id=self.gpu_id,\n            get_model_path=lambda: self.model_config.model_path,\n            get_model=lambda: self.model,\n        )\n\n    def init_remote_instance_weight_transporter(self):\n        self.remote_instance_weight_transporter = RemoteInstanceWeightTransporter(\n            get_model=lambda: self.model,\n            tp_rank=self.ps.tp_rank,\n            gpu_id=self.gpu_id,\n        )\n\n    def init_ngram_embedding_manager(self):\n        self.ngram_embedding_manager = NgramEmbeddingManager.from_model(\n            model=self.model,\n            model_config=self.model_config,\n            req_to_token_pool=self.req_to_token_pool,\n            max_running_requests=self.max_running_requests,\n            device=self.device,\n        )\n\n    def init_kv_cache_configurator(self):\n        self.kv_cache_configurator = KVCacheConfigurator(\n            device=self.device,\n            gpu_id=self.gpu_id,\n            ps=self.ps,\n            pp_group=self.pp_group,\n            model=self.model,\n            model_config=self.model_config,\n            server_args=self.server_args,\n            kv_cache_dtype=self.kv_cache_dtype,\n            kv_cache_dtype_str=self.kv_cache_dtype_str,\n            model_dtype=self.dtype,\n            page_size=self.page_size,\n            sliding_window_size=self.sliding_window_size,\n            spec_algorithm=self.spec_algorithm,\n            is_draft_worker=self.is_draft_worker,\n            post_capture_kv_active=is_post_capture_kv_active(\n                server_args=self.server_args, is_draft_worker=self.is_draft_worker\n            ),\n            spec_aux_config=self.spec_aux_config,\n            is_hybrid_swa=self.is_hybrid_swa,\n            is_hybrid_swa_compress=self.is_hybrid_swa_compress,\n            use_mla_backend=self.use_mla_backend,\n            layer_info=self.layer_info,\n            forward_stream=self.forward_stream,\n            req_to_token_pool=self.req_to_token_pool,\n            token_to_kv_pool_allocator=self.token_to_kv_pool_allocator,\n            memory_pool_config=self.memory_pool_config,\n            draft_model_idx=self.draft_model_idx,\n        )\n\n    def init_mindspore_runner(self):\n        # Init the mindspore runner\n        # for now, there is only some communication initialization work\n        if get_model().model_impl.lower() == ModelImpl.MINDSPORE and _is_npu:\n            from sglang.srt.model_executor.mindspore_runner import init_ms_distributed\n\n            init_ms_distributed(\n                world_size=self.ps.tp_size * self.ps.pp_size,\n                rank=self.ps.tp_size * self.ps.pp_rank + self.ps.tp_rank,\n                local_rank=self.gpu_id,\n                server_args=self.server_args,\n                port=self.dist_port,\n            )\n\n    def initialize(self):\n        self.init_memory_saver_adapter()\n        self.maybe_init_remote_instance_transfer_engine()\n        self.maybe_init_expert_location_metadata()\n        self.maybe_init_lplb_solvers()\n        self.maybe_init_eplb_manager()\n        self.expert_location_updater = ExpertLocationUpdater()\n        self.maybe_init_elastic_ep()\n        self.init_token_oracle()\n        self.sampler = create_sampler()\n        self.load_model()\n        from marlin_capture import bind_model\n        bind_model(self)\n        prepare_moe_topk(\n            model=self.model,\n            model_config=self.model_config,\n            server_args=self.server_args,\n            moe_ep_size=self.ps.moe_ep_size,\n            moe_ep_rank=self.ps.moe_ep_rank,\n        )\n\n        self.maybe_init_dwdp()\n\n        # Must run before backend/graph init so no draft graph records a\n        # routed-experts capture-write kernel.\n        if self.is_draft_worker:\n            disable_routed_experts_capture_for_draft(self.model)\n        self.maybe_init_expert_backup_client()\n        self.remote_instance_weight_transporter.maybe_register_and_publish_weight_info()\n        self.layer_info: ModelLayerInfo = resolve_layer_indices(\n            model=self.model,\n            model_config=self.model_config,\n            is_draft_worker=self.is_draft_worker,\n            spec_algorithm=self.spec_algorithm,\n        )\n        adjust_hybrid_swa_layer_ids(\n            model_config=self.model_config,\n            start_layer=self.layer_info.start_layer,\n            end_layer=self.layer_info.end_layer,\n            is_hybrid_swa=self.is_hybrid_swa,\n        )\n        self.maybe_apply_post_load_model_transforms()\n        self.maybe_init_lora_manager()\n        self.maybe_enable_batch_invariant_mode()\n        self.configure_kv_cache_dtype()\n\n    def init_memory_saver_adapter(self):\n        self.memory_saver_adapter = TorchMemorySaverAdapter.create(\n            enable=get_exec().features.enable_memory_saver\n        )\n\n    def maybe_init_remote_instance_transfer_engine(self):\n        if remote_instance_transfer_engine_enabled(load_format=self.draft_load_format):\n            self.remote_instance_weight_transporter.init_engine()\n\n    def maybe_init_expert_location_metadata(self):\n        if self.is_draft_worker:\n            return\n        expert_rank = self.ps.moe_ep_rank + (\n            get_parallel().config.ep_join_rank_offset if is_ep_scale_joiner() else 0\n        )\n        set_global_expert_location_metadata(\n            compute_initial_expert_location_metadata(\n                model_config=self.model_config,\n                moe_ep_rank=expert_rank,\n            )\n        )\n        if self.ps.tp_rank == 0 and envs.SGLANG_LOG_EXPERT_LOCATION_METADATA.get():\n            logger.info(\n                "Initial expert_location_metadata:\\n%s",\n                format_expert_location_layout(get_global_expert_location_metadata()),\n            )\n        set_global_expert_distribution_recorder(\n            ExpertDistributionRecorder.init_new(\n                self.server_args,\n                get_global_expert_location_metadata(),\n                rank=expert_rank,\n            )\n        )\n\n    def maybe_init_lplb_solvers(self):\n        if get_exec().moe.ep_dispatch_algorithm == "lp" and not self.is_draft_worker:\n            init_lplb_solvers(model_config=self.model_config)\n\n    def maybe_init_eplb_manager(self):\n        self.eplb_manager = (\n            EPLBManager(\n                model_config=self.model_config,\n                ps=self.ps,\n                get_model=lambda: self.model,\n                get_expert_location_updater=lambda: self.expert_location_updater,\n                get_expert_backup_client=lambda: self.expert_backup_client,\n                get_weight_updater=lambda: self.weight_updater,\n            )\n            if get_exec().moe.enable_eplb and (not self.is_draft_worker)\n            else None\n        )\n\n    def maybe_init_elastic_ep(self):\n        if get_exec().moe.elastic_ep_backend:\n            ElasticEPStateManager.init(self.server_args)\n\n    def init_token_oracle(self):\n        # The oracle sampler is process-wide, so a draft would overwrite the\n        # target\'s with its own vocab -- which a DFlash draft does not have.\n        if self.is_draft_worker:\n            self._token_oracle_manager = None\n            return\n        self._token_oracle_manager = install_token_oracle_from_env(\n            vocab_size=self.model_config.vocab_size,\n        )\n\n    def maybe_init_expert_backup_client(self):\n        self.expert_backup_client = (\n            ExpertBackupClient(\n                server_args=self.server_args,\n                model_config=self.model_config,\n                moe_ep_size=self.ps.moe_ep_size,\n                moe_ep_rank=self.ps.moe_ep_rank,\n                get_model=lambda: self.model,\n            )\n            if (\n                get_exec().moe.enable_elastic_expert_backup\n                and get_exec().moe.elastic_ep_backend is not None\n            )\n            else None\n        )\n\n    def maybe_apply_post_load_model_transforms(self):\n        supports_torch_tp = getattr(self.model, "supports_torch_tp", False)\n        if self.ps.tp_size > 1 and supports_torch_tp:\n            self.apply_torch_tp()\n\n    def maybe_init_lora_manager(self):\n        # Adapters apply to the target model only; the draft runs unadapted.\n        if get_lora().enable_lora and not self.is_draft_worker:\n            self.init_lora_manager()\n\n    def maybe_enable_batch_invariant_mode(self):\n        if get_exec().deterministic.enable_deterministic_inference:\n            from sglang.srt.batch_invariant_ops import enable_batch_invariant_mode\n\n            enable_batch_invariant_mode()\n\n    def get_pp_proxy_topk_size(self) -> Optional[int]:\n        return misc_utils.resolve_pp_proxy_topk_size(\n            model_config=self.model_config,\n            pp_size=self.ps.pp_size,\n            pp_rank=self.ps.pp_rank,\n            start_layer=self.layer_info.start_layer,\n        )\n\n    def get_pp_proxy_residual_num_blocks(self) -> Optional[int]:\n        return misc_utils.resolve_pp_proxy_residual_num_blocks(\n            model_config=self.model_config,\n            pp_size=self.ps.pp_size,\n            pp_rank=self.ps.pp_rank,\n            start_layer=self.layer_info.start_layer,\n        )\n\n    def decode_num_tokens_per_req(\n        self, *, num_draft_tokens: Optional[int] = None\n    ) -> int:\n        """Logits rows per decode batch slot."""\n        if self.spec_algorithm.is_speculative():\n            if self.spec_algorithm.is_dspark() and self.is_draft_worker:\n                from sglang.srt.speculative.dspark_components.dspark_config import (\n                    get_dspark_sample_from_anchor,\n                )\n\n                if not get_dspark_sample_from_anchor(self.model_config.hf_config):\n                    if num_draft_tokens is None:\n                        num_draft_tokens = get_spec().speculative_num_draft_tokens\n                    return int(num_draft_tokens)\n            return resolve_num_tokens_per_req(\n                phase="target_verify",\n                spec_algorithm=self.spec_algorithm,\n                is_draft_worker=self.is_draft_worker,\n                num_draft_tokens=num_draft_tokens,\n            )\n        dllm_config = DllmConfig.from_server_args(self.server_args)\n        return dllm_config.block_size if dllm_config is not None else 1\n\n    def max_decode_logits_rows(self) -> int:\n        """Rows the shared logits buffer needs."""\n        num_tokens_per_req = self.decode_num_tokens_per_req()\n        capture_bs, _ = get_batch_sizes_to_capture(self, num_tokens_per_req)\n        return max(capture_bs) * num_tokens_per_req\n\n    def alloc_memory_pool(self, memory_pool_config: Optional[MemoryPoolConfig] = None):\n        """Allocate KV cache memory pools only (no backends or cuda graphs)."""\n        if memory_pool_config is not None:\n            self.memory_pool_config = memory_pool_config\n\n        self.init_kv_cache_configurator()\n        result = self.kv_cache_configurator.configure(\n            pre_model_load_memory=self.pre_model_load_memory\n        )\n        self.max_total_num_tokens = result.max_total_num_tokens\n        self.max_running_requests = result.max_running_requests\n        self.req_to_token_pool = result.req_to_token_pool\n        self.token_to_kv_pool = result.token_to_kv_pool\n        self.token_to_kv_pool_allocator = result.token_to_kv_pool_allocator\n        self.memory_pool_config = result.memory_pool_config\n        if self.is_hybrid_swa:\n            self.full_max_total_num_tokens = result.full_max_total_num_tokens\n            self.swa_max_total_num_tokens = result.swa_max_total_num_tokens\n        # Keep a reference so the shared byte buffer is not GC\'d.\n        self._unified_memory_pool = result.unified_memory_pool\n\n        self._init_post_memory_pool_components()\n\n    def _init_post_memory_pool_components(self):\n        """Post-pool component wiring, split out of alloc_memory_pool so forks\n        that build bespoke memory pools can reuse it after allocating them."""\n        # Must be called AFTER init_memory_pool so the pool object exists for\n        # canary to monkey-patch, and BEFORE init_decode_cuda_graph so warmup\n        # forwards captured into the graph see the patched pool methods.\n        self.canary_manager = install_canary(\n            server_args=self.server_args,\n            model_runner=self,\n            token_oracle_manager=self._token_oracle_manager,\n        )\n\n        # Init ngram embedding token table\n        self.init_ngram_embedding_manager()\n\n        self.maybe_init_hisparse_coordinator()\n\n        self.init_routed_experts_capturer()\n        self.init_indexer_capturer()\n\n        self.graph_shared_output = None\n\n    def maybe_init_hisparse_coordinator(self):\n        if not self.enable_hisparse:\n            return\n        from sglang.srt.managers.hisparse_coordinator import (\n            HiSparseCoordinator,\n            resolve_shared_index_layers,\n        )\n        from sglang.srt.mem_cache.sparsity import parse_hisparse_config\n\n        hisparse_cfg = parse_hisparse_config(self.server_args)\n        hisparse_top_k = getattr(\n            self.model_config.hf_text_config, "index_topk", hisparse_cfg.top_k\n        )\n        self.hisparse_coordinator = HiSparseCoordinator(\n            req_to_token_pool=self.req_to_token_pool,\n            token_to_kv_pool_allocator=self.token_to_kv_pool_allocator,\n            top_k=hisparse_top_k,\n            device_buffer_size=hisparse_cfg.device_buffer_size,\n            device=self.device,\n            tp_group=(\n                self.attention_tp_group.cpu_group\n                if get_parallel().config.enable_dp_attention\n                else self.tp_group.cpu_group\n            ),\n            host_to_device_ratio=hisparse_cfg.host_to_device_ratio,\n            swap_in_block_size=hisparse_cfg.swap_in_block_size,\n            shared_index_layers=resolve_shared_index_layers(\n                hf_text_config=self.model_config.hf_text_config,\n                pp_size=self.ps.pp_size,\n                is_speculative=self.spec_algorithm.is_speculative(),\n            ),\n        )\n\n    def post_capture_resize_kv_pool(self):\n        resize = compute_post_capture_kv_resize(self)\n        self.max_total_num_tokens = resize.max_total_num_tokens\n        if self.is_hybrid_swa:\n            self.full_max_total_num_tokens = resize.full_max_total_num_tokens\n            self.swa_max_total_num_tokens = resize.swa_max_total_num_tokens\n        if self.memory_pool_config is not None:\n            self.memory_pool_config.max_total_num_tokens = resize.max_total_num_tokens\n            self.memory_pool_config.full_max_total_num_tokens = (\n                resize.full_max_total_num_tokens\n            )\n            self.memory_pool_config.swa_max_total_num_tokens = (\n                resize.swa_max_total_num_tokens\n            )\n        if resize.capped_max_running_requests is not None:\n            self.max_running_requests = resize.capped_max_running_requests\n            if self.memory_pool_config is not None:\n                self.memory_pool_config.max_running_requests = (\n                    resize.capped_max_running_requests\n                )\n\n    def post_capture_elastic_ep_recover(self):\n        join_process_groups()\n\n        global_ep_rank = self.ps.tp_rank + get_parallel().config.ep_join_rank_offset\n        broadcast_global_expert_location_metadata(\n            model_config=self.model_config,\n            moe_ep_rank=global_ep_rank,\n            src_rank=get_healthy_expert_location_src_rank(\n                invoked_in_elastic_ep_rejoin_path=True\n            ),\n        )\n        set_global_expert_distribution_recorder(\n            ExpertDistributionRecorder.init_new(\n                self.server_args,\n                get_global_expert_location_metadata(),\n                rank=global_ep_rank,\n            )\n        )\n\n        ElasticEPStateManager.instance().reset()\n\n    def init_attention_backends(self):\n        """Initialize attention backends only (no cuda graph capture)."""\n        # Must be called BEFORE init_decode_cuda_graph() so CUDA graph capture\n        # runs with aux hidden state capture enabled.\n        configure_aux_hidden_state_capture(\n            model=self.model,\n            eagle_use_aux_hidden_state=self.spec_aux_config.eagle_use_aux_hidden_state,\n            eagle_aux_hidden_state_layer_ids=self.spec_aux_config.eagle_aux_hidden_state_layer_ids,\n            dflash_use_aux_hidden_state=self.spec_aux_config.dflash_use_aux_hidden_state,\n            dflash_target_layer_ids=self.spec_aux_config.dflash_target_layer_ids,\n            is_dspark=self.spec_algorithm.is_dspark(),\n        )\n        # Resolve before building: backends read the pair off the runner while\n        # they construct (the FlashInfer KV-access check).\n        resolved = resolve_attention_backend_strs(model_runner=self)\n        self.prefill_attention_backend_str = resolved.prefill\n        self.decode_attention_backend_str = resolved.decode\n        backends = build_attention_backends(model_runner=self)\n        self.attn_backend = backends.attn_backend\n        self.decode_attn_backend = backends.decode_attn_backend\n        self.decode_attn_backend_group = backends.decode_attn_backend_group\n\n        if get_parallel().dcp_enabled and get_parallel().config.dcp_replicate_q_proj:\n            self._prepare_replicated_q_proj()\n\n    def _prepare_replicated_q_proj(self) -> None:\n        # --dcp-replicate-q-proj: gather each rank\'s attn_tp head-shard of\n        # q_b_proj / w_kc into full-head buffers once here (pre-capture) so the\n        # MLA decode path can skip the per-layer Q all-gather. bf16/fp16 only.\n        from sglang.srt.layers.quantization.unquant import UnquantizedLinearMethod\n        from sglang.srt.models.deepseek_v2 import DeepseekV2AttentionMLA\n\n        dcp_group = get_parallel().dcp_group\n        if dcp_group.world_size <= 1:\n            return\n        n_prepared = 0\n        for m in self.model.modules():\n            if not isinstance(m, DeepseekV2AttentionMLA):\n                continue\n            if m.w_kc is None:\n                continue\n            qp = m.q_b_proj if m.has_q_b_proj else m.q_proj\n            # q-replicate only supports the unquantized bf16/fp16 absorb path;\n            # quantized q-proj (packed weights) and non-16-bit w_kc keep the\n            # per-layer Q all-gather.\n            if (\n                m.w_kc.dtype not in (torch.bfloat16, torch.float16)\n                or not isinstance(qp.quant_method, UnquantizedLinearMethod)\n                or qp.weight.dtype not in (torch.bfloat16, torch.float16)\n            ):\n                logger.warning(\n                    "dcp_replicate_q_proj: skipping quantized q-proj/w_kc "\n                    "(bf16/fp16 only); this layer keeps the Q all-gather."\n                )\n                continue\n            m.w_kc_qrep = dcp_group.all_gather(m.w_kc.contiguous(), dim=0)\n            m.q_b_proj_qrep_weight = dcp_group.all_gather(\n                qp.weight.data.contiguous(), dim=0\n            )\n            n_prepared += 1\n        logger.info(\n            "dcp_replicate_q_proj: prepared full-head Q weights for %d MLA layers",\n            n_prepared,\n        )\n\n    def init_cuda_graphs(self, capture_decode_cuda_graph: bool = True):\n        capture = capture_cuda_graphs(\n            model_runner=self, capture_decode_cuda_graph=capture_decode_cuda_graph\n        )\n        self.eager_runner = capture.eager_runner\n        self.prefill_cuda_graph_runner = capture.prefill.runner\n        self.decode_cuda_graph_runner = capture.decode.runner\n        self.graph_memory_usage = capture.memory_usage\n        self.graph_time_usage = capture.time_usage\n        # gdn_mtp_cache_mode=none: capture the per-bucket SSM-state recovery graphs\n        # now that target_verify capture above allocated the recovery stash at its\n        # final addresses. Self-guards (no-op in full mode / non-recovery paths).\n        self.maybe_capture_gdn_recovery_graphs()\n\n    def maybe_capture_gdn_recovery_graphs(self):\n        """Capture per-bucket FlashInfer SSM-state recovery cuda graphs at warmup.\n\n        Called from init_cuda_graphs after the decode/target_verify graphs are\n        captured, so the per-layer recovery stash is already allocated at its\n        final addresses. HybridLinearAttnBackend pads the serving batch up to a\n        captured bucket and replays on the side stream, and owns the\n        capture-failure fallback to eager recovery -- so nothing is wrapped here.\n        """\n        if self.device != "cuda" or self.is_draft_worker:\n            return\n        if self.server_args.gdn_mtp_cache_mode == "full":\n            return\n\n        from sglang.srt.layers.attention.hybrid_linear_attn_backend import (\n            HybridLinearAttnBackend,\n        )\n\n        if not isinstance(self.attn_backend, HybridLinearAttnBackend):\n            return\n        if self.decode_cuda_graph_runner is None:\n            return\n        self.attn_backend.capture_recovery_graphs(\n            self.decode_cuda_graph_runner.capture_bs\n        )\n\n    def init_routed_experts_capturer(self):\n        if self.is_draft_worker:\n            # Capture is target-only. The draft worker runs in the same process\n            # as its target and inits after it, so installing a capturer here\n            # would overwrite the target\'s process-global one.\n            return\n\n        set_global_experts_capturer(\n            RoutedExpertsCapturer.create(\n                model=self.model,\n                model_config=self.model_config,\n                num_tokens=self.max_token_pool_size + self.page_size,\n                max_running_requests=self.max_running_requests,\n                device=self.device,\n            )\n        )\n\n    def init_indexer_capturer(self):\n        set_global_indexer_capturer(\n            create_indexer_capturer(\n                model_config=self.model_config,\n                num_tokens=self.max_token_pool_size + self.page_size,\n                max_running_requests=self.max_running_requests,\n                device=self.device,\n            )\n        )\n\n    def check_quantized_moe_compatibility(self):\n        check_quantized_moe_compatibility(\n            model_config=self.model_config,\n            tp_size=self.ps.tp_size,\n            moe_ep_size=self.ps.moe_ep_size,\n            moe_dp_size=self.ps.moe_dp_size,\n        )\n\n    def init_torch_distributed(self):\n        result = bootstrap.init_torch_distributed(\n            server_args=self.server_args,\n            model_config=self.model_config,\n            device=self.device,\n            ps=self.ps,\n            dist_port=self.dist_port,\n            is_draft_worker=self.is_draft_worker,\n            local_omp_cpuid=self.local_omp_cpuid if self.device == "cpu" else None,\n        )\n        self.tp_group = result.tp_group\n        self.pp_group = result.pp_group\n        self.attention_tp_group = result.attention_tp_group\n        self.pre_model_load_memory = result.pre_model_load_memory\n\n    def init_shared_mooncake_transfer_engine(self):\n        maybe_init_shared_mooncake_transfer_engine(gpu_id=self.gpu_id)\n\n    def load_model(self):\n        tic_total = time.perf_counter()\n        before_avail_memory = get_available_gpu_memory(self.device, self.gpu_id)\n        logger.info(\n            f"Load weight begin. avail mem={get_available_gpu_memory(self.device, self.gpu_id):.2f} GB"\n        )\n\n        # This can reduce thread conflicts and speed up weight loading.\n        if self.device != "cpu":\n            torch.set_num_threads(1)\n        if self.device == "cuda":\n            maybe_downgrade_dtype_for_legacy_gpu(model_config=self.model_config)\n\n        set_cuda_arch()\n\n        draft_load_format = self.draft_load_format\n        self.load_config = build_load_config(\n            server_args=self.server_args,\n            load_format=draft_load_format,\n            tp_rank=self.ps.tp_rank,\n            remote_instance_weight_transporter_engine=self.remote_instance_weight_transporter.engine,\n            remote_instance_weight_transporter_session_id=self.remote_instance_weight_transporter.session_id,\n            draft_model_idx=self.draft_model_idx,\n            weight_cache_mode=get_model().weight_cache_mode,\n            weight_cache_socket=get_model().weight_cache_socket,\n        )\n\n        # If the weight cache is enabled, override the load format to IPC_CACHE\n        # and derive the per-rank daemon socket. Idempotent across reloads.\n        maybe_enable_ipc_weight_cache(\n            load_config=self.load_config,\n            tp_size=self.ps.tp_size,\n            pp_rank=self.ps.pp_rank,\n            tp_rank=self.ps.tp_rank,\n        )\n        if self.device == "cpu":\n            self.model_config = adjust_config_with_unaligned_cpu_tp(\n                self.model_config, self.load_config, self.ps.tp_size\n            )\n\n        maybe_trigger_remote_instance_nccl_send_group(\n            tp_rank=self.ps.tp_rank,\n            load_format=draft_load_format,\n        )\n\n        with self._load_format_scope(draft_load_format):\n            loaded = load_model_with_memory_saver(\n                server_args=self.server_args,\n                model_config=self.model_config,\n                load_config=self.load_config,\n                device=self.device,\n                gpu_id=self.gpu_id,\n                memory_saver_adapter=self.memory_saver_adapter,\n                is_draft_worker=self.is_draft_worker,\n            )\n        self.loader = loaded.loader\n        self.model = loaded.model\n        self.startup_weight_load = loaded.startup_weight_load\n        if loaded.remote_instance_weight_info is not None:\n            self.remote_instance_weight_transporter.weight_info = (\n                loaded.remote_instance_weight_info\n            )\n\n        if not self.is_draft_worker:\n            get_offloader().post_init()\n\n        self.maybe_precompile_model_kernels_after_loading()\n\n        # Register model for layerwise NVTX profiling if enabled\n        if get_exec().comm.enable_layerwise_nvtx_marker:\n            pyt_hooks = PytHooks()\n            pyt_hooks.register_hooks(self.model, module_prefix="model")\n\n        # Same leaf `configure_kv_cache_dtype` reads: the bag, not the startup\n        # record, so the FP8 gate and the pool cannot disagree after an\n        # override. (The runner\'s own stamp is not set yet -- load_model runs\n        # before configure_kv_cache_dtype.)\n        load_kv_cache_scales(\n            model=self.model,\n            server_args=self.server_args,\n            kv_cache_dtype=get_model().kv_cache_dtype,\n        )\n\n        self.sliding_window_size = resolve_sliding_window_size(\n            self.model, self.model_config\n        )\n\n        self.prefill_aware_swa = (\n            hasattr(self.model, "is_prefill_aware_swa")\n            and self.model.is_prefill_aware_swa()\n        )\n\n        self.dtype = self.model_config.dtype\n\n        after_avail_memory = get_available_gpu_memory(self.device, self.gpu_id)\n        self.weight_load_mem_usage = before_avail_memory - after_avail_memory\n        self.weight_load_time = time.perf_counter() - tic_total\n        # Get quantization config from ModelConfig\n        # This handles both config.json (standard) and hf_quant_config.json (ModelOpt)\n        quant_str = self.model_config.get_quantization_config_log_str()\n\n        if self.startup_weight_load is None:\n            logger.info(\n                f"Load weight end. "\n                f"elapsed={self.weight_load_time:.2f} s, "\n                f"type={type(self.model).__name__}, "\n                f"{quant_str + \', \' if quant_str else \'\'}"\n                f"avail mem={after_avail_memory:.2f} GB, "\n                f"mem usage={self.weight_load_mem_usage:.2f} GB."\n            )\n\n        report_online_quantization(\n            model=self.model,\n        )\n\n        maybe_register_debug_tensor_dump_hook(\n            model=self.model,\n            spec_algorithm=self.spec_algorithm,\n            is_draft_worker=self.is_draft_worker,\n            tp_size=self.ps.tp_size,\n            tp_rank=self.ps.tp_rank,\n            pp_rank=self.ps.pp_rank,\n        )\n\n        if dumper.may_enable:\n            dumper.apply_source_patches()\n            dumper.register_non_intrusive_dumper(self.model)\n\n        # Pre-expand RoPE cache before CUDA Graph capture\n        reserve_rope_cache_for_long_sequences(\n            self.model,\n            self.model_config,\n            logger,\n        )\n\n        if self.startup_weight_load is None:\n            dist_barrier_after_load(\n                elastic_ep_backend=get_exec().moe.elastic_ep_backend,\n                tp_rank=self.ps.tp_rank,\n                is_ep_joiner=self.server_args.is_ep_joiner,\n            )\n\n    def start_startup_weight_load(self) -> None:\n        assert self.startup_weight_load is not None\n        self.startup_weight_load.start_prefetch()\n\n    def finalize_startup_weight_load(self) -> None:\n        """Commit the real weights, then run the post-load barrier.\n\n        The barrier moves here because ``load_model`` returns with sentinel\n        values under overlap, so this is the first point at which "weights are\n        loaded" is true for this rank. It follows the commit and its validation\n        deliberately: a rank that fails to commit must not report readiness. A\n        commit failure is terminal for the process, so peer ranks observe it as\n        a barrier timeout rather than a clean collective abort, which matches\n        the existing startup contract for load failures.\n        """\n        assert self.startup_weight_load is not None\n        self.startup_weight_load.finalize()\n        dist_barrier_after_load(\n            elastic_ep_backend=get_exec().moe.elastic_ep_backend,\n            tp_rank=self.ps.tp_rank,\n            is_ep_joiner=is_ep_joiner(),\n        )\n        self.startup_weight_load = None\n\n    def maybe_precompile_model_kernels_after_loading(self) -> None:\n        maybe_precompile_model_kernels_after_loading(self.model, self.device)\n\n    def maybe_init_dwdp(self):\n        if self.is_draft_worker:\n            return\n        if get_parallel().config.dwdp_size <= 1:\n            return\n        from sglang.srt.layers.moe.dwdp import DwdpManager\n\n        manager = DwdpManager(self.server_args)\n        set_global_dwdp_manager(manager)\n        manager.setup(self.model)\n\n    def init_lora_manager(self):\n        self.lora_manager = LoRAManager(\n            base_model=self.model,\n            base_hf_config=self.model_config.hf_config,\n            max_loras_per_batch=get_lora().max_loras_per_batch,\n            load_config=self.load_config,\n            dtype=self.dtype,\n            server_args=self.server_args,\n            lora_backend=get_lora().lora_backend,\n            tp_size=self.ps.tp_size,\n            tp_rank=self.ps.tp_rank,\n            max_lora_rank=get_lora().max_lora_rank,\n            target_modules=get_lora().lora_target_modules,\n            lora_paths=get_lora().lora_paths,\n        )\n        if not cuda_graph_fully_disabled():\n            init_lora_cuda_graph_moe_buffers(\n                model=self.model,\n                lora_manager=self.lora_manager,\n                dtype=self.dtype,\n            )\n\n    def load_lora_adapter(self, lora_ref: LoRARef):\n        """Load a new lora adapter from disk or huggingface."""\n        return self.lora_manager.load_lora_adapter(lora_ref)\n\n    def load_lora_adapter_from_tensors(\n        self, lora_ref: LoRARef, tensors, config_dict, added_tokens_config=None\n    ):\n        return self.lora_manager.load_lora_adapter_from_tensors(\n            lora_ref, tensors, config_dict, added_tokens_config\n        )\n\n    def unload_lora_adapter(self, lora_ref: LoRARef):\n        """Unload a lora adapter that was previously loaded during initialization or dynamic loading."""\n        return self.lora_manager.unload_lora_adapter(lora_ref)\n\n    @property\n    def effective_max_total_num_tokens(self):\n        """Return the max token pool size considering hybrid swa settings."""\n        if self.is_hybrid_swa:\n            capacity = self.full_max_total_num_tokens or self.swa_max_total_num_tokens\n        else:\n            capacity = self.max_total_num_tokens\n        if (req_to_token_pool := getattr(self, "req_to_token_pool", None)) is not None:\n            return req_to_token_pool.schedulable_token_capacity(capacity)\n        return capacity\n\n    @property\n    def max_token_pool_size(self):\n        """Return the max token pool size considering hybrid swa and hisparse settings."""\n        if self.enable_hisparse:\n            # HiSparse uses the host-backed full pool capacity.\n            size_full = getattr(self.token_to_kv_pool_allocator, "size_full", None)\n            if size_full is not None:\n                return size_full\n        return self.effective_max_total_num_tokens\n\n    def _load_format_scope(self, load_format: Optional[str]):\n        """Make this runner\'s load format the published one while it loads.\n\n        Model code reads it off the bag during construction (Inkling replaces\n        per-element noise in its shared-expert scales under dummy loading), so a\n        draft loading a different way than the target needs its own value live\n        for the load, and the target\'s back afterwards.\n        """\n        if load_format is None:\n            return contextlib.nullcontext()\n        return get_model().override(load_format=load_format)\n\n    def _resolve_draft_load_format(self) -> Optional[str]:\n        """``--speculative-draft-load-format``, for a draft runner only.\n\n        The draft loads its own checkpoint, so its load format is this runner\'s\n        own resolved value; the target keeps ``--load-format``.\n        """\n        if not self.is_draft_worker:\n            return None\n        load_format = get_spec().speculative_draft_load_format\n        if load_format is not None:\n            logger.info(f"Using draft model load_format: \'{load_format}\'")\n        return load_format\n\n    def configure_kv_cache_dtype(self):\n        spec_algorithm = getattr(self, "spec_algorithm", None)\n        resolved_kv_cache_dtype, self.kv_cache_dtype = (\n            kv_cache_dtype.configure_kv_cache_dtype(\n                server_args_kv_cache_dtype=get_model().kv_cache_dtype,\n                model=getattr(self, "model", None),\n                model_dtype=getattr(self, "dtype", torch.bfloat16),\n                is_draft_worker=getattr(self, "is_draft_worker", False),\n                is_dflash=(\n                    spec_algorithm.is_dflash_family()\n                    if spec_algorithm is not None\n                    else False\n                ),\n                speculative_draft_attention_backend=self.draft_attention_backend,\n                speculative_draft_kv_cache_dtype=get_spec().speculative_draft_kv_cache_dtype,\n            )\n        )\n        # This runner\'s OWN resolved dtype string (target or draft). Attention\n        # backends read it directly instead of the process-global get_model()\n        # bag: a draft runner does not publish its args, so the bag would carry\n        # the target\'s dtype and mis-drive the draft\'s FP8 cast/descale paths.\n        self.kv_cache_dtype_str = (\n            resolved_kv_cache_dtype\n            if resolved_kv_cache_dtype is not None\n            else get_model().kv_cache_dtype\n        )\n\n    def _get_attention_backend(self, init_new_workspace: bool = False):\n        return get_attention_backend(\n            model_runner=self, init_new_workspace=init_new_workspace\n        )\n\n    def _decode_cuda_graph_runner_cls(self):\n        """Decode CUDA-graph runner class to construct.\n\n        Subclasses can override this to install specialized decode graph runners.\n        """\n        from sglang.srt.model_executor.runner.decode_cuda_graph_runner import (\n            DecodeCudaGraphRunner,\n        )\n\n        return DecodeCudaGraphRunner\n\n    def init_decode_cuda_graph(self):\n        self.decode_cuda_graph_runner = None\n        capture = capture_decode_graph(model_runner=self)\n        self.decode_cuda_graph_runner = capture.runner\n        self.graph_memory_usage = replace_graph_memory_usage(\n            self.graph_memory_usage,\n            capture.memory_usage,\n            phases=("decode", "target_verify", "draft_decode"),\n        )\n        self.graph_time_usage = replace_graph_time_usage(\n            self.graph_time_usage,\n            capture.time_usage,\n            phases=("decode", "target_verify", "draft_decode"),\n        )\n\n    def init_prefill_cuda_graph(self, force_for_draft_worker: bool = False):\n        self.prefill_cuda_graph_runner = None\n        capture = capture_prefill_graph(\n            model_runner=self,\n            eager_runner=self.eager_runner,\n            force_for_draft_worker=force_for_draft_worker,\n        )\n        self.prefill_cuda_graph_runner = capture.runner\n        self.graph_memory_usage = replace_graph_memory_usage(\n            self.graph_memory_usage,\n            capture.memory_usage,\n            phases=("prefill", "draft_prefill"),\n        )\n        self.graph_time_usage = replace_graph_time_usage(\n            self.graph_time_usage,\n            capture.time_usage,\n            phases=("prefill", "draft_prefill"),\n        )\n\n    def init_threads_binding(self):\n        # With --enable-dp-attention, dp partitions the existing TP group\n        # rather than spawning additional processes, so dp_size must not be\n        # multiplied into the process count here (unlike regular DP, where\n        # dp_size * tp_size * pp_size is the true worker count).\n        dp_size = 1 if get_parallel().config.enable_dp_attention else self.ps.dp_size\n        self.local_omp_cpuid = numa_utils.init_threads_binding(\n            numa_index=self.gpu_id,\n            world_size=dp_size * self.ps.tp_size * self.ps.pp_size,\n        )\n\n    def apply_torch_tp(self):\n        model_parallel.apply_torch_tp(\n            model=self.model, device=self.device, tp_size=self.ps.tp_size\n        )\n\n    def update_decode_attn_backend(self, stream_idx: int):\n        self.decode_attn_backend = self.decode_attn_backend_group[stream_idx]\n\n    def prepare_dummy_forward_batch(self, forward_batch: ForwardBatch) -> ForwardBatch:\n        """Customize a runner-created dummy batch before attention metadata initialization."""\n        return forward_batch\n\n    def _prepare_eager_forward_batch(self, forward_batch: ForwardBatch) -> None:\n        """Pad / normalize a batch for the eager (non-cuda-graph) forward.\n\n        Runs the DP/MLP-sync padding, the attn-tp num_token_non_padded\n        normalization, and the hisparse-coordinator refresh that the eager\n        forward path needs — the cuda-graph path does the equivalent inside the\n        runner\'s capture/replay, so this is skipped there.\n        """\n        # For MLP sync\n        if forward_batch.global_num_tokens_cpu is not None:\n            forward_batch.prepare_mlp_sync_batch(self)\n        else:\n            forward_batch.prepare_attn_tp_scatter_input(self)\n\n        # Normalize num_token_non_padded to be local to this attention TP rank if needed.\n        # The skip is scoped to DSACPLayerCommunicator-style CP (DSA, MLA): those\n        # flavors already feed a zigzag-split rank-local layout whose token count\n        # should not be further divided by attn_tp_size. MHA-arch prefill CP\n        # (Qwen3/Qwen2 MoE) keeps the attn_tp-replicated layout and wants the\n        # adjustment to run — see docs/design/prefill-cp-mla.md §Phase 5.\n        if (\n            forward_batch.num_token_non_padded is not None\n            and forward_batch.global_num_tokens_gpu is not None\n            and require_gathered_buffer()\n            and not is_dsa_enable_prefill_cp()\n            and not is_mla_prefill_cp_enabled()\n        ):\n            forward_batch.adjust_num_token_non_padded_for_attn_tp()\n\n        # Hisparse coordinator — backends now read it from self.model_runner.\n        if self.hisparse_coordinator is not None:\n            self.hisparse_coordinator.num_real_reqs.fill_(forward_batch.batch_size)\n\n    def _pp_kwargs(self, pp_proxy_tensors) -> dict:\n        """Build the pp_proxy_tensors forward kwarg, in one place.\n\n        Pipeline-parallel proxy tensors are threaded into model.forward only\n        when the model accepts them (``support_pp``).\n        """\n        return {"pp_proxy_tensors": pp_proxy_tensors} if self.support_pp else {}\n\n    def _extend_forward_kwargs(\n        self, forward_batch: ForwardBatch, pp_proxy_tensors\n    ) -> dict:\n        """Build the extend/prefill model.forward kwargs (pp_proxy_tensors +\n        input_embeds / replace_embeds overrides + get_embedding), shared by the\n        prefill cuda-graph path and the EagerRunner\'s eager extend path."""\n        kwargs = self._pp_kwargs(pp_proxy_tensors)\n        if forward_batch.input_embeds is not None:\n            kwargs["input_embeds"] = forward_batch.input_embeds.bfloat16()\n        if (\n            forward_batch.replace_embeds is not None\n            and forward_batch.replace_positions is not None\n        ):\n            # Token embedding overrides: get base embeddings, scatter replacements\n            if "input_embeds" not in kwargs:\n                embed_layer = self.model.get_input_embeddings()\n                kwargs["input_embeds"] = embed_layer(forward_batch.input_ids)\n            kwargs["input_embeds"][forward_batch.replace_positions] = (\n                forward_batch.replace_embeds.to(kwargs["input_embeds"].dtype)\n            )\n        if not self.is_generation:\n            kwargs["get_embedding"] = True\n        return kwargs\n\n    def forward_split_prefill(\n        self,\n        forward_batch: ForwardBatch,\n        reinit_attn_backend: bool = False,\n        forward_count: int = 1,\n    ) -> LogitsProcessorOutput:\n        if forward_batch.split_index == 0 or reinit_attn_backend:\n            self.attn_backend.init_forward_metadata(forward_batch)\n        next_split_index = min(\n            forward_batch.split_index + forward_count,\n            self.model_config.num_hidden_layers,\n        )\n        with device_timer_ctx(self.device_timer, "split_prefill"):\n            ret = self.model.forward_split_prefill(\n                forward_batch.input_ids,\n                forward_batch.positions,\n                forward_batch,\n                (forward_batch.split_index, next_split_index),\n            )\n        forward_batch.split_index = next_split_index\n        return ret\n\n    def forward(\n        self,\n        forward_batch: ForwardBatch,\n        skip_attn_backend_init: Optional[bool] = None,  # deprecated\n        pp_proxy_tensors: Optional[PPProxyTensors] = None,\n        reinit_attn_backend: bool = False,\n        split_forward_count: int = 1,\n    ) -> ModelRunnerOutput:\n        # Deprecated kwarg: pre-planners mark the batch themselves now.\n        forward_batch.apply_deprecated_skip_attn_backend_init(skip_attn_backend_init)\n\n        self.forward_pass_id += 1\n\n        # Try msprob debugger\n        if self.msprobe_debugger is not None:\n            rank_id = (\n                self.gpu_id\n                if self.ps.attn_dp_size is not None and self.ps.attn_dp_size > 1\n                else None\n            )\n            self.msprobe_debugger.start(model=self.model, rank_id=rank_id)\n\n        # Step span\n        step_span_ctx = profile_range(build_step_span_name(forward_batch))\n\n        canary_ctx = (\n            context_tuple(\n                c.with_ops_outside_graph(\n                    single_forward_indices=[0],\n                    maybe_inaccurate_forward_batch=forward_batch,\n                ),\n                c.with_active_single_forward_manager(0),\n            )\n            if not self.is_draft_worker and ((c := self.canary_manager) is not None)\n            else contextlib.nullcontext()\n        )\n\n        with (\n            canary_ctx,\n            step_span_ctx,\n            get_global_expert_distribution_recorder().with_forward_pass(\n                self.forward_pass_id,\n                forward_batch,\n            ) as recorder_outputs,\n        ):\n            output = self._forward_raw(\n                forward_batch,\n                pp_proxy_tensors,\n                reinit_attn_backend,\n                split_forward_count,\n            )\n            if self.enable_elastic_ep:\n                output = self._maybe_rebalance_after_rank_fault(\n                    output,\n                    forward_batch,\n                    pp_proxy_tensors,\n                    reinit_attn_backend,\n                    split_forward_count,\n                )\n        output.expert_distribution_metrics = recorder_outputs.get("metrics")\n\n        no_copy_to_cpu = not get_schedule().disable_overlap_schedule\n        if (\n            not self.is_draft_worker\n            and (experts_capturer := get_global_experts_capturer()) is not None\n        ):\n            output.routed_experts_output = experts_capturer.on_forward_end(\n                forward_batch=forward_batch,\n                can_run_graph=output.can_run_graph,\n                cuda_graph_batch=getattr(self.decode_cuda_graph_runner, "bs", None),\n                no_copy_to_cpu=no_copy_to_cpu,\n            )\n\n        if (indexer_capturer := get_global_indexer_capturer()) is not None:\n            output.indexer_topk_output = indexer_capturer.on_forward_end(\n                forward_batch=forward_batch,\n                can_run_graph=output.can_run_graph,\n                cuda_graph_batch=getattr(self.decode_cuda_graph_runner, "bs", None),\n                no_copy_to_cpu=no_copy_to_cpu,\n            )\n\n        if self.eplb_manager is not None:\n            self.eplb_manager.on_forward_pass_end()\n\n        if dumper.may_enable:\n            dumper.step()\n\n        if self.msprobe_debugger is not None:\n            self.msprobe_debugger.stop()\n            self.msprobe_debugger.step()\n\n        if get_exec().moe.elastic_ep_backend is not None:\n            self.maybe_join_ep_ranks()\n\n        return output\n\n    def _maybe_execute_deferred_mamba_cow_and_clear(\n        self, forward_batch: ForwardBatch\n    ) -> None:\n        """Run deferred clear/COW on the forward stream, before the mamba layers\n        read the pool, so the copies don\'t race the scheduler copy stream.\n\n        No-op unless this is an extend forward on a mamba model\'s target worker;\n        COW/clear only happen at prefix match on extend.\n        """\n        pool = self.req_to_token_pool\n        if (\n            not isinstance(pool, HybridReqToTokenPool)\n            or self.is_draft_worker\n            or not forward_batch.forward_mode.is_extend()\n            or forward_batch.forward_mode.is_target_verify()\n            or forward_batch.forward_mode.is_draft_extend_v2()\n        ):\n            return\n        if (\n            forward_batch.mamba_clear_indices is not None\n            and len(forward_batch.mamba_clear_indices) > 0\n        ):\n            # mamba_pool is a pure PHYSICAL store; translate before zeroing or\n            # clear_slots zeroes the wrong physical slots.\n            pool.mamba_pool.clear_slots(\n                pool.translate_mamba_indices(forward_batch.mamba_clear_indices)\n            )\n        if (\n            forward_batch.mamba_cow_src_indices is not None\n            and len(forward_batch.mamba_cow_src_indices) > 0\n        ):\n            # A HiCache load publishes the radix slot layerwise on its H->D\n            # stream.  Deferred COW reads the complete source slot at once, so\n            # it must wait for the final layer rather than racing ahead of the\n            # per-layer attention waits and copying an uninitialized source\n            # over the independently restored request slot.\n            pool.wait_for_hicache_load_complete()\n            if pool.mamba_ckpt_pool is not None:\n                # int8 checkpoints: dequantize src int8 ckpt slot into the active bf16 dst.\n                pool.mamba_ckpt_pool.load_to_active(\n                    pool.mamba_pool,\n                    forward_batch.mamba_cow_src_indices,\n                    forward_batch.mamba_cow_dst_indices,\n                )\n            else:\n                # mamba_pool is a pure PHYSICAL store; translate both COW slot ids.\n                pool.mamba_pool.copy_from(\n                    pool.translate_mamba_indices(forward_batch.mamba_cow_src_indices),\n                    pool.translate_mamba_indices(forward_batch.mamba_cow_dst_indices),\n                )\n        forward_batch.mamba_clear_indices = None\n        forward_batch.mamba_cow_src_indices = None\n        forward_batch.mamba_cow_dst_indices = None\n\n    def _forward_raw(\n        self,\n        forward_batch: ForwardBatch,\n        pp_proxy_tensors: Optional[PPProxyTensors],\n        reinit_attn_backend: bool = False,\n        split_forward_count: int = 1,\n    ) -> ModelRunnerOutput:\n        if has_forward_context():\n            ctx_mgr = contextlib.nullcontext()\n        else:\n            ctx_mgr = forward_context(ForwardContext(attn_backend=self.attn_backend))\n        with ctx_mgr:\n            mode_check = (\n                forward_batch.forward_mode.is_cpu_graph\n                if self.device == "cpu"\n                else forward_batch.forward_mode.is_cuda_graph\n            )\n            can_run_graph = bool(\n                mode_check()\n                and self.decode_cuda_graph_runner\n                and self.decode_cuda_graph_runner.can_run_graph(forward_batch)\n            )\n\n            if (\n                forward_batch.forward_mode.is_decode()\n                and self.hisparse_coordinator is not None\n            ):\n                forward_batch.hisparse_coordinator = self.hisparse_coordinator\n                self.hisparse_coordinator.wait_for_pending_backup()\n                self.hisparse_coordinator.num_real_reqs.fill_(forward_batch.batch_size)\n\n            # Replay cuda graph if applicable\n            if can_run_graph:\n                ret = self.decode_cuda_graph_runner.execute(\n                    forward_batch,\n                    pp_proxy_tensors=pp_proxy_tensors,\n                )\n                return ModelRunnerOutput(logits_output=ret, can_run_graph=can_run_graph)\n\n            # DP / MLP-sync padding + attn-tp normalization. Only the decode\n            # cuda-graph path above pre-pads its static buffers and returns\n            # early; split prefill, the prefill cuda graph, and the eager\n            # forward all run the live batch and need this first — it sets\n            # global_dp_buffer_len / padded token counts that graph eligibility\n            # and the collectives depend on.\n            self._prepare_eager_forward_batch(forward_batch)\n\n            # Deferred mamba COW/clear on the forward stream, before the extend\n            # dispatch below reads the pool.\n            self._maybe_execute_deferred_mamba_cow_and_clear(forward_batch)\n\n            dwdp_mgr = get_global_dwdp_manager()\n            if dwdp_mgr is not None:\n                dwdp_mgr.prefetch_first_layers()\n\n            if forward_batch.forward_mode.is_split_prefill():\n                # Layer-split mode; stays on ModelRunner, not the eager runner.\n                ret = self.forward_split_prefill(\n                    forward_batch,\n                    reinit_attn_backend=reinit_attn_backend,\n                    forward_count=split_forward_count,\n                )\n            elif (\n                forward_batch.forward_mode.is_extend(include_draft_extend_v2=True)\n                and not isinstance(self.prefill_cuda_graph_runner, EagerRunner)\n                and self.prefill_cuda_graph_runner is not None\n                and self.prefill_cuda_graph_runner.can_run_graph(forward_batch)\n                and _prefill_cuda_graph_allows_context_parallel(\n                    self.prefill_cuda_graph_runner, forward_batch\n                )\n            ):\n                # Prefill cuda graph (piecewise).\n                kwargs = self._extend_forward_kwargs(forward_batch, pp_proxy_tensors)\n                category = (\n                    "target_verify"\n                    if forward_batch.forward_mode.is_target_verify()\n                    else "extend"\n                )\n                # TODO: the timing here is too broad -- it also includes\n                # load_batch time. Move it into the prefill cuda graph runner\n                # to capture only the model.forward part.\n                with device_timer_ctx(self.device_timer, category):\n                    ret = self.prefill_cuda_graph_runner.execute(\n                        forward_batch, **kwargs\n                    )\n                can_run_graph = True\n            else:\n                # Eager: decode / extend / idle dispatched inside the runner.\n                ret = self.eager_runner.execute(\n                    forward_batch, pp_proxy_tensors=pp_proxy_tensors\n                )\n\n            if (\n                forward_batch.global_num_tokens_cpu is not None\n                and self.pp_group.is_last_rank\n            ):\n                forward_batch.post_forward_mlp_sync_batch(ret)\n\n            return ModelRunnerOutput(logits_output=ret, can_run_graph=can_run_graph)\n\n    def _preprocess_logits(\n        self,\n        logits_output: LogitsProcessorOutput,\n        sampling_info: SamplingBatchInfo,\n        observer: Optional[SamplingObserver] = None,\n    ):\n        # NOTE: In overlap mode, the function update_regex_vocab_mask (in sample)\n        #       was executed after we processed last batch\'s results.\n\n        # Calculate logits bias and apply it to next_token_logits.\n        sampling_info.update_regex_vocab_mask()\n        observer_state = None\n        if observer is not None:\n            observer_state = sampling_info.apply_logits_bias_with_observer(\n                logits_output.next_token_logits,\n                observer=observer,\n            )\n        else:\n            sampling_info.apply_logits_bias(logits_output.next_token_logits)\n\n        # Release the vocab_mask GPU tensor immediately after it has been applied\n        # to the logits. In overlap scheduling, the sampling_info (and its\n        # grammar_mask) can be kept alive by the delay_sample_func closure and\n        # batch_record_buf until the next iteration, causing a steady VRAM leak\n        # when structured output (grammar) is used.\n        sampling_info.grammar_mask = None\n        return observer_state\n\n    def sample(\n        self,\n        logits_output: LogitsProcessorOutput,\n        forward_batch: ForwardBatch,\n    ) -> torch.Tensor:\n        """Sample and compute logprobs and update logits_output.\n\n        Args:\n            logits_output: The logits output from the model forward\n            forward_batch: The forward batch that generates logits_output\n\n        Returns:\n            A list of next_token_ids\n        """\n        # LogitsProcessorOutput is normally invocation-scoped, but CUDA graph\n        # runners may reuse backing objects. Never leak an auxiliary result from\n        # a previous replay into a request with no observer state.\n        logits_output.auxiliary_device_output = None\n        observer = self.sampling_observer\n        # Preserve two-argument overrides when observation is inactive.\n        if observer is not None and observer.is_active(forward_batch.sampling_info):\n            observer_state = self._preprocess_logits(\n                logits_output,\n                forward_batch.sampling_info,\n                observer=observer,\n            )\n        else:\n            observer_state = self._preprocess_logits(\n                logits_output, forward_batch.sampling_info\n            )\n\n        # Sample the next tokens\n        next_token_ids = self.sampler(\n            logits_output,\n            forward_batch.sampling_info,\n            forward_batch.return_logprob,\n            forward_batch.top_logprobs_nums,\n            forward_batch.token_ids_logprobs,\n            # For prefill, we only use the position of the last token.\n            (\n                forward_batch.positions\n                if forward_batch.forward_mode.is_decode()\n                else forward_batch.seq_lens - 1\n            ),\n        )\n        if observer_state is not None:\n            logits_output.auxiliary_device_output = observer.after_sample(\n                observer_state,\n                next_token_ids,\n            )\n        self.ngram_embedding_manager.update_after_decode(\n            next_token_ids=next_token_ids,\n            forward_batch=forward_batch,\n        )\n        return next_token_ids\n\n    def compute_logprobs_only(\n        self,\n        logits_output: LogitsProcessorOutput,\n        forward_batch: ForwardBatch,\n    ) -> None:\n        """\n        Compute token_ids_logprobs without performing sampling.\n\n        Optimized path for prefill-only requests that need token_ids_logprobs but don\'t\n        require next token generation. Skips expensive sampling operations\n        while still providing requested probability information.\n\n        Args:\n            logits_output: The logits output from the model forward\n            forward_batch: The forward batch that generates logits_output\n        """\n        logits_output.auxiliary_device_output = None\n        if not forward_batch.token_ids_logprobs:\n            return\n\n        self._preprocess_logits(logits_output, forward_batch.sampling_info)\n\n        # Delegate to sampler for logprob-only computation\n        # This populates logits_output with requested token probabilities\n        self.sampler.compute_logprobs_only(\n            logits_output,\n            forward_batch.sampling_info,\n            forward_batch.top_logprobs_nums,\n            forward_batch.token_ids_logprobs,\n        )\n\n    def check_weights(self, action: str, allow_quant_error: bool = False):\n        return self._weight_checker.handle(\n            action=action, allow_quant_error=allow_quant_error\n        )\n\n    def _expand_eplb_metadata_for_scale(\n        self,\n        from_ep_size: int,\n        effective_size: int,\n    ) -> None:\n        metadata = get_global_expert_location_metadata()\n        if metadata is None:\n            return\n        old_num_physical = metadata.num_physical_experts\n        num_local = old_num_physical // from_ep_size\n        added = num_local * effective_size - old_num_physical\n        if added <= 0:\n            return\n\n        initial_ep_size = get_parallel().config.elastic_ep_initial_size\n        assert initial_ep_size is not None\n        get_context().override("elastic_ep.scale", ep_size=effective_size)\n\n        expanded_p2l = append_trivial_expert_slots(\n            metadata.physical_to_logical_map,\n            added,\n            metadata.num_logical_experts,\n            start=old_num_physical - num_local * initial_ep_size,\n        )\n        new_metadata = ExpertLocationMetadata.init_by_mapping(\n            self.model_config,\n            physical_to_logical_map=expanded_p2l,\n            moe_ep_rank=self._elastic_global_rank(),\n        )\n        set_global_expert_location_metadata(new_metadata, allow_overwrite=True)\n\n    def _elastic_global_rank(self) -> int:\n        return self.ps.tp_rank + get_parallel().config.ep_join_rank_offset\n\n    def _rearm_eplb_after_elastic_scale(self) -> None:\n        if self.eplb_manager is None:\n            return\n        recorder = get_global_expert_distribution_recorder()\n        if not recorder.recording:\n            recorder.start_record()\n        self.eplb_manager.enable_rebalance()\n\n    def _reset_eplb_after_elastic_scale_failure(self) -> None:\n        if self.eplb_manager is None:\n            return\n        set_global_expert_distribution_recorder(\n            ExpertDistributionRecorder.init_new(\n                self.server_args,\n                get_global_expert_location_metadata(),\n                rank=self._elastic_global_rank(),\n            )\n        )\n        self._rearm_eplb_after_elastic_scale()\n\n    def _report_elastic_scale_failure(self, error: str, effective_size: int) -> None:\n        if self.ps.tp_rank != 0 or is_ep_scale_joiner():\n            return\n        from sglang.srt.managers.io_struct import ElasticScaleUpdateReq\n\n        self._pending_elastic_scale_update = ElasticScaleUpdateReq(\n            success=False,\n            effective_ep_size=effective_size,\n            error=error,\n        )\n\n    def _elastic_scale_ready_barrier(self, target_size: int, log_tag: str) -> None:\n        if self.ps.tp_rank == 0:\n            logger.debug(\n                "[Elastic EP][scale] %s entering post-scale WORLD barrier "\n                "(target_ep_size=%d)",\n                log_tag,\n                target_size,\n            )\n        dist.barrier(group=dist.group.WORLD)\n        if self.ps.tp_rank == 0:\n            logger.debug(\n                "[Elastic EP][scale] %s passed post-scale WORLD barrier "\n                "(target_ep_size=%d)",\n                log_tag,\n                target_size,\n            )\n\n    def _finalize_scale_up(\n        self,\n        ranks_to_join: list[int],\n        target_size: int,\n        effective_size: int,\n    ) -> None:\n        self.forward_pass_id = 0\n        ElasticEPStateManager.mark_configuring_data_plane()\n\n        state = ElasticEPStateManager.instance()\n        for rank in ranks_to_join:\n            state.active_ranks[rank] = 1\n        state.snapshot_active_to_last()\n        state.sync_active_to_cpu()\n        if self.eplb_manager is not None:\n            self.eplb_manager.reset_generator()\n\n        self._expand_eplb_metadata_for_scale(\n            from_ep_size=effective_size,\n            effective_size=target_size,\n        )\n        broadcast_global_expert_location_metadata(\n            model_config=self.model_config,\n            moe_ep_rank=self._elastic_global_rank(),\n            src_rank=0,\n        )\n\n        ElasticEPStateManager.on_scale(effective_size, target_size)\n        set_global_expert_distribution_recorder(\n            ExpertDistributionRecorder.init_new(\n                self.server_args,\n                get_global_expert_location_metadata(),\n                rank=self._elastic_global_rank(),\n            )\n        )\n\n        if self.eplb_manager is not None:\n            self.eplb_manager.disable_rebalance(\n                "EPLB rebalance is disabled while elastic EP scale-up "\n                "is being finalized"\n            )\n\n        from sglang.srt.layers.dp_attention import update_dp_attention_post_scale\n\n        update_dp_attention_post_scale(\n            new_dp_size=target_size,\n            new_dp_rank=self._elastic_global_rank(),\n        )\n        get_context().override("elastic_ep.scale", dp_size=target_size)\n\n        ElasticEPStateManager.mark_syncing_new_world()\n        self._elastic_scale_ready_barrier(\n            target_size=target_size,\n            log_tag="JOINER" if is_ep_scale_joiner() else "PRIMARY",\n        )\n        ElasticEPStateManager.commit_scale()\n        self._rearm_eplb_after_elastic_scale()\n\n        if self.ps.tp_rank == 0 and not is_ep_scale_joiner():\n            from sglang.srt.managers.io_struct import ElasticScaleUpdateReq\n\n            self._pending_elastic_scale_update = ElasticScaleUpdateReq(\n                success=True,\n                effective_ep_size=target_size,\n                slot_offset=effective_size,\n                slot_count=target_size - effective_size,\n            )\n            logger.info(\n                "[Elastic EP] Scale completed: old_ep_size=%d "\n                "new_ep_size=%d joined_ranks=%s",\n                effective_size,\n                target_size,\n                ranks_to_join,\n            )\n\n    def maybe_join_ep_ranks(self) -> None:\n        if not ElasticEPStateManager.is_scaling():\n            return\n\n        state = ElasticEPStateManager.instance()\n        effective_size = ElasticEPStateManager.get_effective_ep_size()\n        pending_size = ElasticEPStateManager.get_pending_ep_size()\n\n        if pending_size is None:\n            if state is not None and state.has_scaled:\n                error = (\n                    "Elastic EP rank recovery is unsupported after runtime scale-up. "\n                    "Restart the expanded deployment."\n                )\n                ElasticEPStateManager.fail_recovery(error)\n                self._report_elastic_scale_failure(error, effective_size)\n                if self.ps.tp_rank == 0 and not is_ep_scale_joiner():\n                    logger.error("[Elastic EP] %s", error)\n                return\n\n            recovered = maybe_recover_ep_ranks(\n                tp_group=self.tp_group,\n                eplb_manager=self.eplb_manager,\n                model_config=self.model_config,\n                moe_ep_rank=self._elastic_global_rank(),\n            )\n            if recovered:\n                self.forward_pass_id = 0\n            return\n\n        local_timeout = (\n            state.pending_since is not None\n            and time.monotonic() - state.pending_since\n            > get_exec().moe.elastic_ep_scale_timeout\n        )\n        timeout = state.active_ranks.new_tensor(int(local_timeout))\n        dist.all_reduce(timeout, op=dist.ReduceOp.MAX, group=dist.group.WORLD)\n        if timeout.item():\n            error = f"Timed out waiting for ranks to join target EP size {pending_size}"\n            ElasticEPStateManager.fail_scale(error)\n            self._reset_eplb_after_elastic_scale_failure()\n            self._report_elastic_scale_failure(error, effective_size)\n            if self.ps.tp_rank == 0 and not is_ep_scale_joiner():\n                logger.error("[Elastic EP] %s", error)\n            return\n\n        if state.scale_phase == "waiting_for_cohort":\n            cohort_target = get_scale_cohort_target(effective_size)\n            if cohort_target is None:\n                return\n            if cohort_target != pending_size:\n                error = (\n                    f"Requested target EP size {pending_size} does not match "\n                    f"joining cohort target {cohort_target}"\n                )\n                ElasticEPStateManager.fail_scale(error)\n                self._reset_eplb_after_elastic_scale_failure()\n                self._report_elastic_scale_failure(error, effective_size)\n                if self.ps.tp_rank == 0 and not is_ep_scale_joiner():\n                    logger.error("[Elastic EP] %s", error)\n                return\n            if not ElasticEPStateManager.begin_scale():\n                return\n\n        ranks_to_join = list(range(effective_size, pending_size))\n        if not ranks_to_join:\n            return\n\n        current_platform.synchronize()\n        ElasticEPStateManager.mark_joining()\n        if try_admit_scale_ranks(ranks_to_join):\n            self._finalize_scale_up(\n                ranks_to_join=ranks_to_join,\n                target_size=pending_size,\n                effective_size=effective_size,\n            )\n\n    def _maybe_rebalance_after_rank_fault(\n        self,\n        output: ModelRunnerOutput,\n        forward_batch: ForwardBatch,\n        pp_proxy_tensors: Optional[PPProxyTensors],\n        reinit_attn_backend: bool,\n        split_forward_count: int,\n    ) -> ModelRunnerOutput:\n        if maybe_rebalance_after_rank_fault(eplb_manager=self.eplb_manager):\n            output = self._forward_raw(\n                forward_batch,\n                pp_proxy_tensors,\n                reinit_attn_backend,\n                split_forward_count,\n            )\n        return output\n\n    def update_model_fields(\n        self,\n        new_model: torch.nn.Module,\n        *,\n        model_path: str,\n        load_format: str,\n        load_config: LoadConfig,\n    ) -> None:\n        self.model = new_model\n        # The record says what model this PROCESS serves; a draft\'s weight\n        # update is not that (its own state is on the runner).\n        if not self.is_draft_worker:\n            get_context().override(\n                "model_runner.update_model_fields",\n                model_path=model_path,\n                load_format=load_format,\n            )\n        self.load_config = load_config\n', 'sglang/srt/model_executor/runner/decode_cuda_graph_runner.py': '# Copyright 2023-2026 SGLang Team\n# Licensed under the Apache License, Version 2.0 (the "License");\n# you may not use this file except in compliance with the License.\n# You may obtain a copy of the License at\n#\n#     http://www.apache.org/licenses/LICENSE-2.0\n#\n# Unless required by applicable law or agreed to in writing, software\n# distributed under the License is distributed on an "AS IS" BASIS,\n# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n# See the License for the specific language governing permissions and\n# limitations under the License.\n# ==============================================================================\n"""DecodeCudaGraphRunner — runs DECODE / TARGET_VERIFY / DLLM_EXTEND under\na pluggable backend.\n\nBackend selection comes from cuda_graph_config.decode:\n  - "full"      — default, FullCudaGraphBackend: one\n                      torch.cuda.CUDAGraph per shape.\n  - "breakable" — experimental, BreakableCudaGraphBackend:\n                      segmented capture (no torch.compile).\n  - "tc_piecewise"     — not implemented for decode; logs a one-shot warning\n                      and falls back to "full".\n"""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport inspect\nimport logging\nimport os\nfrom types import SimpleNamespace\nfrom typing import TYPE_CHECKING, Callable, Optional, Union\n\nimport torch\nimport tqdm\nfrom torch.profiler import ProfilerActivity, profile\n\nfrom sglang.srt.compilation import torch_compile_decoration\nfrom sglang.srt.compilation.torch_compile_decoration import set_torch_compile_config\nfrom sglang.srt.distributed.parallel_state import (\n    graph_capture,\n    set_pdmux_status,\n)\nfrom sglang.srt.dllm.config import DllmConfig\nfrom sglang.srt.environ import envs\nfrom sglang.srt.layers.attention.base_attn_backend import (\n    AttentionBackend,\n    SharedReadEnds,\n)\nfrom sglang.srt.layers.attention.dsa.utils import is_dsa_enable_prefill_cp\nfrom sglang.srt.layers.dp_attention import (\n    DpPaddingMode,\n    set_dp_buffer_len,\n    set_is_extend_in_batch,\n)\nfrom sglang.srt.layers.logits_processor import LogitsProcessorOutput\nfrom sglang.srt.layers.utils.cp_utils import is_mla_prefill_cp_enabled\nfrom sglang.srt.model_executor.cuda_graph_buffer_registry import (\n    CudaGraphBufferRegistry,\n    build_decode_registry,\n)\nfrom sglang.srt.model_executor.forward_batch_info import (\n    CaptureHiddenMode,\n    ForwardBatch,\n    ForwardMode,\n    PPProxyTensors,\n    compute_local_num_token_non_padded,\n    enable_num_token_non_padded,\n    get_required_capture_hidden_mode,\n)\nfrom sglang.srt.model_executor.forward_context import ForwardContext, forward_context\nfrom sglang.srt.model_executor.runner.base_cuda_graph_runner import (\n    BaseCudaGraphRunner,\n    freeze_gc,\n    get_batch_sizes_to_capture,\n)\nfrom sglang.srt.model_executor.runner.flashinfer_autotune import (\n    maybe_flashinfer_autotune_speculative_draft,\n)\nfrom sglang.srt.model_executor.runner.shape_key import ShapeKey\nfrom sglang.srt.model_executor.runner_backend.breakable_cuda_graph_backend import (\n    BreakableCudaGraphBackend,\n)\nfrom sglang.srt.model_executor.runner_backend.utils import resolve_decode_backend\nfrom sglang.srt.model_executor.runner_backend_utils import (\n    CUDA_GRAPH_CAPTURE_FAILED_MSG,\n)\nfrom sglang.srt.model_executor.runner_utils.buffers import (\n    DecodeInputBuffers,\n)\nfrom sglang.srt.model_executor.runner_utils.capture_mode import (\n    _set_capture_dsa_variant,\n    _set_capture_lora_variant,\n    model_capture_mode,\n)\nfrom sglang.srt.model_executor.runner_utils.deepep_adapter import (\n    DeepEPCudaGraphRunnerAdapter,\n)\nfrom sglang.srt.model_executor.runner_utils.shared_read_event import make_external_event\nfrom sglang.srt.multiplex.pdmux_context import get_current_stream_idx, get_stream_groups\nfrom sglang.srt.runtime_context import (\n    get_exec,\n    get_flags,\n    get_parallel,\n    get_spec,\n)\nfrom sglang.srt.speculative.ragged_verify import resolve_ragged_verify_layout\nfrom sglang.srt.utils import (\n    empty_context,\n    get_available_gpu_memory,\n    is_hip,\n    require_attn_tp_gather,\n    require_mlp_tp_gather,\n)\nfrom sglang.srt.utils.device_timer import device_timer_ctx\nfrom sglang.srt.utils.profile_utils import (\n    export_cuda_graph_capture_trace,\n    graph_capture_profile_dir,\n)\n\ntry:\n    from kt_kernel import KTMoEWrapper\n\n    KTRANSFORMERS_AVAILABLE = True\nexcept ImportError:\n    KTRANSFORMERS_AVAILABLE = False\n\nlogger = logging.getLogger(__name__)\n\nif TYPE_CHECKING:\n    from sglang.srt.model_executor.model_runner import ModelRunner\n    from sglang.srt.speculative.spec_info import SpeculativeAlgorithm\n\n\ndef ragged_verify_compact_graphs_enabled(spec_algorithm: SpeculativeAlgorithm) -> bool:\n    if not spec_algorithm.supports_ragged_verify():\n        return False\n    from sglang.srt.speculative.ragged_verify import ragged_verify_compact_enabled\n\n    return ragged_verify_compact_enabled()\n\n\ndef build_replay_fb_view(\n    forward_batch: ForwardBatch,\n    buffers: DecodeInputBuffers,\n    bs: int,\n    raw_bs: int,\n    num_tokens: int,\n    seq_len_fill_value: int,\n    capture_forward_mode: ForwardMode,\n    is_encoder_decoder: bool,\n) -> SimpleNamespace:\n    """Construct a ForwardBatch-like view for backend replay-side init.\n\n    Combines the original forward_batch (for unpadded / per-iter\n    fields like spec_info, out_cache_loc, and the runtime\n    actual_forward_mode) with the padded capture-time buffers from\n    buffers (for req_pool_indices, seq_lens, seq_lens_cpu,\n    positions, encoder_lens).\n\n    forward_mode is the capture-time mode (used by backends for\n    bucket / dispatch decisions); actual_forward_mode is the\n    runtime mode (may be IDLE while the captured graph targets DECODE\n    — DSV4\'s replay metadata prep uses this for IDLE substitution).\n\n    Subsumes the _replay_forward_batch side channel that DSV4 used to\n    read out-of-band before the init_forward_metadata 3-method ABC.\n    """\n    return SimpleNamespace(\n        batch_size=bs,\n        forward_mode=capture_forward_mode,\n        actual_forward_mode=forward_batch.forward_mode,\n        input_ids=buffers.input_ids[:num_tokens],\n        positions=buffers.positions[:num_tokens],\n        req_pool_indices=buffers.req_pool_indices[:bs],\n        seq_lens=buffers.seq_lens[:bs],\n        seq_lens_sum=(\n            None\n            if forward_batch.seq_lens_sum is None\n            else forward_batch.seq_lens_sum + (bs - raw_bs) * seq_len_fill_value\n        ),\n        # Propagate mirror absence: the pinned buffer is not refreshed when the\n        # batch has no CPU mirror; a stale non-None tensor defeats None-guards.\n        seq_lens_cpu=(\n            None if forward_batch.seq_lens_cpu is None else buffers.seq_lens_cpu[:bs]\n        ),\n        num_padding=bs - raw_bs,\n        encoder_lens=buffers.encoder_lens[:bs] if is_encoder_decoder else None,\n        out_cache_loc=getattr(forward_batch, "out_cache_loc", None),\n        out_cache_loc_dsv4=getattr(forward_batch, "out_cache_loc_dsv4", None),\n        # The mamba-track registry slot (VIRTUAL ids) is the v2p translate SOURCE\n        # for the backend, which copies the result into its own static buffer and\n        # reads THAT in the decode track-save — this slot is never mutated. None\n        # when mamba-track is disabled, slice to [:bs] like every other buffer\n        mamba_track_indices=(\n            None\n            if buffers.mamba_track_indices is None\n            else buffers.mamba_track_indices[:bs]\n        ),\n        spec_info=forward_batch.spec_info,\n    )\n\n\nclass DecodeCudaGraphRunner(BaseCudaGraphRunner):\n    """Decode-phase CUDA graph runner.\n\n    Owns: static input buffers (DecodeInputBuffers), capture-bs list,\n    attention backend, two-batch-overlap plugin, DeepEP adapter, and the\n    pluggable self.backend that handles the actual capture/replay.\n    """\n\n    def __init__(\n        self,\n        model_runner: ModelRunner,\n        *,\n        attn_backend=None,\n        speculative_num_steps: Optional[int] = None,\n        speculative_num_draft_tokens: Optional[int] = None,\n    ):\n        super().__init__(model_runner)\n\n        # In-graph metadata prep: shared buffers -> in-graph private data\n        self.in_graph_metadata_prep_done: Optional[torch.cuda.Event] = None\n\n        # --- core state ------------------------------------------------\n        self.enable_torch_compile = get_flags().capture.enable_torch_compile\n        self.disable_padding = model_runner.server_args.disable_cuda_graph_padding\n        self.is_encoder_decoder = model_runner.model_config.is_encoder_decoder\n        self.require_mlp_tp_gather = (\n            require_mlp_tp_gather() and not self._forward_is_dp_local(model_runner)\n        )\n        self.require_attn_tp_gather = require_attn_tp_gather()\n        # Composite predicates derive from the instance values so the dp-local\n        # draft exemption above stays consistent (require_gathered_buffer ==\n        # mlp_tp_gather or attn_tp_gather; require_mlp_sync adds dp attention).\n        self.require_gathered_buffer = (\n            self.require_mlp_tp_gather or self.require_attn_tp_gather\n        )\n        self.require_mlp_sync = (\n            get_parallel().config.enable_dp_attention or self.require_gathered_buffer\n        )\n        self.enable_two_batch_overlap = (\n            model_runner.server_args.enable_two_batch_overlap\n        )\n        self.use_ngram_embedding = model_runner.ngram_embedding_manager.enabled\n        if self.use_ngram_embedding:\n            hf_config = model_runner.model_config.hf_config\n            self.ngram_embedding_n = hf_config.ngram_embedding_n\n            self.ngram_embedding_k = hf_config.ngram_embedding_k\n        self.speculative_algorithm = get_spec().speculative_algorithm\n        self.enable_profile_cuda_graph = (\n            model_runner.server_args.enable_profile_cuda_graph\n        )\n\n        # --- DSA dense-decode dual-graph -------------------------------\n        # Capture a "dense" (k-only, skip-indexer) and a "sparse" (full indexer)\n        # decode graph per bs bucket, and dispatch on max_kv_len vs index_topk at\n        # replay. Auto-enabled for DSA models (index_topk present in the HF\n        # config) — correct for mixed lengths since any request with\n        # kv_len > index_topk falls back to the sparse graph. Adds ~52 graphs and\n        # ~2x capture time.\n        #\n        # Scoped to HIP (AMD): the k-only dense-decode fast path has only been\n        # validated on MI355X. This is common (non-hardware-gated) code, so on\n        # CUDA we deliberately keep the original behavior (no dual-graph) to\n        # avoid silently changing the CUDA decode path for DSA models (e.g.\n        # DeepSeek-V3.2). CUDA can opt in later once validated there.\n        self.dsa_dual_graph = False\n        self.dsa_index_topk: Optional[int] = None\n        from sglang.srt.configs.model_config import (\n            get_dsa_index_topk,\n            is_deepseek_dsa,\n        )\n\n        hf_config = model_runner.model_config.hf_config\n        if is_hip() and is_deepseek_dsa(hf_config):\n            self.dsa_index_topk = get_dsa_index_topk(hf_config)\n            self.dsa_dual_graph = True\n            logger.info(\n                "[dense-decode] DSA dual-graph enabled: capturing "\n                "dense (k-only) + sparse (full indexer) decode graphs; "\n                "dispatch on max_kv_len vs index_topk=%d.",\n                self.dsa_index_topk,\n            )\n\n        self.attn_tp_size = get_parallel().attn_tp_size\n        self.attn_tp_rank = get_parallel().attn_tp_rank\n        # True if a DSACPLayerCommunicator-style prefill-CP flavor is active\n        # (DSA or MLA). These flavors feed a zigzag-split rank-local layout\n        # into the runner; MHA-arch prefill CP (Qwen3/Qwen2 MoE via PR\n        # #18233) uses the plain LayerCommunicator with an attn_tp-replicated\n        # layout and is intentionally excluded so the attn_tp-local\n        # num_token_non_padded adjustment still runs for it.\n        self.enable_prefill_cp = (\n            is_dsa_enable_prefill_cp() or is_mla_prefill_cp_enabled()\n        )\n\n        self.deepep_adapter = DeepEPCudaGraphRunnerAdapter()\n\n        self.dllm_config = DllmConfig.from_server_args(model_runner.server_args)\n        self.is_dllm = self.dllm_config is not None\n        self.attn_backend = attn_backend or model_runner.attn_backend\n        self.speculative_num_steps = (\n            get_spec().speculative_num_steps\n            if speculative_num_steps is None\n            else speculative_num_steps\n        )\n        self.speculative_num_draft_tokens = (\n            get_spec().speculative_num_draft_tokens\n            if speculative_num_draft_tokens is None\n            else speculative_num_draft_tokens\n        )\n\n        # --- capture mode + tokens-per-bs ------------------------------\n        self.capture_forward_mode = ForwardMode.DECODE\n        self.capture_hidden_mode = self.return_hidden_states_mode\n        # Static capture width.\n        self.captured_req_width = model_runner.decode_num_tokens_per_req(\n            num_draft_tokens=self.speculative_num_draft_tokens\n        )\n        if model_runner.spec_algorithm.is_speculative():\n            if self.model_runner.is_draft_worker:\n                # Draft workers can use TARGET_VERIFY mode.\n                if (\n                    not self.model_runner.spec_algorithm.supports_target_verify_for_draft()\n                ):\n                    raise RuntimeError("This should not happen")\n            self.capture_forward_mode = ForwardMode.TARGET_VERIFY\n        elif self.is_dllm:\n            self.capture_forward_mode = ForwardMode.DLLM_EXTEND\n\n        # --- bucket sizes ---------------------------------------------\n        self.capture_bs, self.compile_bs = get_batch_sizes_to_capture(\n            model_runner, self.captured_req_width\n        )\n        if KTRANSFORMERS_AVAILABLE:\n            KTMoEWrapper.set_capture_batch_sizes(self.capture_bs)\n\n        self.ragged_verify_mode = (\n            ragged_verify_compact_graphs_enabled(self.model_runner.spec_algorithm)\n            and (self.capture_forward_mode == ForwardMode.TARGET_VERIFY)\n            and not self.model_runner.is_draft_worker\n        )\n        self.capture_num_tokens: Optional[list[int]] = (\n            self._build_ragged_verify_token_buckets()\n            if self.ragged_verify_mode\n            else None\n        )\n        self._ragged_graph_size = 0\n        # Per-tier capture layouts; their verify_lens / qo_indptr tensors are\n        # baked into the captured graphs and refreshed in place each replay.\n        self._captured_ragged_layouts: dict[int, object] = {}\n        if self.ragged_verify_mode and (\n            self.enable_two_batch_overlap\n            or model_runner.lora_manager is not None\n            or self.disable_padding\n        ):\n            raise ValueError(\n                "Compact ragged verify does not support two-batch-overlap, "\n                "LoRA, or disable-cuda-graph-padding (bs pads to the captured "\n                "tier); disable SGLANG_RAGGED_VERIFY_MODE or the conflicting "\n                "feature."\n            )\n\n        # Attention backend\n        self.max_bs = max(self.capture_bs)\n        self.max_num_token = self.max_bs * self.captured_req_width\n        self.attn_backend.init_cuda_graph_state(self.max_bs, self.max_num_token)\n\n        # Init PDMux if needed\n        self.maybe_init_pdmux()\n        self.seq_len_fill_value = (\n            self.attn_backend.get_cuda_graph_seq_len_fill_value()\n            if self.dllm_config is None\n            else self.dllm_config.block_size\n        )\n\n        # Non-zero encoder length ensures cross-attention kernels are captured in the graph.\n        self.encoder_len_fill_value = (\n            getattr(model_runner.model_config.hf_config, "max_source_positions", 0)\n            if self.is_encoder_decoder\n            else 0\n        )\n\n        if self.enable_torch_compile:\n            set_torch_compile_config()\n\n        if self.model_runner.lora_manager is not None:\n            # Phase 2 of LoRA CUDA graph init: dense LoRA batch metadata.\n            # Phase 1 (MoE buffers) was handled earlier in ModelRunner via\n            # lora_manager.init_cuda_graph_moe_buffers().\n            self.model_runner.lora_manager.init_cuda_graph_batch_info(\n                max_bs_in_cuda_graph=self.max_bs,\n                num_tokens_per_req=self.captured_req_width,\n            )\n\n        enable_mamba_track = (\n            self.model_runner.server_args.enable_mamba_extra_buffer()\n            and self.model_runner.spec_algorithm.is_none()\n        )\n\n        if self.require_gathered_buffer:\n            assert self.require_mlp_tp_gather or self.require_attn_tp_gather\n\n        # --- buffers ---------------------------------------------------\n        self.buffers: DecodeInputBuffers = DecodeInputBuffers.create(\n            device=self.device,\n            max_bs=self.max_bs,\n            max_num_token=self.max_num_token,\n            hidden_size=self.model_runner.model_config.hidden_size,\n            next_token_logits_buffer=self.model_runner.graph_shared_output.get_logits_buffer(\n                # A draft worker whose FR-Spec hot-vocabulary head was\n                # assembled replicated across TP (init_lm_head) projects the\n                # full hot width on every rank; mirror the draft-extend\n                # runner\'s len(hot_token_id) rule. Absent on target workers\n                # and TP1, which keep the model-config width unchanged.\n                getattr(self.model_runner, "hot_vocab_width", None)\n                or self.model_runner.model_config.vocab_size,\n                rows=self.max_num_token,\n            ),\n            dtype=self.model_runner.model_config.dtype,\n            dp_size=self.dp_size,\n            pp_size=self.pp_size,\n            is_encoder_decoder=self.is_encoder_decoder,\n            require_mlp_tp_gather=self.require_mlp_tp_gather,\n            seq_len_fill_value=self.seq_len_fill_value,\n            encoder_len_fill_value=self.encoder_len_fill_value,\n            num_tokens_per_req=self.captured_req_width,\n            cache_loc_dtype=self._cache_loc_dtype(),\n            enable_mamba_track=enable_mamba_track,\n            ne_token_table=(\n                model_runner.ngram_embedding_manager.table\n                if self.use_ngram_embedding\n                else None\n            ),\n            hc_hidden_size=getattr(\n                self.model_runner.model_config, "hc_hidden_size", None\n            ),\n            pp_proxy_topk_size=self.model_runner.get_pp_proxy_topk_size(),\n            pp_proxy_residual_num_blocks=(\n                self.model_runner.get_pp_proxy_residual_num_blocks()\n            ),\n        )\n        self.buffers.share_buffers()\n        # FB-shared slot registry adopting DecodeInputBuffers storage (same\n        # physical tensors, stable data_ptr for capture vs replay). Provides\n        # the unified fill_from / slot access surface, replacing\n        # populate_from_forward_batch on capture/replay paths.\n        self.buffer_registry: CudaGraphBufferRegistry = build_decode_registry(\n            device=self.device,\n            max_bs=self.max_bs,\n            max_num_token=self.max_num_token,\n            seq_len_fill_value=self.seq_len_fill_value,\n            cache_loc_dtype=self._cache_loc_dtype(),\n            enable_mamba_track=enable_mamba_track,\n            is_encoder_decoder=self.is_encoder_decoder,\n            encoder_len_fill_value=self.encoder_len_fill_value,\n            enable_num_token_non_padded=enable_num_token_non_padded(),\n            require_gathered_buffer=self.require_gathered_buffer,\n            enable_prefill_cp=self.enable_prefill_cp,\n            require_mlp_tp_gather=self.require_mlp_tp_gather,\n            dp_size=self.dp_size,\n            source=self.buffers,\n        )\n\n        # --- backend ---------------------------------------------------\n        self.backend = resolve_decode_backend(self)\n\n        # --- capture --------------------------------------------------\n        try:\n            with model_capture_mode():\n                self.capture()\n        except RuntimeError as e:\n            raise Exception(\n                f"Capture cuda graph failed: {e}\\n" f"{CUDA_GRAPH_CAPTURE_FAILED_MSG}"\n            )\n\n    def _record_in_graph_metadata_prep_done(self):\n        # Purely a marker at this point in the graph; where the shared reads\n        # actually end is the attn backend\'s call.\n        if not self.device_module.is_current_stream_capturing():\n            # Warmup shares this body. Breakable capture still plants: it opens\n            # segment 1 on context entry and every segment re-arms the node.\n            # Routed through device_module so XPU (torch.xpu) is picked up\n            # instead of hitting torch.cuda dummy stubs on non-CUDA builds.\n            return\n        if self.in_graph_metadata_prep_done is None:\n            self.in_graph_metadata_prep_done = make_external_event(self.device_module)\n        event = self.in_graph_metadata_prep_done\n        if event is not None:\n            # Stays None without external-event support, so the read-end\n            # resolution below never hands out an unrecorded event.\n            event.record()\n\n    def _replay_attn_backend(self) -> AttentionBackend:\n        # Under pdmux each stream replays on its own group member.\n        if self.enable_pdmux:\n            return self.model_runner.decode_attn_backend_group[get_current_stream_idx()]\n        return self.attn_backend\n\n    def _resolve_shared_read_ends(self, attn_backend, forward_mode) -> SharedReadEnds:\n        declared = attn_backend.shared_read_ends(forward_mode)\n        if (\n            declared is SharedReadEnds.IN_REPLAY\n            and self.in_graph_metadata_prep_done is None\n        ):\n            # TODO: this lands EARLIER than declared; POST_REPLAY is the sound one.\n            return SharedReadEnds.PRE_REPLAY\n        return declared\n\n    def _publish_read_done(self, in_graph: bool):\n        """Hand the scheduler\'s WAR barrier the event marking this phase\'s\n        shared-buffer reads as done."""\n        if in_graph:\n            # Reads end at the in-graph marker: wire it through, don\'t re-record.\n            self.model_runner.shared_read_done_event = self.in_graph_metadata_prep_done\n        else:\n            read_done = self.device_module.Event()\n            read_done.record()\n            self.model_runner.shared_read_done_event = read_done\n\n    def _build_ragged_verify_token_buckets(self) -> list[int]:\n        buckets = sorted({bs * self.captured_req_width for bs in self.capture_bs})\n        assert buckets and buckets[0] > 0, f"{buckets=}"\n        return buckets\n\n    def _autotune_buffers(self):\n        """Reuse these static decode buffers (sized to max_bs) for the warmup\n        flashinfer-autotune dummy forward instead of allocating a throwaway set\n        — see BaseRunner._autotune_buffers / BaseRunner._dummy_run.\n\n        The dummy forward derives its shape from max_bs and must match these\n        buffers exactly; _dummy_run asserts that. Every autotune-reachable\n        decode shape (plain decode, spec target-verify) matches. DLLM would not\n        (its buffers hold block_size tokens/bs while the dummy run derives 1),\n        but DLLM does not use a flashinfer MoE backend, so autotune never runs\n        for it and this is never reached there.\n        """\n        return self.buffers, self.max_bs\n\n    def maybe_init_pdmux(self):\n        if self.enable_pdmux:\n            self.stream_groups = get_stream_groups()\n            for attn_backend in self.model_runner.decode_attn_backend_group:\n                attn_backend.init_cuda_graph_state(self.max_bs, self.max_num_token)\n\n    def _cache_loc_dtype(self):\n        return torch.int64\n\n    def _make_graph_key(\n        self, size, stream_idx=None, variant_label=None, dsa_variant=None\n    ):\n        return ShapeKey(\n            size=size,\n            stream_idx=stream_idx,\n            variant_label=variant_label,\n            dsa_variant=dsa_variant,\n        )\n\n    def _capture_graph_size(self, *, bs: int, num_tokens: int) -> int:\n        return num_tokens if self.ragged_verify_mode else bs\n\n    def _resolve_dsa_variant(self, forward_batch: ForwardBatch) -> Optional[str]:\n        """Host dispatch: pick which pre-captured DSA decode graph to replay\n        from the batch-max kv_len. If any request has kv_len > index_topk\n        the dense (k-only) graph would be wrong for it, so the whole batch uses\n        the sparse (full indexer) graph. Returns None when dual-graph is off."""\n        if not getattr(self, "dsa_dual_graph", False):\n            return None\n        seq_lens_cpu = getattr(forward_batch, "seq_lens_cpu", None)\n        if seq_lens_cpu is not None and seq_lens_cpu.numel() > 0:\n            # Host-side mirror (maintained incrementally for plain decode) — no\n            # d2h sync needed.\n            max_kv_len = int(seq_lens_cpu.max().item())\n        elif forward_batch.seq_lens is not None and forward_batch.seq_lens.numel() > 0:\n            # Fallback: a single scalar reduction d2h (cheap, per-step).\n            max_kv_len = int(forward_batch.seq_lens.max().item())\n        else:\n            # No length info: be safe and use the correct-for-all sparse graph.\n            return "sparse"\n        return "dense" if max_kv_len <= self.dsa_index_topk else "sparse"\n\n    def _resolve_lora_variant(self, forward_batch: ForwardBatch):\n        if not getattr(self, "record_nolora_graph", False):\n            return None\n        if forward_batch.lora_ids is not None and any(\n            uid is not None for uid in forward_batch.lora_ids\n        ):\n            return "lora"\n        return "nolora"\n\n    @staticmethod\n    def _forward_is_dp_local(model_runner) -> bool:\n        """The DSpark dense draft runs attn-TP-local (draft_tp_context): each\n        DP rank drafts independently with no cross-DP collective, so its\n        hand-built batches carry no dp-global metadata and must key graphs by\n        local batch size. Everything else keeps the dp-global padding path."""\n        if not model_runner.is_draft_worker:\n            return False\n        if not model_runner.spec_algorithm.is_dspark():\n            return False\n        from sglang.srt.speculative.dspark_components.dspark_config import (\n            draft_is_deepseek_v4,\n        )\n\n        return not draft_is_deepseek_v4()\n\n    def _ragged_capture_slots(self, num_tokens: int) -> int:\n        if envs.SGLANG_TEST_RAGGED_VERIFY_FORCE_UNIFORM_CAPTURE.get():\n            return num_tokens // self.captured_req_width\n        return min(num_tokens, self.max_bs)\n\n    def _capture_ragged_verify_layout(self, num_tokens: int):\n        if not self.ragged_verify_mode:\n            return None\n        if envs.SGLANG_TEST_RAGGED_VERIFY_FORCE_UNIFORM_CAPTURE.get():\n            return None\n        from sglang.srt.speculative.ragged_verify import (\n            RaggedVerifyLayout,\n            build_capture_verify_lens,\n        )\n\n        verify_lens_cpu = build_capture_verify_lens(\n            num_tokens=num_tokens,\n            num_slots=self._ragged_capture_slots(num_tokens),\n            num_draft_tokens=self.captured_req_width,\n        )\n        layout = RaggedVerifyLayout.from_verify_lens(\n            verify_lens_cpu=verify_lens_cpu,\n            device=self.device,\n            grid=self.capture_num_tokens,\n        )\n        self._captured_ragged_layouts[num_tokens] = layout\n        return layout\n\n    def _stage_ragged_verify_layout(self, ragged_layout, graph_size_key: int) -> None:\n        # Without this refresh every replay reuses the capture-time synthetic\n        # verify_lens / qo_indptr and mis-slices the packed q rows.\n        cap_layout = self._captured_ragged_layouts.get(graph_size_key)\n        if cap_layout is None:\n            return\n        live = ragged_layout\n        if live.bs != cap_layout.bs or live.cap is None:\n            live = live.padded_to_bucket(\n                padded_bs=cap_layout.bs, cap=self.captured_req_width\n            )\n        cap_layout.verify_lens.copy_(live.verify_lens)\n        cap_layout.qo_indptr_device.copy_(live.qo_indptr_device)\n\n    @staticmethod\n    def _max_dp_batch_size(forward_batch: ForwardBatch) -> int:\n        request_counts = forward_batch.original_global_num_tokens_cpu\n        if request_counts is None:\n            raise RuntimeError(\n                "DP CUDA graph replay requires raw per-rank request counts"\n            )\n        return max(request_counts)\n\n    def can_run_graph(self, forward_batch: ForwardBatch):\n        # Disable for token embedding overrides (dynamic per-request)\n        if forward_batch.replace_embeds is not None:\n            return False\n\n        ragged_layout = (\n            resolve_ragged_verify_layout(forward_batch)\n            if self.ragged_verify_mode\n            else None\n        )\n        if ragged_layout is not None:\n            return self._can_run_ragged_verify_graph(forward_batch, ragged_layout)\n        if self.ragged_verify_mode and forward_batch.forward_mode.is_target_verify():\n            return False\n\n        # Uniform-width replay invariant: the batch\'s actual per-request width\n        # must match this runner\'s capture width; anything else falls back to\n        # eager. (Unset widths pass: not every path fills the field yet.)\n        spec_info = forward_batch.spec_info\n        if (\n            spec_info is not None\n            and spec_info.num_tokens_per_req > 0\n            and spec_info.num_tokens_per_req != self.captured_req_width\n        ):\n            return False\n\n        if self.require_mlp_tp_gather:\n            cuda_graph_bs = self._max_dp_batch_size(forward_batch)\n        else:\n            cuda_graph_bs = forward_batch.batch_size\n\n        graph_key = self._make_graph_key(\n            cuda_graph_bs,\n            stream_idx=get_current_stream_idx() if self.enable_pdmux else None,\n            variant_label=self._resolve_lora_variant(forward_batch),\n        )\n\n        is_bs_supported = (\n            self.backend.can_run(forward_batch, graph_key)\n            if self.disable_padding\n            else cuda_graph_bs <= self.max_bs\n        )\n\n        if self.require_mlp_sync:\n            is_bs_supported = (\n                is_bs_supported and forward_batch.can_run_decode_cuda_graph\n            )\n\n        # NOTE: cuda graph cannot handle mixed batch (encoder_len = 0)\n        # If mixed batch cannot be supported, then encoder_lens can be removed in cuda graph\n        # because the full_text_row_masked_out_mask tensor will always be ones\n        is_encoder_lens_supported = (\n            torch.all(forward_batch.encoder_lens > 0)\n            if self.is_encoder_decoder\n            else True\n        )\n\n        is_tbo_supported = (\n            forward_batch.can_run_tbo if self.enable_two_batch_overlap else True\n        )\n\n        is_ngram_supported = (\n            (\n                forward_batch.batch_size * self.captured_req_width\n                == forward_batch.input_ids.numel()\n            )\n            if self.model_runner.spec_algorithm.is_ngram()\n            else True\n        )\n\n        return (\n            is_bs_supported\n            and is_encoder_lens_supported\n            and is_tbo_supported\n            and is_ngram_supported\n        )\n\n    def _can_run_ragged_verify_graph(self, forward_batch: ForwardBatch, ragged_layout):\n        if not self.attn_backend.supports_ragged_verify_graph:\n            return False\n\n        admission_tokens = ragged_layout.graph_num_tokens\n        is_tokens_supported = admission_tokens <= self.capture_num_tokens[\n            -1\n        ] and forward_batch.batch_size <= self._ragged_capture_slots(admission_tokens)\n\n        is_dp_supported = (\n            forward_batch.can_run_decode_cuda_graph if self.require_mlp_sync else True\n        )\n\n        is_encoder_lens_supported = (\n            torch.all(forward_batch.encoder_lens > 0)\n            if self.is_encoder_decoder\n            else True\n        )\n\n        capture_hidden_mode_matches = (\n            forward_batch.capture_hidden_mode <= self.capture_hidden_mode\n        )\n\n        return (\n            is_tokens_supported\n            and is_dp_supported\n            and is_encoder_lens_supported\n            and capture_hidden_mode_matches\n        )\n\n    def _graph_batch_capture_active(self) -> bool:\n        """Whether the per-batch-size capture-trace feature is active.\n\n        Gated by SGLANG_GRAPH_BATCH_CAPTURE. The original single-trace export\n        (SGLANG_ENABLE_CUDA_GRAPH_CAPTURE_TRACE) takes precedence: when both are\n        set we fall back to the original behavior.\n        """\n        return (\n            envs.SGLANG_GRAPH_BATCH_CAPTURE.get()\n            and not envs.SGLANG_ENABLE_CUDA_GRAPH_CAPTURE_TRACE.get()\n        )\n\n    def _init_profile_context_and_memory_record(self):\n        if self._graph_batch_capture_active():\n            # Per-batch-size capture traces (SGLANG_GRAPH_BATCH_CAPTURE): a\n            # scheduled profiler is stepped once per batch size (see\n            # FullCudaGraphBackend.capture_one) and on_trace_ready writes one\n            # chrome trace per bs.\n            rank = get_parallel().tp_rank\n            runner_name = type(self).__name__\n            trace_dir = graph_capture_profile_dir()\n            os.makedirs(trace_dir, exist_ok=True)\n\n            # Track which BS is currently being captured for trace file naming\n            self._profile_bs_list = list(reversed(self.capture_bs))\n            self._profile_bs_idx = 0\n\n            def on_trace_ready(prof):\n                bs = self._profile_bs_list[self._profile_bs_idx]\n                trace_file = os.path.join(\n                    trace_dir, f"{runner_name}_bs_{bs}_rank{rank}.json.gz"\n                )\n                prof.export_chrome_trace(trace_file)\n                logger.info(f"Saved trace for bs={bs} to {trace_file}")\n                self._profile_bs_idx += 1\n\n            profile_context = profile(\n                activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],\n                # Schedule: wait=2 (skip 2 dummy runs), warmup=0, active=1\n                # (capture run); repeat=0 repeats the cycle so each batch size\n                # gets its own trace.\n                schedule=torch.profiler.schedule(wait=2, warmup=0, active=1, repeat=0),\n                record_shapes=True,\n                with_stack=True,\n                with_flops=True,\n                profile_memory=True,\n                on_trace_ready=on_trace_ready,\n            )\n        else:\n            # a single unscheduled pass over the whole\n            # capture. The combined trace (if any) is exported in\n            # _post_process_after_profile via export_cuda_graph_capture_trace,\n            # gated by SGLANG_ENABLE_CUDA_GRAPH_CAPTURE_TRACE.\n            profile_context = profile(\n                activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],\n                record_shapes=True,\n            )\n        torch.cuda.memory._record_memory_history()\n        return profile_context\n\n    def _post_process_after_profile(self, prof_context):\n        torch.cuda.memory._dump_snapshot("cuda_graph_runner_memory_usage.pickle")\n        torch.cuda.memory._record_memory_history(enabled=None)\n        log_message = (\n            "Sorted by CUDA Time:\\n"\n            + prof_context.key_averages(group_by_input_shape=True).table(\n                sort_by="cuda_time_total", row_limit=10\n            )\n            + "\\n\\nSorted by CPU Time:\\n"\n            + prof_context.key_averages(group_by_input_shape=True).table(\n                sort_by="cpu_time_total", row_limit=10\n            )\n            + "\\n\\nMemory Usage is saved to cuda_graph_runner_memory_usage.pickle\\n"\n        )\n        logger.info(log_message)\n\n        # single-trace export for the whole capture pass; no-op unless\n        # SGLANG_ENABLE_CUDA_GRAPH_CAPTURE_TRACE is set. In per-bs mode\n        # (SGLANG_GRAPH_BATCH_CAPTURE) that env is unset, so this stays a no-op\n        # and the per-bs on_trace_ready handles export instead.\n        export_cuda_graph_capture_trace(\n            prof_context,\n            runner_name=type(self).__name__,\n            tp_rank=get_parallel().tp_rank,\n        )\n\n    def capture_prepare(\n        self,\n        size: int,\n        stream_idx: Optional[int] = None,\n        num_tokens: Optional[int] = None,\n    ):\n        """Build the dummy decode ForwardBatch for capture at size (=bs),\n        populate static input buffers, choose the active attn backend, and\n        optionally build pp_proxy_tensors.\n\n        num_tokens defaults to the uniform bs * num_tokens_per_req; ragged\n        verify capture passes the decoupled (slots, tier tokens) pair.\n\n        Returns (forward_batch, attn_backend, pp_proxy_tensors);\n        pp_proxy_tensors is None unless pp_size > 1.\n        """\n        bs = size\n        buffers: DecodeInputBuffers = self.buffers\n        if num_tokens is None:\n            num_tokens = bs * self.captured_req_width\n\n        # Registry-owned FB-shared slots come through the registry (which\n        # shares physical storage with self.buffers via source=...); the rest\n        # still come off buffers directly.\n        registry = self.buffer_registry\n\n        def _slot(name):\n            return registry.get_slot(name).slice_for(bs, num_tokens)\n\n        input_ids = _slot("input_ids")\n        req_pool_indices = _slot("req_pool_indices")\n        seq_lens = _slot("seq_lens")\n        seq_lens_cpu = _slot("seq_lens_cpu")\n        out_cache_loc = _slot("out_cache_loc")\n        positions = _slot("positions")\n        encoder_lens = (\n            _slot("encoder_lens") if registry.has_slot("encoder_lens") else None\n        )\n        mrope_positions = _slot("mrope_positions")\n        next_token_logits_buffer = buffers.next_token_logits_buffer[:num_tokens]\n        rids_int = buffers.rids_int[:bs] if buffers.rids_int is not None else None\n        bootstrap_room_ids_int = (\n            buffers.bootstrap_room_ids_int[:bs]\n            if buffers.bootstrap_room_ids_int is not None\n            else None\n        )\n\n        # Adjust for attention TP if needed (matching replay path in\n        # populate_from_forward_batch).\n        buffers.num_token_non_padded[...] = num_tokens\n        if (\n            enable_num_token_non_padded()\n            and self.require_gathered_buffer\n            and not self.enable_prefill_cp\n        ):\n            local = compute_local_num_token_non_padded(\n                global_num_token_non_padded=buffers.num_token_non_padded,\n                num_tokens_per_dp=num_tokens,\n            )\n            buffers.num_token_non_padded.copy_(local)\n\n        pp_proxy_tensors = None\n        # pipeline parallelism\n        if self.pp_size > 1:\n            pp_proxy_tensors = PPProxyTensors(\n                {k: v[:num_tokens] for k, v in buffers.pp_proxy_tensors.items()}\n            )\n\n        if self.require_mlp_tp_gather:\n            global_num_tokens_cpu = [num_tokens] * self.dp_size\n        elif self.require_attn_tp_gather:\n            global_num_tokens_cpu = [num_tokens]\n        else:\n            global_num_tokens_cpu = None\n\n        if global_num_tokens_cpu is not None:\n            global_dp_buffer_len = sum(global_num_tokens_cpu)\n            num_tokens_tensor = torch.tensor(\n                global_num_tokens_cpu, dtype=torch.int32, device=input_ids.device\n            )\n            buffers.global_num_tokens_gpu.copy_(num_tokens_tensor)\n            buffers.global_num_tokens_for_logprob_gpu.copy_(num_tokens_tensor)\n        else:\n            global_dp_buffer_len = None\n\n        spec_info = self.get_spec_info(num_tokens)\n        self.capture_hidden_mode = get_required_capture_hidden_mode(\n            self.capture_hidden_mode,\n            spec_info,\n        )\n\n        if self.model_runner.lora_manager is not None:\n            # It is safe to capture CUDA graph using empty LoRA id, as the LoRA kernels will always be launched whenever\n            # `--enable-lora` is set to True (and return immediately if the LoRA id is empty for perf optimization).\n            lora_ids = [None] * bs\n        else:\n            lora_ids = None\n\n        # mamba state tracking (registry-owned when enabled)\n        mamba_track_indices = (\n            _slot("mamba_track_indices")\n            if registry.has_slot("mamba_track_indices")\n            else None\n        )\n        mamba_track_mask = (\n            _slot("mamba_track_mask") if registry.has_slot("mamba_track_mask") else None\n        )\n\n        if stream_idx is None:\n            attn_backend = self.attn_backend\n        else:\n            assert self.enable_pdmux\n            attn_backend = self.model_runner.decode_attn_backend_group[stream_idx]\n\n        forward_batch = ForwardBatch(\n            forward_mode=self.capture_forward_mode,\n            batch_size=bs,\n            input_ids=input_ids,\n            req_pool_indices=req_pool_indices,\n            seq_lens=seq_lens,\n            seq_lens_cpu=seq_lens_cpu,\n            next_token_logits_buffer=next_token_logits_buffer,\n            orig_seq_lens=seq_lens,\n            out_cache_loc=out_cache_loc,\n            seq_lens_sum=seq_lens.sum().item(),\n            mamba_track_indices=mamba_track_indices,\n            mamba_track_mask=mamba_track_mask,\n            mamba_track_seqlens=None,\n            encoder_lens=encoder_lens,\n            return_logprob=False,\n            positions=positions,\n            global_num_tokens_gpu=buffers.global_num_tokens_gpu,\n            global_num_tokens_for_logprob_gpu=buffers.global_num_tokens_for_logprob_gpu,\n            dp_padding_mode=DpPaddingMode.get_default_mode_in_cuda_graph(),\n            global_dp_buffer_len=global_dp_buffer_len,\n            global_num_tokens_cpu=global_num_tokens_cpu,\n            mrope_positions=mrope_positions,\n            spec_algorithm=self.model_runner.spec_algorithm,\n            spec_info=spec_info,\n            capture_hidden_mode=self.capture_hidden_mode,\n            num_token_non_padded=buffers.num_token_non_padded,\n            global_forward_mode=self.capture_forward_mode,\n            lora_ids=lora_ids,\n            rids_int=rids_int,\n            bootstrap_room_ids_int=bootstrap_room_ids_int,\n        )\n\n        # Trip the coordinator so the hisparse code path is captured into the\n        # graph; backends read it from self.model_runner.hisparse_coordinator.\n        forward_batch.hisparse_coordinator = self.model_runner.hisparse_coordinator\n        if forward_batch.hisparse_coordinator is not None:\n            forward_batch.hisparse_coordinator.num_real_reqs.fill_(bs)\n\n        if buffers.ngram_embedding_info is not None:\n            forward_batch.ngram_embedding_info = buffers.ngram_embedding_info.slice(bs)\n\n        return forward_batch, attn_backend, pp_proxy_tensors\n\n    def capture(self) -> None:\n        # Warm up + autotune kernels once before capture (run-once across the\n        # decode + prefill runners; see BaseRunner.warmup).\n        self.warmup()\n        # warmup() may disable torch.compile for a model whose _can_torch_compile\n        # is False; recompute the compile bucket so capture matches.\n        if self.enable_torch_compile and not (get_flags().capture.enable_torch_compile):\n            self.enable_torch_compile = False\n            _, self.compile_bs = get_batch_sizes_to_capture(\n                self.model_runner, self.captured_req_width\n            )\n        profile_context = empty_context()\n        # Holds the active torch profiler during capture so the backend can\n        # advance its schedule (profiler.step()) per batch size. Only the\n        # scheduled per-bs profiler (SGLANG_GRAPH_BATCH_CAPTURE) needs stepping;\n        # the original unscheduled pass leaves this None.\n        self._profiler = None\n        if self.enable_profile_cuda_graph:\n            profile_context = self._init_profile_context_and_memory_record()\n            if self._graph_batch_capture_active():\n                self._profiler = profile_context\n\n        # share_buffers() coalesces seq_lens / seq_lens_cpu through the process-\n        # wide pool, so they may alias a buffer seeded by an earlier runner (the\n        # eager registry fills them with 0). The capture-time attention-metadata\n        # plan reads these as the per-request KV length, and the prefill wrapper\n        # (DLLM_EXTEND) asserts kv_len >= qo_len, so restore the fill value the\n        # captured graph needs before capturing.\n        self.buffers.seq_lens.fill_(self.seq_len_fill_value)\n        self.buffers.seq_lens_cpu.fill_(self.seq_len_fill_value)\n        # Capture runs real forwards, so a mid-serving recapture would index --\n        # and write KV -- through the previous batch\'s live values. Replay is\n        # already covered by the registry\'s padding policy.\n        self.buffers.reset_index_buffers()\n\n        # Trigger CUDA graph capture for specific shapes.\n        # Capture the large shapes first so that the smaller shapes\n        # can reuse the memory pool allocated for the large shapes.\n        with freeze_gc(self.model_runner.server_args.enable_cudagraph_gc):\n            if not self.enable_pdmux:\n                with graph_capture() as graph_capture_context, profile_context as prof:\n                    self.stream = graph_capture_context.stream\n                    with self.backend.capture_session(self.stream):\n                        self._capture_one_stream()\n            else:\n                set_pdmux_status(False)\n                for i, sg in enumerate(self.stream_groups):\n                    with (\n                        graph_capture(stream=sg[1]) as graph_capture_context,\n                        profile_context as prof,\n                    ):\n                        self.stream = graph_capture_context.stream\n                        with self.backend.capture_session(self.stream):\n                            self._capture_one_stream(i)\n\n        if self.enable_profile_cuda_graph:\n            self._post_process_after_profile(prof)\n        self._profiler = None\n\n        # No pool-side pin to clear: the captured full-physical write loc rides the\n        # backend\'s `ForwardMetadata.out_cache_loc_full_physical` (-> KVWriteLoc.full_loc).\n\n    def _capture_one_stream(self, stream_idx: Optional[int] = None) -> None:\n        avail_mem = get_available_gpu_memory(\n            self.model_runner.device,\n            self.model_runner.gpu_id,\n            empty_cache=False,\n        )\n        # Reverse so cuda graphs share memory better.\n        capture_range = (\n            tqdm.tqdm(list(reversed(self.capture_bs)))\n            if get_parallel().tp_rank == 0\n            else reversed(self.capture_bs)\n        )\n        lora_variants = (\n            [("lora", True), ("nolora", False)]\n            if getattr(self, "record_nolora_graph", False)\n            else [(None, None)]\n        )\n        # DSA: capture a dense (k-only) and a sparse (full indexer) graph\n        # per bs bucket. Order: dense first so its (smaller) capture-time peak\n        # runs while the shared pool is fresh; sparse\'s peak subsumes it.\n        # getattr default: subclasses like EAGLEDraftCudaGraphRunner reuse this\n        # capture() but don\'t run DecodeCudaGraphRunner.__init__ (so they never\n        # set dsa_dual_graph) and override capture_one_shape with a signature that\n        # has no dsa_variant. Default to no dual-graph and, for the None variant,\n        # call capture_one_shape without the extra arg so those overrides work.\n        dsa_variants = (\n            ["dense", "sparse"] if getattr(self, "dsa_dual_graph", False) else [None]\n        )\n        for bs in capture_range:\n            if get_parallel().tp_rank == 0:\n                avail_mem = get_available_gpu_memory(\n                    self.model_runner.device,\n                    self.model_runner.gpu_id,\n                    empty_cache=False,\n                )\n                capture_range.set_description(\n                    f"Capturing batches ({bs=} {avail_mem=:.2f} GB)"\n                )\n\n            for variant_label, _variant_has_lora in lora_variants:\n                _set_capture_lora_variant(variant_label)\n                for dsa_variant in dsa_variants:\n                    _set_capture_dsa_variant(dsa_variant)\n                    with torch_compile_decoration.patch_model(\n                        self.model_runner.model,\n                        bs in self.compile_bs,\n                        num_tokens=bs * self.captured_req_width,\n                        tp_group=self.model_runner.tp_group,\n                    ) as forward:\n                        if dsa_variant is None:\n                            self.capture_one_shape(\n                                bs, forward, stream_idx, variant_label\n                            )\n                        else:\n                            self.capture_one_shape(\n                                bs, forward, stream_idx, variant_label, dsa_variant\n                            )\n        _set_capture_dsa_variant(None)\n\n    def capture_one_shape(\n        self,\n        size: int,\n        forward: Callable,\n        stream_idx: Optional[int] = None,\n        variant_label: Optional[str] = None,\n        dsa_variant: Optional[str] = None,\n    ):\n        num_tokens = size * self.captured_req_width\n        bs = self._ragged_capture_slots(num_tokens) if self.ragged_verify_mode else size\n\n        # Sanity-check: --debug-cuda-graph requires breakable backend.\n        if get_exec().graph.debug_cuda_graph:\n            assert isinstance(\n                self.backend, BreakableCudaGraphBackend\n            ), "Breakable CUDA graph is required for --debug-cuda-graph"\n\n        forward_batch, attn_backend, pp_proxy_tensors = self.capture_prepare(\n            bs, stream_idx=stream_idx, num_tokens=num_tokens\n        )\n\n        # All setup hooks below read get_attn_backend() (TboForwardBatchPreparer,\n        # DeepEP adapter, …) so they must run inside the same ForwardContext\n        # that wraps the warmup/capture forward.\n        with forward_context(ForwardContext(attn_backend=attn_backend)):\n            self.tbo_plugin.capture_one_batch_size(forward_batch, num_tokens=num_tokens)\n\n            if forward_batch.lora_ids is not None:\n                self.model_runner.lora_manager.prepare_lora_batch(forward_batch)\n\n            attn_backend.init_forward_metadata_out_graph(forward_batch, in_capture=True)\n\n            def run_once():\n                # Graph-recordable metadata-prep hook. The unified memory pool\n                # records ZERO translate nodes here: all its read/write translates\n                # run eagerly in `init_forward_metadata_out_graph` (replay-prep), so\n                # the captured graph reads already-physical locs. Base no-op for triton.\n                attn_backend.init_forward_metadata_in_graph(forward_batch)\n                self._record_in_graph_metadata_prep_done()\n\n                # No invalidate_loc_cache() here: the unified pool translates its\n                # locs in `init_forward_metadata_out_graph`, so no cache to invalidate.\n\n                forward_batch.dp_local_start_pos = forward_batch.dp_local_num_tokens = (\n                    None\n                )\n                set_dp_buffer_len(\n                    forward_batch.global_dp_buffer_len,\n                    num_tokens,\n                    forward_batch.dp_padding_mode.is_max_len(),\n                    forward_batch.global_num_tokens_cpu,\n                )\n                set_is_extend_in_batch(False)\n\n                kwargs = {}\n                if (\n                    self.pp_size > 1\n                    and "pp_proxy_tensors" in inspect.signature(forward).parameters\n                ):\n                    kwargs["pp_proxy_tensors"] = PPProxyTensors(\n                        {k: v.clone() for k, v in pp_proxy_tensors.tensors.items()}\n                    )\n                if (\n                    self.model_runner.spec_algorithm.is_dflash_family()\n                    and self.model_runner.is_draft_worker\n                    and "input_embeds" in inspect.signature(forward).parameters\n                    and not hasattr(self.model_runner.model, "forward_embed")\n                ):\n                    kwargs["input_embeds"] = self.buffers.input_embeds[:num_tokens]\n\n                from marlin_capture import begin_capture\n                begin_capture(self, shape_key)\n                out = forward(\n                    forward_batch.input_ids,\n                    forward_batch.positions,\n                    forward_batch,\n                    **kwargs,\n                )\n                for capture_hook in self.model_runner.capture_tail_hooks:\n                    capture_hook(self, out, forward_batch, num_tokens)\n                from marlin_capture import end_capture\n                end_capture()\n                return out\n\n            self.deepep_adapter.capture(is_extend_in_batch=False)\n            canary_ctx = (\n                c.with_active_single_forward_manager(0)\n                if (c := self.model_runner.canary_manager) is not None\n                else contextlib.nullcontext()\n            )\n            # Full-physical write loc lives in the attention metadata (the backend\'s\n            # `out_cache_loc_full_physical` -> KVWriteLoc.full_loc), so the runner\n            # wires no buffer here. (SWA write loc rides the `swa_out_cache_loc` rail.)\n\n            with canary_ctx:\n                shape_key = self._make_graph_key(\n                    self._capture_graph_size(bs=bs, num_tokens=num_tokens),\n                    stream_idx,\n                    variant_label,\n                    dsa_variant,\n                )\n                post_warmup_hook = getattr(\n                    self.model_runner.attn_backend,\n                    "on_after_cuda_graph_warmup",\n                    None,\n                )\n                maybe_flashinfer_autotune_speculative_draft(\n                    self,\n                    run_once,\n                    post_warmup_hook=post_warmup_hook,\n                    run_lm_head=True,\n                )\n                self.backend.capture_one(\n                    shape_key,\n                    run_once,\n                    capture_inputs=None,\n                    post_warmup_hook=post_warmup_hook,\n                )\n\n    def _validate_capture_hidden_mode(self, forward_batch: ForwardBatch) -> None:\n        if self.capture_hidden_mode < forward_batch.capture_hidden_mode:\n            raise RuntimeError(\n                "The runtime hidden-state mode exceeds the fixed CUDA graph "\n                f"capture mode ({self.capture_hidden_mode.name})."\n            )\n\n    def load_batch(\n        self,\n        forward_batch: ForwardBatch,\n        pp_proxy_tensors: Optional[PPProxyTensors] = None,\n    ):\n        ragged_layout = (\n            resolve_ragged_verify_layout(forward_batch)\n            if self.ragged_verify_mode\n            else None\n        )\n        is_ragged = ragged_layout is not None\n\n        self.deepep_adapter.replay()\n\n        if not forward_batch.needs_forward_metadata_init():\n            # Pre-planned (plan-stream load_batch already ran).\n            # In speculative decoding, these two fields are still needed.\n            graph_size_key = (\n                self._ragged_graph_size\n                if is_ragged\n                else self._capture_graph_size(\n                    bs=self.bs, num_tokens=self.bs * self.captured_req_width\n                )\n            )\n            if is_ragged:\n                assert self.raw_num_token == ragged_layout.graph_num_tokens, (\n                    f"stale ragged raw_num_token {self.raw_num_token} != "\n                    f"{ragged_layout.graph_num_tokens}"\n                )\n                self._stage_ragged_verify_layout(ragged_layout, graph_size_key)\n            self.buffers.input_ids[: self.raw_num_token].copy_(forward_batch.input_ids)\n            self.buffers.positions[: self.raw_num_token].copy_(forward_batch.positions)\n            if (\n                not is_ragged\n                and self.model_runner.spec_algorithm.is_dflash_family()\n                and self.model_runner.is_draft_worker\n                and forward_batch.input_embeds is not None\n            ):\n                self.buffers.input_embeds[: self.raw_num_token].copy_(\n                    forward_batch.input_embeds\n                )\n            variant_label = self._resolve_lora_variant(forward_batch)\n            dsa_variant = self._resolve_dsa_variant(forward_batch)\n            stream_idx = get_current_stream_idx() if self.enable_pdmux else None\n            self._replay_graph_key = self._make_graph_key(\n                graph_size_key, stream_idx, variant_label, dsa_variant\n            )\n            return\n\n        buffers = self.buffers\n        self._validate_capture_hidden_mode(forward_batch)\n\n        raw_bs = forward_batch.batch_size\n\n        if is_ragged:\n            raw_num_token = ragged_layout.graph_num_tokens\n            graph_size_key = self._ragged_graph_num_tokens(raw_num_token)\n            assert graph_size_key == ragged_layout.graph_num_tokens, (\n                f"ragged verify tier mismatch: runner tier {graph_size_key} != "\n                f"layout graph_num_tokens {ragged_layout.graph_num_tokens}"\n            )\n            bs = self._ragged_capture_slots(graph_size_key)\n            assert bs >= raw_bs, (\n                f"ragged capture slots {bs} (tier {graph_size_key}) < raw_bs "\n                f"{raw_bs}; the planner must reject this batch before replay"\n            )\n            padded_num_tokens = graph_size_key\n            self._stage_ragged_verify_layout(ragged_layout, graph_size_key)\n        else:\n            raw_num_token = raw_bs * self.captured_req_width\n            if self.require_mlp_tp_gather:\n                max_batch_size = self._max_dp_batch_size(forward_batch)\n                bs = self._pad_to_bucket(max_batch_size, self.capture_bs)\n            else:\n                bs = self._pad_to_bucket(raw_bs, self.capture_bs)\n            padded_num_tokens = bs * self.captured_req_width\n            graph_size_key = self._capture_graph_size(\n                bs=bs, num_tokens=padded_num_tokens\n            )\n\n        self.buffer_registry.fill_from(\n            forward_batch,\n            raw_bs=raw_bs,\n            padded_bs=bs,\n            raw_num_tokens=raw_num_token,\n            padded_num_tokens=padded_num_tokens,\n            pp_proxy_tensors=pp_proxy_tensors,\n        )\n\n        if (\n            not is_ragged\n            and self.model_runner.spec_algorithm.is_dflash_family()\n            and self.model_runner.is_draft_worker\n            and forward_batch.input_embeds is not None\n        ):\n            buffers.input_embeds[:raw_num_token].copy_(forward_batch.input_embeds)\n        # Padded tokens aren\'t read, so skip zeroing. Ragged input_ids arrive\n        # from the planner already padded to the tier, invalid slots zeroed.\n        if self.enable_two_batch_overlap:\n            self.tbo_plugin.replay_prepare(\n                forward_mode=self.capture_forward_mode,\n                bs=bs,\n                num_token_non_padded=len(forward_batch.input_ids),\n                spec_info=forward_batch.spec_info,\n            )\n        if (\n            not is_ragged\n            and forward_batch.forward_mode.is_idle()\n            and forward_batch.spec_info is not None\n        ):\n            forward_batch.spec_info.custom_mask = buffers.custom_mask\n\n        attn_backend = self._replay_attn_backend()\n        fb_view = build_replay_fb_view(\n            forward_batch=forward_batch,\n            buffers=buffers,\n            bs=bs,\n            raw_bs=raw_bs,\n            num_tokens=padded_num_tokens,\n            seq_len_fill_value=self.seq_len_fill_value,\n            capture_forward_mode=self.capture_forward_mode,\n            is_encoder_decoder=self.is_encoder_decoder,\n        )\n        attn_backend.init_forward_metadata_out_graph(fb_view)\n\n        self.raw_bs = raw_bs\n        self.raw_num_token = raw_num_token\n        self.bs = bs\n        if is_ragged:\n            self._ragged_graph_size = graph_size_key\n\n        if self.model_runner.hisparse_coordinator is not None:\n            self.model_runner.hisparse_coordinator.num_real_reqs.fill_(raw_bs)\n\n        variant_label = self._resolve_lora_variant(forward_batch)\n        dsa_variant = self._resolve_dsa_variant(forward_batch)\n        stream_idx = get_current_stream_idx() if self.enable_pdmux else None\n        self._replay_graph_key = self._make_graph_key(\n            graph_size_key, stream_idx, variant_label, dsa_variant\n        )\n\n    def _ragged_graph_num_tokens(self, total_verify_tokens: int) -> int:\n        from sglang.srt.speculative.ragged_verify import round_up_grid\n\n        return round_up_grid(total_verify_tokens, self.capture_num_tokens)\n\n    def execute(\n        self,\n        forward_batch: ForwardBatch,\n        pp_proxy_tensors: Optional[PPProxyTensors] = None,\n    ) -> Union[LogitsProcessorOutput, PPProxyTensors]:\n        timer_ctx = device_timer_ctx(\n            self.model_runner.device_timer, forward_batch.forward_mode.name.lower()\n        )\n        shared_read_ends = self._resolve_shared_read_ends(\n            self._replay_attn_backend(), forward_batch.forward_mode\n        )\n        with timer_ctx, self.backend.replay_session():\n            self.load_batch(forward_batch, pp_proxy_tensors)\n            if envs.SGLANG_LOG_DECODE_GRAPH_KEY.get():\n                logger.info(\n                    "Decode graph replay: worker=%s key_size=%s (%s) mode=%s raw_bs=%d%s",\n                    "draft" if self.model_runner.is_draft_worker else "target",\n                    self._replay_graph_key.size,\n                    "num_tokens" if self.ragged_verify_mode else "bs",\n                    forward_batch.forward_mode.name,\n                    forward_batch.batch_size,\n                    (\n                        f" slots={self._ragged_capture_slots(self._replay_graph_key.size)}"\n                        if self.ragged_verify_mode\n                        else ""\n                    ),\n                )\n            if shared_read_ends is SharedReadEnds.PRE_REPLAY:\n                self._publish_read_done(in_graph=False)\n\n            output = self.backend.replay(self._replay_graph_key, forward_batch)\n            from marlin_capture import after_execute\n            after_execute(self, forward_batch)\n\n            if shared_read_ends is SharedReadEnds.IN_REPLAY:\n                self._publish_read_done(in_graph=True)\n\n            if shared_read_ends is SharedReadEnds.POST_REPLAY:\n                self._publish_read_done(in_graph=False)\n\n        if isinstance(output, LogitsProcessorOutput):\n            if self.is_dllm:\n                next_token_logits = None\n                full_logits = (\n                    output.full_logits[: self.raw_num_token]\n                    if output.full_logits is not None\n                    else None\n                )\n            else:\n                full_logits = None\n                next_token_logits = (\n                    output.next_token_logits[: self.raw_num_token]\n                    if output.next_token_logits is not None\n                    else None\n                )\n\n            return LogitsProcessorOutput(\n                next_token_logits=next_token_logits,\n                full_logits=full_logits,\n                hidden_states=(\n                    output.hidden_states[: self.raw_num_token]\n                    if output.hidden_states is not None\n                    else None\n                ),\n                customized_info=output.customized_info,\n            )\n        else:\n            assert isinstance(output, PPProxyTensors)\n            return PPProxyTensors({k: v[: self.bs] for k, v in output.tensors.items()})\n\n    def get_spec_info(self, num_tokens: int):\n        spec_info = None\n        if (\n            self.model_runner.spec_algorithm.is_eagle()\n            or self.model_runner.spec_algorithm.is_standalone()\n        ):\n            from sglang.srt.speculative.eagle_info import EagleVerifyInput\n\n            if self.model_runner.is_draft_worker:\n                raise RuntimeError("This should not happen.")\n            else:\n\n                capture_mode = (\n                    CaptureHiddenMode.NULL\n                    if self.model_runner.spec_algorithm.is_standalone()\n                    else CaptureHiddenMode.FULL\n                )\n                spec_info = EagleVerifyInput(\n                    draft_token=None,\n                    custom_mask=self.buffers.custom_mask,\n                    positions=None,\n                    retrieve_index=None,\n                    retrieve_next_token=None,\n                    retrieve_next_sibling=None,\n                    retrieve_cum_len=None,\n                    spec_steps=self.speculative_num_steps,\n                    topk=get_spec().speculative_eagle_topk,\n                    draft_token_num=self.speculative_num_draft_tokens,\n                    capture_hidden_mode=capture_mode,\n                    seq_lens_sum=None,\n                    seq_lens_cpu=None,\n                )\n                # MTP models (e.g. deepseek_nextn) read spec_info.hidden_states\n                spec_info.hidden_states = torch.zeros(\n                    (num_tokens, self.model_runner.model_config.hidden_size),\n                    dtype=self.model_runner.dtype,\n                    device=self.model_runner.device,\n                )\n        elif self.model_runner.spec_algorithm.is_dflash_family():\n            from sglang.srt.speculative.dflash_info import DFlashVerifyInput\n            from sglang.srt.speculative.dflash_utils import (\n                resolve_dflash_verify_mask_policy,\n            )\n\n            # Avoid enabling custom-mask modes during graph capture for backends that\n            # can express DFLASH verify via their built-in causal path.\n            _, build_custom_mask = resolve_dflash_verify_mask_policy(\n                self.model_runner.attn_backend\n            )\n            spec_info = DFlashVerifyInput(\n                draft_token=None,\n                positions=None,\n                draft_token_num=self.captured_req_width,\n                custom_mask=(\n                    None\n                    if (self.model_runner.is_draft_worker or not build_custom_mask)\n                    else self.buffers.custom_mask\n                ),\n                capture_hidden_mode=(\n                    CaptureHiddenMode.NULL\n                    if self.model_runner.is_draft_worker\n                    else CaptureHiddenMode.FULL\n                ),\n                ragged_verify_layout=self._capture_ragged_verify_layout(num_tokens),\n            )\n\n        elif self.model_runner.spec_algorithm.is_ngram():\n            from sglang.srt.speculative.ngram_info import NgramVerifyInput\n\n            spec_info = NgramVerifyInput(\n                draft_token=None,\n                custom_mask=self.buffers.custom_mask,\n                positions=None,\n                retrieve_index=None,\n                retrieve_next_token=None,\n                retrieve_next_sibling=None,\n                draft_token_num=self.captured_req_width,\n            )\n            spec_info.capture_hidden_mode = CaptureHiddenMode.NULL\n\n        return spec_info\n'}
patch={'sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py': {'before': 'fab293ee5aaa0d285d161c3bbf23606d2908fe3273eae5f32eb5b49db52f81e4', 'after': '78faf70b70d4a5b3bf267fcf5c4bf976aa7262ade9956b8a41ed57dc2d570477'}, 'sglang/srt/layers/moe/moe_runner/marlin.py': {'before': 'e1db2b78559fd7ed482710f0b95104339ebcd41721a3cb85e17fbc63ab683734', 'after': '8842730a53bb45094860ec8f8956103815c3215b833caaaf25b7d875a8dcd6f7'}, 'sglang/srt/hardware_backend/gpu/quantization/gptq_kernels.py': {'before': '375085abdd2f4abca3d5831045d5f11d012c1877e0a19f79ebe89593b5568370', 'after': '7db88d354b022ca52234c45afdb71a8fdb5f4060f4c903475f258a9e6812a2b2'}, 'sglang/srt/model_executor/model_runner.py': {'before': '8d84ad370871e4073973922d3b0cf4231bd3b4c793c0275e3c3b11370eba57a8', 'after': '250c2a8dce847fadf931b3fe16e5e0d8e1743152373b78eafeb1eb590704f6d1'}, 'sglang/srt/model_executor/runner/decode_cuda_graph_runner.py': {'before': 'ed359b9f3a0ad36bf290c4e71237fa9103e2b6ba82b28f4ef07ad2fe23e61140', 'after': '99bc9c0e7b8c45a6ead96ffb06df6d460cd10e141eaad359ee86eaef482c8302'}}
for name,text in modules.items():(KIT/name).write_text(text)
script="import hashlib,json,pathlib,sglang\nroot=pathlib.Path(sglang.__file__).parent.parent\n"
script+="sources="+repr(replacements)+"\npatch="+repr(patch)+"\n"
script+="for name,text in sources.items():\n p=root/name\n assert hashlib.sha256(p.read_bytes()).hexdigest()==patch[name]['before'],name\n"
script+="for name,text in sources.items():\n p=root/name\n p.write_text(text)\n assert hashlib.sha256(p.read_bytes()).hexdigest()==patch[name]['after'],name\n"
script+="(pathlib.Path('/kaggle/working/marlin-replay')/'installed-patch.json').write_text(json.dumps(patch,indent=2))\n"
(KIT/'apply_patch.py').write_text(script)
run([PYTHON,KIT/'apply_patch.py'],env=env)
env['PYTHONPATH']=str(KIT)


In [ ]:
remaining=SESSION_DEADLINE-time.monotonic()
assert remaining>=2520,'insufficient time for fixed replay and evidence'
try:
    with (OUT/'benchmark.log').open('xb') as log:
        done=subprocess.run([PYTHON,str(KIT/'benchmark.py'),'--capture',str(SOURCE/'capture'),'--out',str(OUT/'benchmark')],env=env,stdout=log,stderr=subprocess.STDOUT,timeout=2400)
    assert done.returncode==0,'fixed replay worker failed; preserve evidence, no retry'
finally:
    hashes={str(p.relative_to(OUT)):_sha(p) for p in OUT.rglob('*') if p.is_file() and p.name!='output-hashes.json'}
    (OUT/'output-hashes.json').write_text(json.dumps(hashes,indent=2))
